In [2]:
# =======================================
# CA-GAT + ACC (robust static seeds + inter-aware weak labels) — Training Cell
# =======================================
import os, re, json, math, atexit, traceback, warnings, random, time
from pathlib import Path
from collections import defaultdict, deque

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from packaging.version import parse as V
from tqdm.auto import tqdm

# PYG
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv

# ------------------------------ top-level config ------------------------------
SPLIT                 = "train"  # "train" | "valid" | "test"
GC_DIR                = f"Dataset/{SPLIT}/hetero_ready_gcbert"
UNIFIED_JSON_DIRS     = [f"Dataset/{SPLIT}/unified_json", f"Dataset/{SPLIT}/unified"]
AUG_JSON_DIRS         = [f"Dataset/{SPLIT}/unified_aug", f"Dataset/{SPLIT}/unified"]
LOGDIR                = f"out/cagat_acc_interproc_{SPLIT}"

NODE_TYPE             = "node"
DEVICE                = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CUDA & memory knobs
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

USE_AMP            = True   # mixed precision
USE_CHECKPOINT     = True   # gradient checkpointing in GAT blocks
CLEAR_CACHE_EVERY  = 10
SKIP_ON_OOM        = True

# Train hyperparams
EPOCHS                = 5
BATCH_SIZE            = 1
HIDDEN                = 128
HEADS                 = 4
LAYERS                = 6
DROPOUT               = 0.10
LR                    = 2e-3
WEIGHT_DECAY          = 1e-4
SAVE_EVERY_STEPS      = 200
LOG_JSONL_EVERY       = 10

# Call-chain contextualization
CALL_CHAIN_ATTN       = True
CALL_CHAIN_MAX_HOPS   = 5
CALL_CHAIN_SPARSE_NTHRESH = 999999

# Loss weights (scheduled)
LAMBDA_NODE_BCE_BASE       = 1.0
LAMBDA_EDGE_SMOOTH_BASE    = 0.05
LAMBDA_INTER_CONTRAST_BASE = 0.10
LAMBDA_PATH_SUPER_BASE     = 0.25

# Edge weights for smoothness
EDGE_WEIGHTS = {
    'CFG': 0.5, 'DFG': 1.1, 'CFG_REV': 0.3, 'DFG_REV': 0.9,
    'CALL': 1.8, 'CALL_REV': 1.0,
    'ARG2PARAM': 2.2, 'ARG2PARAM_REV': 1.5,
    'RET2CALL': 2.2, 'RET2CALL_REV': 1.5,
    'RET2LHS': 1.8, 'RET2LHS_REV': 1.2,
}

warnings.filterwarnings("ignore", message="You are using `torch.load` with `weights_only=False`", category=FutureWarning)

# ------------------------------ AMP handling (Torch 2.4.x clean) ------------------------------
_TVER = V(torch.__version__)
USE_NEW_AMP_API = _TVER >= V("2.5.0")
if USE_NEW_AMP_API:
    from torch.amp import autocast as _autocast_new, GradScaler as _GradScalerNew
    def autocast_ctx(enabled=True): 
        return _autocast_new(device_type=("cuda" if DEVICE.type=="cuda" else "cpu"),
                             dtype=torch.float16, enabled=enabled)
    SCALER = _GradScalerNew("cuda" if DEVICE.type=="cuda" else "cpu", enabled=(USE_AMP and DEVICE.type=="cuda"))
else:
    from torch.cuda.amp import autocast as _autocast_old, GradScaler as _GradScalerOld
    warnings.filterwarnings("ignore", category=FutureWarning, message="`torch.cuda.amp.autocast")
    warnings.filterwarnings("ignore", category=FutureWarning, message="`torch.cuda.amp.GradScaler")
    def autocast_ctx(enabled=True): 
        return _autocast_old(dtype=torch.float16, enabled=(enabled and DEVICE.type=="cuda"))
    SCALER = _GradScalerOld(enabled=(USE_AMP and DEVICE.type=="cuda"))
print(f"[AMP] torch={torch.__version__} device={DEVICE.type}")

# ------------------------------ IO helpers ------------------------------
def safe_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def find_json(basename: str) -> str | None:
    for d in AUG_JSON_DIRS:
        jd = Path(d)
        for suf in (".json",".aug.json",".unified.json",".jsonl",".txt"):
            p = jd / f"{basename}{suf}"
            if p.exists(): return str(p)
        ms = sorted(jd.glob(f"{basename}.*"))
        if ms: return str(ms[0])
    return None

def _strip_comments_commas(txt: str) -> str:
    txt = re.sub(r'//.*?$', '', txt, flags=re.M)
    txt = re.sub(r'/\*.*?\*/', '', txt, flags=re.S)
    txt = txt.replace('\ufeff','').replace('\x00','')
    txt = re.sub(r',\s*(\}|\])', r'\1', txt)
    return txt

def _extract_nodes_array(txt: str):
    m = re.search(r'"nodes"\s*:', txt)
    if not m: return None
    i = m.end()
    while i < len(txt) and txt[i] != '[': i += 1
    if i>=len(txt) or txt[i] != '[': return None
    depth=0; start=i
    for j,ch in enumerate(txt[i:], start=i):
        if ch=='[': depth+=1
        elif ch==']':
            depth-=1
            if depth==0: return txt[start:j+1]
    return None

def load_aug_json(path: str) -> dict | list | None:
    if not path or not os.path.exists(path): return None
    p = Path(path)
    for enc in ("utf-8","utf-8-sig","latin-1"):
        try:
            raw = p.read_text(enc)
            break
        except Exception:
            raw = None
    if raw is None: return None
    try:
        j = json.loads(raw); 
        if isinstance(j,(dict,list)): return j
    except json.JSONDecodeError: pass
    san = _strip_comments_commas(raw)
    try:
        j = json.loads(san); 
        if isinstance(j,(dict,list)): return j
    except json.JSONDecodeError: pass
    arr = _extract_nodes_array(san)
    if arr:
        try:
            nodes = json.loads(arr)
            if isinstance(nodes, list): return {"nodes": nodes}
        except json.JSONDecodeError: pass
    nodes=[]; paths=[]
    for line in san.splitlines():
        line=line.strip()
        if not line or line[0] not in "{[": continue
        try:
            o = json.loads(line)
            if isinstance(o, dict):
                if isinstance(o.get("nodes"), list): nodes += o["nodes"]
                if isinstance(o.get("vulnerable_paths"), list): paths += o["vulnerable_paths"]
                for k in ("sinks","vulnerabilities","vul_nodes","labels","positives"):
                    if isinstance(o.get(k), list):
                        nodes += [{'_id': x, 'is_sink': True} for x in o[k]]
            elif isinstance(o, list):
                nodes += o
        except json.JSONDecodeError:
            continue
    if nodes or paths:
        d={"nodes": nodes} if nodes else {}
        if paths: d["vulnerable_paths"]=paths
        return d if d else None
    return None

def _coerce_int(x):
    if isinstance(x, int): return x
    if isinstance(x, str) and x.strip().lstrip("-").isdigit():
        try: return int(x)
        except ValueError: return x
    return x

def _harvest_ids_from_any_json(j):
    pos = set()
    if j is None: return pos
    nodes = j["nodes"] if isinstance(j, dict) and isinstance(j.get("nodes"), list) else (j if isinstance(j, list) else [])
    for n in nodes:
        if not isinstance(n, dict): continue
        nid = _coerce_int(n.get("_id"))
        lab = (n.get("label") or n.get("_label") or n.get("class") or n.get("tag") or "")
        lab_u = str(lab).upper()
        is_sink = bool(n.get("is_sink")) or ("SINK" in lab_u) or ("VULN" in lab_u) or ("VULNERABLE" in lab_u)
        if isinstance(nid, int) and is_sink:
            pos.add(nid)
    if isinstance(j, dict):
        for k in ("sinks","vulnerabilities","vul_nodes","labels","positives","positive_nodes"):
            if isinstance(j.get(k), list):
                for x in j[k]:
                    xi = _coerce_int(x)
                    if isinstance(xi, int): pos.add(xi)
        vps = j.get("vulnerable_paths", [])
        if isinstance(vps, list):
            for p in vps:
                if isinstance(p, (list, tuple)) and len(p)>0:
                    a = _coerce_int(p[0]); b = _coerce_int(p[-1])
                    if isinstance(a, int): pos.add(a)
                    if isinstance(b, int): pos.add(b)
    return pos

def node_sink_labels_from_json(json_path: str, g: HeteroData, use_paths_as_pos=True) -> torch.Tensor | None:
    if not json_path or not os.path.exists(json_path): return None
    j = load_aug_json(json_path)
    if j is None: return None
    pos_ids = _harvest_ids_from_any_json(j)
    st = g[NODE_TYPE]
    device = (st.x_text.device if hasattr(st,"x_text") else (st.x.device if hasattr(st,"x") else "cpu"))
    if hasattr(st, "nid"):
        ids = [ _coerce_int(i) for i in st.nid.view(-1).tolist() ]
        return torch.tensor([ 1.0 if (isinstance(i,int) and i in pos_ids) else 0.0 for i in ids ],
                            dtype=torch.float32, device=device)
    if isinstance(j, dict) and isinstance(j.get("nodes"), list) and len(j["nodes"]) == st.num_nodes:
        ordered_ids = [ _coerce_int(n.get("_id")) if isinstance(n, dict) else None for n in j["nodes"] ]
        return torch.tensor([ 1.0 if (isinstance(i,int) and i in pos_ids) else 0.0 for i in ordered_ids ],
                            dtype=torch.float32, device=device)
    return None

def read_paths_from_json(json_path: str) -> list:
    if not json_path or not os.path.exists(json_path): return []
    j = load_aug_json(json_path)
    if not isinstance(j, dict): return []
    paths = j.get("vulnerable_paths", [])
    return paths if isinstance(paths, list) else []

# ------------------------------ edges / dataset ------------------------------
def get_edge_index(g: HeteroData, et: tuple[str,str,str]) -> torch.Tensor:
    if et not in g.edge_types:
        dev = (g[NODE_TYPE].x_text.device if hasattr(g[NODE_TYPE],"x_text")
               else (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else 'cpu'))
        return torch.zeros((2,0), dtype=torch.long, device=dev)
    store = g[et]
    ei = getattr(store, "edge_index", None)
    if ei is not None:
        return ei
    adj_t = getattr(store, "adj_t", None)
    if adj_t is not None:
        row, col, _ = adj_t.coo()     # adj_t is transposed
        return torch.stack([col, row], dim=0)
    dev = (g[NODE_TYPE].x_text.device if hasattr(g[NODE_TYPE],"x_text")
           else (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else 'cpu'))
    return torch.zeros((2,0), dtype=torch.long, device=dev)

class GraphDir(Dataset):
    def __init__(self, gc_dir: str):
        self.paths = sorted(Path(gc_dir).glob("*.pt"))
        if not self.paths:
            raise FileNotFoundError(f"No .pt in {gc_dir}")
        print(f"[DATA] {len(self.paths)} graphs in {gc_dir}")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        pt = self.paths[i]
        g  = safe_load(pt, map_location="cpu")
        g.__dict__["_aug_json_path"] = find_json(pt.stem)
        return g

def validate_interprocedural_coverage(g: HeteroData):
    stats = {}
    for (s, r, t) in g.edge_types:
        if r in ['CALL','ARG2PARAM','RET2CALL','RET2LHS']:
            stats[r] = int(get_edge_index(g,(s,r,t)).size(1))
    if sum(stats.values()) == 0:
        print("⚠️  No inter-procedural edges found!")
    else:
        print("✓ Inter-procedural edges:", stats)
    return stats

def make_inter_mask(g: HeteroData):
    st = g[NODE_TYPE]; N = st.num_nodes
    inter_rel = {'CALL','ARG2PARAM','RET2CALL','RET2LHS',
                 'CALL_REV','ARG2PARAM_REV','RET2CALL_REV','RET2LHS_REV'}
    inter_nodes = set()
    for (s, r, t) in g.edge_types:
        if r in inter_rel:
            ei = get_edge_index(g, (s,r,t))
            if ei.numel() > 0:
                inter_nodes.update(ei[0].tolist()); inter_nodes.update(ei[1].tolist())
    device = (st.x_text.device if hasattr(st, 'x_text') else (st.x.device if hasattr(st, 'x') else 'cpu'))
    mask = torch.zeros(N, dtype=torch.bool, device=device)
    if inter_nodes: mask[list(inter_nodes)] = True
    return mask

# ------------------------------ dynamic shrinker ------------------------------
MAX_NODES_FOR_GPU  = 4500
MAX_EDGES_FOR_GPU  = 15000

def shrink_graph_if_needed(g: HeteroData) -> HeteroData:
    st = g[NODE_TYPE]; N  = st.num_nodes
    total_edges = 0
    for et in g.edge_types:
        total_edges += int(get_edge_index(g, et).size(1))
    if N <= MAX_NODES_FOR_GPU and total_edges <= MAX_EDGES_FOR_GPU:
        return g
    keep = set()
    inter_rel = {'CALL','ARG2PARAM','RET2CALL','RET2LHS',
                 'CALL_REV','ARG2PARAM_REV','RET2CALL_REV','RET2LHS_REV'}
    for (s,r,t) in g.edge_types:
        if r in inter_rel:
            ei = get_edge_index(g, (s,r,t))
            if ei.numel()>0:
                keep.update(ei[0].tolist()); keep.update(ei[1].tolist())
    random.seed(0)
    if len(keep) < min(N, MAX_NODES_FOR_GPU):
        others = [i for i in range(N) if i not in keep]
        random.shuffle(others)
        need = min(N, MAX_NODES_FOR_GPU) - len(keep)
        keep.update(others[:max(0, need)])
    keep = sorted(list(keep))
    idx_map = {old:i for i, old in enumerate(keep)}
    g2 = HeteroData()
    if hasattr(st, "x_text"):
        g2[NODE_TYPE].x_text = st.x_text[keep].to(dtype=torch.float16, device=st.x_text.device)
    if hasattr(st, "x"):
        g2[NODE_TYPE].x = st.x[keep].to(dtype=torch.float16, device=st.x.device)
    if hasattr(st, "nid"):
        g2[NODE_TYPE].nid = st.nid[keep]
    g2[NODE_TYPE].num_nodes = len(keep)
    for (s,r,t) in g.edge_types:
        ei = get_edge_index(g, (s,r,t))
        if ei.numel() == 0:
            g2[(s,r,t)].edge_index = ei; continue
        src, dst = ei
        keep_mask = [(u in idx_map and v in idx_map) for u,v in zip(src.tolist(), dst.tolist())]
        if not any(keep_mask):
            g2[(s,r,t)].edge_index = torch.zeros((2,0), dtype=torch.long, device=ei.device); continue
        src_f = [src[i].item() for i, m in enumerate(keep_mask) if m]
        dst_f = [dst[i].item() for i, m in enumerate(keep_mask) if m]
        src_r = torch.tensor([idx_map[u] for u in src_f], dtype=torch.long, device=ei.device)
        dst_r = torch.tensor([idx_map[v] for v in dst_f], dtype=torch.long, device=ei.device)
        if src_r.numel() > MAX_EDGES_FOR_GPU:
            perm = torch.randperm(src_r.numel(), device=ei.device)[:MAX_EDGES_FOR_GPU]
            src_r = src_r[perm]; dst_r = dst_r[perm]
        g2[(s,r,t)].edge_index = torch.stack([src_r, dst_r], dim=0)
    g2.__dict__["_aug_json_path"] = getattr(g, "_aug_json_path", None)
    return g2

# ------------------------------ model ------------------------------
import torch.utils.checkpoint as cp

class CAGatBlock(nn.Module):
    def __init__(self, in_ch, out_ch, heads, edge_types, dropout=0.1, use_skip=True):
        super().__init__()
        assert out_ch % heads == 0
        self.edge_types = edge_types
        self.use_skip   = use_skip
        self.convs = nn.ModuleDict({
            f"{s}_{r}_{t}": GATConv(in_ch, out_ch//heads, heads=heads, concat=True,
                                    add_self_loops=False, dropout=dropout)
            for (s,r,t) in edge_types
        })
        self.gates = nn.ParameterDict({ f"{s}_{r}_{t}": nn.Parameter(torch.zeros(1)) for (s,r,t) in edge_types })
        self.norm  = nn.LayerNorm(out_ch)
        self.drop  = nn.Dropout(dropout)
        self.proj  = nn.Linear(in_ch, out_ch) if in_ch != out_ch else nn.Identity()
        self.skip_alpha = nn.Parameter(torch.tensor(0.5))
    def forward(self, x, g: HeteroData):
        out_dim = self.proj.out_features if isinstance(self.proj, nn.Linear) else x.size(1)
        out_accum = torch.zeros(x.size(0), out_dim, device=x.device, dtype=x.dtype)
        for (s,r,t) in self.edge_types:
            et = (s,r,t)
            ei = get_edge_index(g, et)
            if ei.numel()==0: continue
            key   = f"{s}_{r}_{t}"
            out = self.convs[key](x, ei)
            gate  = torch.sigmoid(self.gates[key])
            out_accum = out_accum + gate * out
        out_accum = self.drop(out_accum)
        res = self.proj(x)
        out_accum = self.skip_alpha * res + (1 - self.skip_alpha) * out_accum
        return self.norm(out_accum)

class CAGAT_ACC_InterProc(nn.Module):
    def __init__(self, in_dim, hidden, heads, layers, edge_types, dropout=0.1,
                 call_chain_attn=False, max_hops=3, sparse_threshold=5000):
        super().__init__()
        self.edge_types = edge_types
        self.hidden     = hidden
        self.projectors = nn.ModuleDict()
        self.blocks     = nn.ModuleList([
            CAGatBlock(hidden, hidden, heads, edge_types, dropout=dropout, use_skip=True)
            for _ in range(layers)
        ])
        self.intra_head = nn.Linear(hidden, hidden)
        he_init = nn.Linear(hidden, hidden); nn.init.xavier_uniform_(he_init.weight); nn.init.zeros_(he_init.bias)
        self.inter_head = he_init
        self.context_gru = nn.GRU(hidden, hidden, batch_first=True)
        self.lin_out   = nn.Linear(hidden, 1)
        self.call_chain_attn = call_chain_attn
        self.max_hops        = max_hops
        self.sparse_threshold= sparse_threshold
    def _project_in(self, x: torch.Tensor) -> torch.Tensor:
        d = x.size(1); key = f"proj_{d}"
        if key not in self.projectors:
            lin = nn.Linear(d, self.hidden, device=x.device, dtype=x.dtype)
            nn.init.kaiming_uniform_(lin.weight, a=math.sqrt(5))
            if lin.bias is not None:
                fan_in, _ = nn.init._calculate_fan_in_and_fan_out(lin.weight)
                bound = 1 / math.sqrt(fan_in); nn.init.uniform_(lin.bias, -bound, bound)
            self.projectors[key] = lin
        else:
            self.projectors[key] = self.projectors[key].to(device=x.device, dtype=x.dtype)
        return F.relu(self.projectors[key](x))
    def _call_chain_attn_sparse(self, call_ei, h, N):
        device = h.device
        idx = call_ei.to(device); vals = torch.ones(idx.size(1), device=device)
        A = torch.sparse_coo_tensor(idx, vals, size=(N, N))
        reach = A
        for _ in range(self.max_hops - 1):
            reach = reach + torch.sparse.mm(A, reach)
        neigh = reach.to_dense().clamp(max=1.0); neigh.fill_diagonal_(0)
        deg = neigh.sum(dim=1, keepdim=True).clamp(min=1)
        ctx = neigh @ h / deg
        return h + 0.1 * ctx
    def _call_chain_attn_bfs(self, call_ei, h, N):
        adj = defaultdict(list)
        for i in range(call_ei.size(1)):
            adj[call_ei[0,i].item()].append(call_ei[1,i].item())
        ctx = torch.zeros_like(h)
        for node_id in range(N):
            neighbors = []
            q = deque([(node_id,0)]); seen = {node_id}
            while q:
                cur, d = q.popleft()
                if d >= self.max_hops: continue
                for nb in adj.get(cur, []):
                    if nb not in seen:
                        seen.add(nb); neighbors.append(nb); q.append((nb, d+1))
            if neighbors: ctx[node_id] = h[neighbors].mean(dim=0)
        return h + 0.1 * ctx
    def forward(self, g: HeteroData):
        st = g[NODE_TYPE]; xs=[]
        if hasattr(st,"x_text"): xs.append(st.x_text)
        if hasattr(st,"x"):      xs.append(st.x.float())
        if len(xs) == 1: x = xs[0].to(device=xs[0].device, dtype=torch.float32)
        else: 
            target_device = xs[0].device; xs = [t.to(device=target_device, dtype=torch.float32) for t in xs]
            x = torch.cat(xs, dim=1)
        h = self._project_in(x)
        for blk in self.blocks:
            if USE_CHECKPOINT and h.requires_grad:
                h = cp.checkpoint(lambda inp: F.elu(blk(inp, g)), h, use_reentrant=False)
            else:
                h = F.elu(blk(h, g))
        inter_mask = make_inter_mask(g).to(h.device)
        h_mix = torch.where(inter_mask.unsqueeze(-1), self.inter_head(h), self.intra_head(h))
        et = (NODE_TYPE,'CALL',NODE_TYPE); ei = get_edge_index(g, et)
        if ei.numel() > 0:
            caller = h_mix[ei[0]]; callee = h_mix[ei[1]]
            seq = torch.stack([caller, callee], dim=1)  # [E,2,H]
            ctx,_ = self.context_gru(seq)
            callee_upd = ctx[:,1,:]
            deg = torch.zeros(h_mix.size(0), device=h_mix.device)
            deg.index_add_(0, ei[1], torch.ones(ei.size(1), device=h_mix.device))
            add = torch.zeros_like(h_mix); add.index_add_(0, ei[1], callee_upd)
            mask = (deg > 0).unsqueeze(-1)
            h_mix = torch.where(mask, h_mix + 0.2 * (add / deg.clamp(min=1).unsqueeze(-1)), h_mix)
        if self.call_chain_attn:
            call_ei = get_edge_index(g, (NODE_TYPE,'CALL',NODE_TYPE))
            if call_ei.numel() > 0:
                N = h_mix.size(0)
                h_mix = self._call_chain_attn_sparse(call_ei, h_mix, N) if N >= self.sparse_threshold else self._call_chain_attn_bfs(call_ei, h_mix, N)
        logit = self.lin_out(h_mix).squeeze(-1)
        return logit, h_mix

# ------------------------------ losses & metrics ------------------------------
def weighted_edge_smoothness(g: HeteroData, node_logit: torch.Tensor):
    loss = 0.0; count = 0
    for (s,r,t) in g.edge_types:
        w = EDGE_WEIGHTS.get(r, 0.0); 
        if w == 0.0: continue
        ei = get_edge_index(g, (s,r,t)); 
        if ei.numel()==0: continue
        u, v = ei[0], ei[1]
        loss = loss + w * F.l1_loss(node_logit[u], node_logit[v])
        count += 1
    return loss / max(1, count)

def inter_procedural_contrastive_loss(g: HeteroData, emb: torch.Tensor, temperature=0.1):
    loss = 0.0; count = 0
    for r in ['CALL','ARG2PARAM','RET2CALL']:
        et = (NODE_TYPE, r, NODE_TYPE)
        ei = get_edge_index(g, et)
        if ei.numel()==0: continue
        a = emb[ei[0]]; b = emb[ei[1]]
        pos_sim = F.cosine_similarity(a, b, dim=-1)
        loss = loss - torch.log(torch.sigmoid(pos_sim / temperature)).mean()
        count += 1
    return loss / max(1, count)

def path_supervision_loss(g: HeteroData, node_logit: torch.Tensor, paths_node_ids: list):
    if not paths_node_ids: 
        return node_logit.new_zeros(())
    st = g[NODE_TYPE]; id2pos = None
    if hasattr(st, "nid"):
        ids = st.nid.view(-1).tolist(); id2pos = {nid:i for i,nid in enumerate(ids)}
    loss = 0.0; cnt  = 0
    for path in paths_node_ids:
        idxs = []
        for nid in path:
            j = id2pos.get(_coerce_int(nid)) if id2pos is not None else (nid if (isinstance(nid, int) and 0 <= nid < st.num_nodes) else None)
            if j is None: idxs=[]; break
            idxs.append(j)
        if len(idxs) < 2: continue
        pl = node_logit[idxs]
        # enforce increasing scores root..sink
        loss = loss + sum(F.relu(pl[i] - pl[i+1]) for i in range(len(pl)-1)) / (len(pl)-1)
        cnt += 1
    return loss / max(1, cnt) if cnt>0 else node_logit.new_zeros(())

def pred_threshold(epoch, total_epochs):
    return 0.30 + 0.20 * (epoch / max(1, total_epochs))

def compute_inter_or_fallback_metrics(g, node_logit, y, thr=0.5):
    mask = make_inter_mask(g)
    if y.sum() == 0:
        return {"note": "no_positives"}
    if mask.sum() > 0 and (y[mask].sum() > 0):
        inter_logits = node_logit[mask]; inter_labels = y[mask]
        pred = (torch.sigmoid(inter_logits) > thr).float()
        tp = (pred * inter_labels).sum()
        prec = tp / (pred.sum() + 1e-8); rec  = tp / (inter_labels.sum() + 1e-8)
        acc  = (pred == inter_labels).float().mean()
        return {'inter_acc': float(acc), 'inter_precision': float(prec), 'inter_recall': float(rec)}
    pred = (torch.sigmoid(node_logit) > thr).float()
    tp = (pred * y).sum()
    prec = tp / (pred.sum() + 1e-8); rec  = tp / (y.sum() + 1e-8)
    acc  = (pred == y).float().mean()
    return {'inter_acc': float(acc), 'inter_precision': float(prec), 'inter_recall': float(rec), 'note': 'fallback_global'}

def get_loss_weights(epoch, total_epochs):
    prog = epoch / total_epochs
    return {
        'node':     LAMBDA_NODE_BCE_BASE,
        'smooth':   LAMBDA_EDGE_SMOOTH_BASE    + 0.15 * prog,  # 0.05 → 0.20
        'contrast': LAMBDA_INTER_CONTRAST_BASE + 0.15 * prog,  # 0.05 → 0.20
        'path':     LAMBDA_PATH_SUPER_BASE     + 0.20 * prog,  # 0.10 → 0.30
    }

# ------------------------------ Weak-supervision miner (fast + robust static seeds) ------------------------------
FAST_MINER          = True        # if False, also use regex over JSON node text
TIME_BUDGET_SEC     = 3.0         # per-graph budget
MAX_JSON_BYTES      = 2_000_000
MAX_FRONTIER        = 4096
MAX_VISITED         = 15000
MINER_MAX_HOPS      = 7
MINER_LIMIT_PATHS   = 12

# Static patterns kept (more robust, broader coverage). These are *additive* with structural seeds.
SINK_NAME_PATTERNS = [
    r"\b(exec|system|popen|spawn|CreateProcess|ShellExecute|eval|Runtime\.exec|ProcessBuilder)\b",
    r"\b(sql|execute(Query|Update)|Statement\.execute|rawQuery|PreparedStatement)\b",
    r"\b(send|write|print|fprintf|fwrite|OutputStream\.write|FileWriter|FileOutputStream)\b",
    r"\b(deserialize|ObjectInputStream|pickle\.loads|JSON\.parse)\b",
]
ROOT_NAME_PATTERNS = [
    r"\b(argv|stdin|getenv|getopt|request|req|input|params?|query|body|post|recv|read|Scanner|BufferedReader)\b",
    r"\b(getParameter|getHeader|getQueryString|getInputStream|readLine)\b",
]

_JSON_CACHE = {}; _JSON_KEYS  = []; _JSON_CACHE_LIMIT = 512
def _json_cache_get(path):
    if not path or not os.path.exists(path): return None
    try:
        if os.path.getsize(path) > MAX_JSON_BYTES: return None
    except Exception: pass
    if path in _JSON_CACHE: return _JSON_CACHE[path]
    j = load_aug_json(path)
    if j is not None:
        _JSON_CACHE[path] = j; _JSON_KEYS.append(path)
        if len(_JSON_KEYS) > _JSON_CACHE_LIMIT:
            old = _JSON_KEYS.pop(0); _JSON_CACHE.pop(old, None)
    return j

def _load_nodes_map_fast(json_path:str):
    j = _json_cache_get(json_path)
    if not isinstance(j, dict) or not isinstance(j.get("nodes"), list):
        return [], {}
    nodes = j["nodes"]; id2node = {}
    for n in nodes:
        if isinstance(n, dict) and "_id" in n:
            id2node[_coerce_int(n["_id"])] = n
    return nodes, id2node

def _node_text_from_json(n:dict) -> str:
    for k in ("code","name","methodFullName","signature","call","label","nodeType"):
        v = n.get(k)
        if isinstance(v,str) and v.strip(): return v
    return ""

def _regex_any(patterns, s):
    if not isinstance(s, str): return False
    for p in patterns:
        if re.search(p, s, flags=re.I): return True
    return False

def _build_adj_fast(g: HeteroData):
    allowed_rels = {'CALL','ARG2PARAM','RET2CALL','RET2LHS','DFG'}
    adj = defaultdict(list)
    for (s,r,t) in g.edge_types:
        if r not in allowed_rels: continue
        ei = get_edge_index(g,(s,r,t))
        if ei.numel()==0: continue
        src, dst = ei
        buckets = defaultdict(list)
        for u,v in zip(src.tolist(), dst.tolist()): buckets[u].append(v)
        for u, vs in buckets.items():
            adj[u].extend(vs[:64])  # cap out-degree
    return adj

def _bfs_paths_bounded(adj, starts, targets, max_hops=5, limit=8, time_budget=1.0):
    t0 = time.monotonic(); paths = []; targets = set(targets)
    visited_budget = 0
    for s in starts:
        q = deque([(s, [s])])
        while q and len(paths) < limit:
            if (time.monotonic() - t0) > time_budget: return paths
            cur, path = q.popleft()
            if len(path) - 1 > max_hops: continue
            if cur in targets and len(path) > 1:
                paths.append(path[:])
                if len(paths) >= limit: break
            nb_list = adj.get(cur, [])[:64]
            for nb in nb_list:
                if nb in path: continue
                q.append((nb, path + [nb])); visited_budget += 1
                if visited_budget > MAX_VISITED: return paths
            if len(q) > MAX_FRONTIER:
                while len(q) > MAX_FRONTIER: q.pop()
    return paths

def mine_roots_sinks_and_paths(g: HeteroData, json_path: str,
                               max_hops=MINER_MAX_HOPS, limit_paths=MINER_LIMIT_PATHS):
    t0 = time.monotonic()
    roots, sinks, paths = [], [], []
    # (A) structural seeds (always available; robust)
    # sinks = CALL callees + RET2CALL/RET2LHS targets
    for r in ('RET2LHS','RET2CALL','CALL'):
        ei = get_edge_index(g,(NODE_TYPE,r,NODE_TYPE))
        if ei.numel()>0: sinks.extend(ei[1].unique().tolist())
    sinks = list(set(sinks))[:256]
    # roots = ARG2PARAM sources + high-outdegree DFG sources
    ei = get_edge_index(g,(NODE_TYPE,'ARG2PARAM',NODE_TYPE))
    if ei.numel()>0: roots = ei[0].unique().tolist()[:256]
    dfg = get_edge_index(g,(NODE_TYPE,'DFG',NODE_TYPE))
    if dfg.numel()>0:
        deg = torch.zeros(g[NODE_TYPE].num_nodes, device=dfg.device); deg.index_add_(0, dfg[0], torch.ones(dfg.size(1), device=deg.device))
        high = torch.topk(deg, k=min(128, deg.numel())).indices.tolist()
        roots = list(set((roots or []) + high))
    # (B) optional lexical seeds from JSON node text (if enabled)
    if (not FAST_MINER) and json_path:
        nodes, id2node = _load_nodes_map_fast(json_path)
        if hasattr(g[NODE_TYPE], "nid") and id2node:
            g_ids = g[NODE_TYPE].nid.view(-1).tolist()
            hit_roots=set(); hit_sinks=set()
            for nid in g_ids[:4096]:
                n = id2node.get(_coerce_int(nid)); 
                if not isinstance(n, dict): continue
                txt = _node_text_from_json(n)
                if _regex_any(SINK_NAME_PATTERNS, txt): hit_sinks.add(nid)
                if _regex_any(ROOT_NAME_PATTERNS, txt): hit_roots.add(nid)
            if hit_sinks: sinks = list(set(sinks + list(hit_sinks)))
            if hit_roots: roots = list(set(roots + list(hit_roots)))
    # Mine paths between roots and sinks
    if roots and sinks:
        adj = _build_adj_fast(g)
        rem = max(0.2, TIME_BUDGET_SEC - (time.monotonic() - t0))
        paths = _bfs_paths_bounded(adj, starts=roots, targets=sinks,
                                   max_hops=max_hops, limit=limit_paths, time_budget=rem)
    # fallback: stub edges
    if not paths and roots and sinks:
        for r in roots[:4]:
            for s in sinks[:4]:
                if r != s:
                    paths.append([r, s])
                    if len(paths) >= limit_paths: break
            if len(paths) >= limit_paths: break
    return roots, sinks, paths

def _ensure_inter_pos(g, y_soft, paths):
    """Guarantee at least one positive inside the inter-procedural slice."""
    mask = make_inter_mask(g)
    if (y_soft[mask] > 0.5).any(): 
        return y_soft, paths
    # try to promote a node on a mined path that lies in the slice
    promoted = False
    id2pos = None
    if hasattr(g[NODE_TYPE],"nid"):
        ids = g[NODE_TYPE].nid.view(-1).tolist(); id2pos = {nid:i for i,nid in enumerate(ids)}
    for path in paths:
        for nid in path[::-1]:
            j = id2pos.get(_coerce_int(nid)) if id2pos is not None else (nid if isinstance(nid,int) else None)
            if j is not None and j < y_soft.numel() and mask[j]:
                y_soft[j] = max(y_soft[j], 0.9); promoted = True; break
        if promoted: break
    # if still none, pick high-degree CALL callee
    if not promoted:
        ei = get_edge_index(g,(NODE_TYPE,'CALL',NODE_TYPE))
        if ei.numel()>0:
            deg = torch.zeros(y_soft.numel(), device=y_soft.device)
            deg.index_add_(0, ei[1], torch.ones(ei.size(1), device=y_soft.device))
            j = int(deg.argmax().item())
            if mask[j]: y_soft[j] = max(y_soft[j], 0.9)
    return y_soft, paths

def build_labels_with_weak_supervision(g: HeteroData):
    json_path = getattr(g, "_aug_json_path", None)
    # 1) try hard labels
    y = node_sink_labels_from_json(json_path, g); hard_paths = read_paths_from_json(json_path)
    if y is not None and (float(y.sum().item()) > 0 or (hard_paths and len(hard_paths)>0)):
        return y.to(y.device), hard_paths, {"mode":"hard_json"}
    # 2) mine paths
    roots, sinks, mined_paths = mine_roots_sinks_and_paths(g, json_path,
                                                           max_hops=MINER_MAX_HOPS,
                                                           limit_paths=MINER_LIMIT_PATHS)
    st = g[NODE_TYPE]
    device = (st.x_text.device if hasattr(st,"x_text") else (st.x.device if hasattr(st,"x") else "cpu"))
    N = st.num_nodes
    y_soft = torch.zeros(N, dtype=torch.float32, device=device)
    # soft labels along paths: ramp 0.3 → 0.9
    if hasattr(st,"nid"):
        ids = st.nid.view(-1).tolist(); id2pos = {nid:i for i,nid in enumerate(ids)}
        for path in mined_paths:
            L = max(2, len(path)); 
            for k, nid in enumerate(path):
                j = id2pos.get(_coerce_int(nid)); 
                if j is None or j >= N: continue
                tgt = 0.3 + 0.6 * (k / (L-1))  # root ~0.3 … sink ~0.9
                y_soft[j] = max(y_soft[j], tgt)
    # ensure at least one inter-proc positive
    y_soft, mined_paths = _ensure_inter_pos(g, y_soft, mined_paths)
    return y_soft, mined_paths, {"mode":"weak_mined", "roots": len(roots), "sinks": len(sinks), "paths": len(mined_paths)}

print(f"GC_DIR={GC_DIR}")
print(f"UNIFIED_JSON_DIRS={UNIFIED_JSON_DIRS}")
print(f"AUG_JSON_DIRS={AUG_JSON_DIRS}")

# ------------------------------ training ------------------------------
def train(
    gc_dir=GC_DIR,
    logdir=LOGDIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    hidden=HIDDEN,
    heads=HEADS,
    layers=LAYERS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    save_every=SAVE_EVERY_STEPS,
    warmup_max_steps=None,    # if set, run bounded warm-up
):
    ds = GraphDir(gc_dir)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=lambda xs: xs[0], pin_memory=False)

    sample = safe_load(ds.paths[0], map_location="cpu")
    st = sample[NODE_TYPE]
    in_dim = (st.x_text.size(1) if hasattr(st, "x_text") else 0) + (st.x.size(1) if hasattr(st, "x") else 0)
    etypes = tuple(sample.edge_types)

    if hidden % heads != 0:
        new_hidden = (hidden // heads + 1) * heads
        print(f"[CFG] HIDDEN={hidden} not divisible by HEADS={heads} → adjusting to {new_hidden}")
        hidden = new_hidden

    model = CAGAT_ACC_InterProc(
        in_dim=in_dim, hidden=hidden, heads=heads, layers=layers,
        edge_types=etypes, dropout=DROPOUT,
        call_chain_attn=CALL_CHAIN_ATTN, max_hops=CALL_CHAIN_MAX_HOPS,
        sparse_threshold=CALL_CHAIN_SPARSE_NTHRESH
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    os.makedirs(logdir, exist_ok=True)
    ckpt_path       = os.path.join(logdir, "ckpt_cagat_acc_interproc.pt")
    best_path       = os.path.join(logdir, "best_model.pt")
    summary_path    = os.path.join(logdir, "train_summary.json")
    log_jsonl_path  = os.path.join(logdir, "training_log.jsonl")

    best_inter_recall = 0.0
    log_jsonl_fh = open(log_jsonl_path, "a", encoding="utf-8")

    def save(tag="SAVE", steps=0, loss_avg=0.0, extras=None, best=False):
        payload = {"model": model.state_dict()}
        torch.save(payload, best_path if best else ckpt_path)
        out = {"loss_avg": loss_avg, "steps": steps, "epochs": epochs}
        if isinstance(extras, dict): out.update(extras)
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(out, f, indent=2)
        tqdm.write(f"[{tag}] -> {(best_path if best else ckpt_path)}")

    atexit.register(lambda: save("ATEXIT", steps=0, loss_avg=0.0))
    atexit.register(lambda: log_jsonl_fh.close())

    steps, loss_sum = 0, 0.0
    graphs_seen = graphs_with_pos = graphs_with_inter_pos = 0

    total_steps = (warmup_max_steps if warmup_max_steps is not None else epochs * len(dl))
    print(f"[TRAIN] Device: {DEVICE}")
    print(f"[TRAIN] Sample: N={st.num_nodes}, in_dim={in_dim}, edge_types={len(etypes)}")
    validate_interprocedural_coverage(sample)

    phase = "Warmup" if warmup_max_steps is not None else "Training"
    pbar = tqdm(total=total_steps, desc=f"{phase}", unit="step", dynamic_ncols=True)

    try:
        epoch_iter = [1] if warmup_max_steps is not None else range(1, epochs + 1)
        for ep in epoch_iter:
            w = get_loss_weights(ep, epochs if warmup_max_steps is None else 1)
            for i, g in enumerate(dl, 1):
                if warmup_max_steps is not None and steps >= warmup_max_steps:
                    break
                tried_shrink = False
                while True:
                    try:
                        g_work = g.to(DEVICE, non_blocking=False)
                        with autocast_ctx(enabled=USE_AMP):
                            node_logit, h = model(g_work)
                            st_g = g_work[NODE_TYPE]

                            # Hard labels → else inter-aware weak labels
                            y = node_sink_labels_from_json(getattr(g_work, "_aug_json_path", None), g_work)
                            paths = read_paths_from_json(getattr(g_work, "_aug_json_path", None))
                            label_info = None
                            if (y is None) or (float(y.sum().item()) == 0 and not paths):
                                y, paths, label_info = build_labels_with_weak_supervision(g_work)

                            y = y.to(node_logit.device)
                            inter_mask = make_inter_mask(g_work)

                            graphs_seen += 1
                            pos_all   = int((y > 0.5).sum().item())
                            pos_inter = int(((y > 0.5).float() * inter_mask.float()).sum().item())
                            if pos_all > 0: graphs_with_pos += 1
                            if pos_inter > 0: graphs_with_inter_pos += 1

                            # BCE supports soft targets
                            pos = (y > 0.5).sum()
                            neg = y.numel() - pos
                            pos_weight = (neg / (pos + 1e-6)).clamp_(1.0, 100.0)
                            node_loss = F.binary_cross_entropy_with_logits(node_logit, y, pos_weight=pos_weight)

                            # Aux losses
                            smooth    = weighted_edge_smoothness(g_work, node_logit)
                            contra    = inter_procedural_contrastive_loss(g_work, h)
                            path_loss = path_supervision_loss(g_work, node_logit, paths) if paths else node_logit.new_zeros(())

                            loss = (w['node'] * node_loss + w['smooth'] * smooth +
                                    w['contrast'] * contra + w['path'] * path_loss)

                        # step
                        opt.zero_grad(set_to_none=True)
                        SCALER.scale(loss).backward()
                        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                        SCALER.step(opt); SCALER.update()

                        # metrics
                        thr = pred_threshold(ep, epochs if warmup_max_steps is None else 1)
                        metrics = compute_inter_or_fallback_metrics(g_work, node_logit.detach(), (y > 0.5).float(), thr=thr)

                        steps += 1; loss_sum += float(loss.detach().cpu())
                        postfix = dict(ep=ep, step=steps, loss=f"{loss_sum/steps:.4f}",
                                       i_acc=(f"{metrics.get('inter_acc', 0):.3f}" if metrics else "N/A"),
                                       i_rec=(f"{metrics.get('inter_recall', 0):.3f}" if metrics else "N/A"))
                        if "note" in metrics: postfix["note"] = metrics["note"]
                        pbar.set_postfix(**postfix); pbar.update(1)

                        # jsonl
                        if (steps % LOG_JSONL_EVERY) == 0:
                            log_line = {
                                "epoch": ep, "step": steps,
                                "loss": float(loss.detach().cpu()),
                                "node_loss": float(node_loss.detach().cpu()),
                                "smooth": float(smooth.detach().cpu()) if torch.is_tensor(smooth) else float(smooth),
                                "contrast": float(contra.detach().cpu()) if torch.is_tensor(contra) else float(contra),
                                "path_loss": float(path_loss.detach().cpu()) if torch.is_tensor(path_loss) else float(path_loss),
                                "pos_any": pos_all, "pos_inter": pos_inter,
                                **{k: float(v) for k,v in metrics.items() if isinstance(v, (int,float))}
                            }
                            if isinstance(metrics.get("note"), str): log_line["note"] = metrics["note"]
                            if label_info: log_line["label_info"] = label_info
                            log_jsonl_fh.write(json.dumps(log_line) + "\n"); log_jsonl_fh.flush()

                        inter_rec = float(metrics.get('inter_recall', 0.0))
                        if inter_rec > best_inter_recall and warmup_max_steps is None:
                            best_inter_recall = inter_rec
                            save("BEST (inter_recall↑)", steps=steps, loss_avg=loss_sum/steps, extras=metrics, best=True)

                        # tidy
                        del node_logit, h, y, loss, node_loss, smooth, contra, path_loss, g_work
                        if (i % CLEAR_CACHE_EVERY) == 0 and DEVICE.type == "cuda":
                            torch.cuda.empty_cache()
                        break

                    except RuntimeError as e:
                        if 'out of memory' in str(e).lower():
                            torch.cuda.empty_cache()
                            if not tried_shrink:
                                g = shrink_graph_if_needed(g); tried_shrink = True
                                tqdm.write("[INFO] Retrying with shrunken graph to avoid OOM.")
                                continue
                            elif SKIP_ON_OOM:
                                tqdm.write("[WARN] Skipping one graph due to CUDA OOM even after shrink."); break
                            else:
                                raise
                        else:
                            raise

            if warmup_max_steps is not None and steps >= warmup_max_steps:
                break

    except Exception:
        save("FINAL (EXC)", steps=steps, loss_avg=(loss_sum / max(1, steps)))
        traceback.print_exc(); raise
    finally:
        save("FINAL", steps=steps, loss_avg=(loss_sum / max(1, steps)))
        pbar.close(); log_jsonl_fh.close()

    # ----------- training report -----------
    report = {
        "phase": ("warmup" if warmup_max_steps is not None else "train"),
        "steps": steps,
        "avg_loss": loss_sum/max(1,steps),
        "graphs_seen": graphs_seen,
        "graphs_with_any_pos": graphs_with_pos,
        "graphs_with_inter_pos": graphs_with_inter_pos,
        "best_inter_recall": (best_inter_recall if warmup_max_steps is None else None),
        "log_jsonl": log_jsonl_path,
        "summary_json": summary_path,
        "ckpt_last": os.path.join(logdir, "ckpt_cagat_acc_interproc.pt"),
        "ckpt_best": (os.path.join(logdir, "best_model.pt") if warmup_max_steps is None else None),
    }
    # Persist report
    with open(os.path.join(logdir, "training_report.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    if warmup_max_steps is not None:
        print(f"[WARMUP] summary: {report}")
    else:
        print(f"[TRAIN] summary: {report}")

# ------------------------------ Convenience wrappers ------------------------------
def warmup_check(max_steps=300):
    print("[WARMUP] starting…")
    train(warmup_max_steps=max_steps)

def full_run():
    print("[FULL] starting…")
    train()

# ========================== toggles (set & run cell) ==========================
DO_WARMUP   = False    # quick sanity
DO_FULL_RUN = True   # full training later

# Miner knobs (you can tweak live)
FAST_MINER      = True     # False => also use regex on JSON node text
TIME_BUDGET_SEC = 2.0      # try 3.0–5.0 if your GPU/CPU can handle

# Make log dir
os.makedirs(LOGDIR, exist_ok=True)

# Execute
if DO_WARMUP:
    warmup_check(max_steps=300)
if DO_FULL_RUN:
    full_run()


c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[AMP] torch=2.4.1+cu121 device=cuda
GC_DIR=Dataset/train/hetero_ready_gcbert
UNIFIED_JSON_DIRS=['Dataset/train/unified_json', 'Dataset/train/unified']
AUG_JSON_DIRS=['Dataset/train/unified_aug', 'Dataset/train/unified']
[FULL] starting…
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert
[TRAIN] Device: cuda
[TRAIN] Sample: N=3141, in_dim=793, edge_types=14
✓ Inter-procedural edges: {'CALL': 608, 'ARG2PARAM': 1097, 'RET2CALL': 608, 'RET2LHS': 96}


Training:   0%|          | 0/17190 [00:00<?, ?step/s]c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\.venv\Lib\site-packages\torch\utils\checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]
Training:   0%|          | 1/17190 [00:00<3:18:17,  1.44step/s, ep=1, i_acc=0.155, i_rec=1.000, loss=0.8332, step=1]

[BEST (inter_recall↑)] -> out/cagat_acc_interproc_train\best_model.pt


Training: 100%|██████████| 17190/17190 [5:17:14<00:00,  1.11s/step, ep=5, i_acc=0.292, i_rec=1.000, loss=0.9529, step=17190]                     

[FINAL] -> out/cagat_acc_interproc_train\ckpt_cagat_acc_interproc.pt
[TRAIN] summary: {'phase': 'train', 'steps': 17190, 'avg_loss': 0.9529057450354134, 'graphs_seen': 17190, 'graphs_with_any_pos': 17060, 'graphs_with_inter_pos': 17060, 'best_inter_recall': 1.0, 'log_jsonl': 'out/cagat_acc_interproc_train\\training_log.jsonl', 'summary_json': 'out/cagat_acc_interproc_train\\train_summary.json', 'ckpt_last': 'out/cagat_acc_interproc_train\\ckpt_cagat_acc_interproc.pt', 'ckpt_best': 'out/cagat_acc_interproc_train\\best_model.pt'}


In [17]:
# ==== CUDA-first training with guaranteed CPU fallback (no DataLoader), strict CCS/CFAM, multi-root beam, saves artifacts ====
import os, json, math, time, random, warnings, re, gc
from pathlib import Path
from collections import defaultdict, deque
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True,max_split_size_mb:64,garbage_collection_threshold:0.6")

import torch, torch.nn as nn, torch.nn.functional as F

warnings.filterwarnings("ignore", message="IProgress not found")
for m in ("pyg-lib","torch-scatter","torch-sparse","torch-cluster","torch-spline-conv"):
    warnings.filterwarnings("ignore", message=m)

try:
    from torch_geometric.data import HeteroData
except Exception as e:
    raise RuntimeError("torch-geometric must be installed (pip install torch-geometric)") from e

# ------------------ Device ------------------
def pick_device_cuda_first():
    if torch.cuda.is_available():
        try:
            dev=torch.device("cuda")
            _=torch.empty(1, device=dev); del _
            print(f"[env] torch={torch.__version__} device=cuda  GPU:", torch.cuda.get_device_name(0))
            try: torch.backends.cudnn.benchmark=True
            except: pass
            try: torch.set_float32_matmul_precision("high")
            except: pass
            return dev
        except RuntimeError:
            print("CUDA present but not usable → using CPU.")
    else:
        print(f"[env] torch={torch.__version__} device=cpu")
    return torch.device("cpu")

DEVICE = pick_device_cuda_first()
AMP_ENABLED = (DEVICE.type=="cuda")

# ------------------ Dirs & run folder ------------------
TRAIN_DIR = "Dataset/train/hetero_ready_gcbert"
VALID_DIR = "Dataset/valid/hetero_ready_gcbert"
TEST_DIR  = "Dataset/test/hetero_ready_gcbert"
OUT_ROOT  = Path("out") / f"cvul_run_{int(time.time())}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Saving to:", OUT_ROOT)

# ------------------ Knobs ------------------
EPOCHS=5
LR=2e-3; WD=1e-4
DROPOUT=0.10
NEG_POS_RATIO=30
ALPHA_NODE=0.7
BEAM_W=32; MAX_HOPS=6
NODE_TYPE="node"

EDGE_KEEP = {'AST','CFG','DFG','CALL','ARG2PARAM','RET2CALL','RET2LHS',
             'AST_REV','CFG_REV','DFG_REV','CALL_REV','ARG2PARAM_REV','RET2CALL_REV','RET2LHS_REV'}
SLICE_FWD = {'DFG','CALL','ARG2PARAM','RET2CALL','RET2LHS','CFG'}
SLICE_REV = {f"{r}_REV" for r in SLICE_FWD}

# ------------------ Safe load for PyG ------------------
def _add_safe_globals():
    try:
        from torch.serialization import add_safe_globals
        from torch_geometric.data.storage import BaseStorage
        from torch_geometric.data import HeteroData as _HD
        add_safe_globals([BaseStorage, _HD])
    except Exception: pass

def safe_load(path):
    _add_safe_globals()
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

# ------------------ Dataset helpers ------------------
def _coerce_int(x):
    if isinstance(x,int): return x
    if isinstance(x,str) and x.strip().lstrip("-").isdigit():
        try: return int(x)
        except: return x
    return x

def near_json(pt_path: Path)->Optional[str]:
    base=pt_path.stem
    for d in [
        "Dataset/train/unified_aug","Dataset/train/unified_json","Dataset/train/unified",
        "Dataset/valid/unified_aug","Dataset/valid/unified_json","Dataset/valid/unified",
        "Dataset/test/unified_aug","Dataset/test/unified_json","Dataset/test/unified",
        "notebooks/Dataset/train/unified_aug"
    ]:
        p=Path(d)
        if not p.exists(): continue
        for suf in (".json",".aug.json",".unified.json",".jsonl"):
            fp=p/f"{base}{suf}"
            if fp.exists(): return str(fp)
    return None

def _json_load(path: Optional[str])->Optional[dict]:
    if not path or not os.path.exists(path): return None
    try: return json.loads(Path(path).read_text("utf-8"))
    except Exception:
        try:
            txt=Path(path).read_text("utf-8", errors="ignore")
            txt=re.sub(r'//.*?$', '', txt, flags=re.M)
            txt=re.sub(r'/\*.*?\*/','', txt, flags=re.S)
            txt=re.sub(r',\s*(\}|\])', r'\1', txt)
            return json.loads(txt)
        except Exception:
            return None

def _map_nid_list_to_idx(g: HeteroData, nids: List[Any]) -> List[int]:
    if not hasattr(g[NODE_TYPE],"nid"): return []
    id2pos={nid:i for i,nid in enumerate(g[NODE_TYPE].nid.view(-1).tolist())}
    out=[]
    for a in nids:
        ai=_coerce_int(a)
        if isinstance(ai,int) and ai in id2pos: out.append(id2pos[ai])
    return out

def read_ground_truth(g: HeteroData) -> dict:
    j=_json_load(getattr(g,"_aug_json_path", None))
    st=g[NODE_TYPE]; N=st.num_nodes
    out={"y":None,"roots":[],"sinks":[],"paths_idx":[],"causal_nodes_idx":[],"spurious_nodes_idx":[]}
    if j is None: return out
    # labels
    if isinstance(j.get("nodes"), list) and hasattr(st,"nid") and len(j["nodes"])==N:
        ids=[_coerce_int(n.get("_id")) if isinstance(n,dict) else None for n in j["nodes"]]
        pos=set()
        for k in ("positives","positive_nodes","vul_nodes","sinks","vulnerabilities"):
            if isinstance(j.get(k), list):
                for v in j[k]:
                    vi=_coerce_int(v)
                    if isinstance(vi,int): pos.add(vi)
        y=torch.tensor([1.0 if (isinstance(i,int) and i in pos) else 0.0 for i in ids], dtype=torch.float32)
        out["y"]=y.to(st.x.device if hasattr(st,"x") else (st.x_text.device if hasattr(st,"x_text") else "cpu"))
    else:
        pos=set()
        for k in ("positives","positive_nodes","vul_nodes","sinks","vulnerabilities"):
            if isinstance(j.get(k), list):
                for v in j[k]:
                    vi=_coerce_int(v)
                    if isinstance(vi,int): pos.add(vi)
        if pos and hasattr(st,"nid"):
            ids=st.nid.view(-1).tolist()
            out["y"]=torch.tensor([1.0 if i in pos else 0.0 for i in ids], dtype=torch.float32, device=(st.x.device if hasattr(st,"x") else "cpu"))
    # roots/sinks/paths
    if isinstance(j.get("roots"), list): out["roots"]=_map_nid_list_to_idx(g, j["roots"])
    if isinstance(j.get("sinks"), list): out["sinks"]=_map_nid_list_to_idx(g, j["sinks"])
    if isinstance(j.get("vulnerable_paths"), list):
        for pth in j["vulnerable_paths"]:
            if isinstance(pth,(list,tuple)) and pth:
                out["paths_idx"].append(_map_nid_list_to_idx(g, list(pth)))
    # explicit causal/spurious annotations (optional)
    if isinstance(j.get("causal_nodes"), list):   out["causal_nodes_idx"]=_map_nid_list_to_idx(g, j["causal_nodes"])
    if isinstance(j.get("spurious_nodes"), list): out["spurious_nodes_idx"]=_map_nid_list_to_idx(g, j["spurious_nodes"])
    return out

# ------------------ Graph + slicing utils ------------------
def ei_of(g: HeteroData, rel: str) -> torch.Tensor:
    for (s,r,t) in g.edge_types:
        if r==rel:
            ei=getattr(g[(s,r,t)], "edge_index", None)
            if ei is None and hasattr(g[(s,r,t)], "adj_t"):
                row,col,_=g[(s,r,t)].adj_t.coo(); ei=torch.stack([col,row], dim=0)
            if ei is None:
                dev=(g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else (g[NODE_TYPE].x_text.device if hasattr(g[NODE_TYPE],"x_text") else "cpu"))
                ei=torch.zeros((2,0), dtype=torch.long, device=dev)
            return ei
    dev=(g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else (g[NODE_TYPE].x_text.device if hasattr(g[NODE_TYPE],"x_text") else "cpu"))
    return torch.zeros((2,0), dtype=torch.long, device=dev)

def build_adj_dicts(g: HeteroData):
    fwd=defaultdict(list); rev=defaultdict(list)
    st=g[NODE_TYPE]; N=st.num_nodes
    for r in SLICE_FWD|SLICE_REV:
        ei=ei_of(g, r)
        if ei.numel()==0: continue
        s,d=ei[0].tolist(), ei[1].tolist()
        for u,v in zip(s,d):
            if 0<=u<N and 0<=v<N:
                fwd[u].append(v); rev[v].append(u)
    return fwd, rev

def slice_causal_nodes(g: HeteroData, roots: List[int], sinks: List[int], paths_idx: List[List[int]]) -> Tuple[List[int], Dict[str,torch.Tensor]]:
    st=g[NODE_TYPE]; N=st.num_nodes
    if not roots and paths_idx: roots=sorted(set([p[0] for p in paths_idx if p and 0<=p[0]<N]))
    if not sinks and paths_idx: sinks=sorted(set([p[-1] for p in paths_idx if p and 0<=p[-1]<N]))
    if not roots or not sinks:  return [], {}
    fwd,rev=build_adj_dicts(g)
    F=set(); Q=deque(roots)
    while Q:
        u=Q.popleft()
        if u in F: continue
        F.add(u)
        for v in fwd.get(u, []):
            if v not in F: Q.append(v)
    B=set(); Q=deque(sinks)
    while Q:
        u=Q.popleft()
        if u in B: continue
        B.add(u)
        for v in rev.get(u, []):
            if v not in B: Q.append(v)
    causal=sorted(list(F.intersection(B)))
    edges={}; 
    for (s,r,t) in g.edge_types:
        if r in EDGE_KEEP:
            edges[r]=ei_of(g, r)
    return causal, edges

def build_slice_edge_mask(g: HeteroData, causal_nodes: List[int]) -> Dict[str, torch.Tensor]:
    cn=set(causal_nodes); out={}
    for (s,r,t) in g.edge_types:
        if r not in EDGE_KEEP: continue
        ei=ei_of(g, r)
        if ei.numel()==0:
            out[r]=torch.zeros(0, dtype=torch.bool, device=(g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE],"x") else "cpu")); continue
        u,v=ei[0], ei[1]
        on_u = torch.isin(u, torch.tensor(list(cn), device=u.device)) if hasattr(torch, "isin") else torch.stack([u.eq(c) for c in cn]).any(0)
        on_v = torch.isin(v, torch.tensor(list(cn), device=v.device)) if hasattr(torch, "isin") else torch.stack([v.eq(c) for c in cn]).any(0)
        out[r]=on_u & on_v
    return out

# ------------------ Feature scan / padding ------------------
def scan_feature_dims(dirs: List[str]) -> Tuple[int,int,Tuple[Tuple[str,str,str],...]]:
    max_text=0; max_num=0; etypes_seen=set()
    for d in dirs:
        p=Path(d)
        if not p.exists(): continue
        for pt in p.glob("*.pt"):
            g=safe_load(pt)
            st=g[NODE_TYPE]
            if hasattr(st,"x_text"): max_text=max(max_text, int(st.x_text.size(1)))
            if hasattr(st,"x"):      max_num =max(max_num,  int(st.x.size(1)))
            for et in g.edge_types:
                if et[1] in EDGE_KEEP: etypes_seen.add(et)
    return max_text, max_num, tuple(sorted(etypes_seen))

def pad_to(feat: torch.Tensor, target: int) -> torch.Tensor:
    if feat.size(1)==target: return feat
    if feat.size(1)<target:
        pad = feat.new_zeros(feat.size(0), target - feat.size(1))
        return torch.cat([feat, pad], dim=1)
    else:
        return feat[:, :target]

# ------------------ Model ------------------
class GatedRGCN(nn.Module):
    def __init__(self, hidden: int, edge_types: Tuple[Tuple[str,str,str],...], dropout=0.1):
        super().__init__()
        self.edge_types=edge_types
        self.self_lin=nn.Linear(hidden, hidden)
        self.msg=nn.ModuleDict({f"{s}_{r}_{t}": nn.Linear(hidden, hidden, bias=False) for (s,r,t) in edge_types})
        self.gate=nn.ParameterDict({f"{s}_{r}_{t}": nn.Parameter(torch.zeros(1)) for (s,r,t) in edge_types})
        self.norm=nn.LayerNorm(hidden); self.drop=nn.Dropout(dropout)
    def _layer(self, h, g: HeteroData):
        out=self.self_lin(h)
        for (s,r,t) in self.edge_types:
            ei=ei_of(g, r); 
            if ei.numel()==0: continue
            u,v=ei[0], ei[1]
            m=self.msg[f"{s}_{r}_{t}"](h[u])
            agg=torch.zeros_like(h); agg.index_add_(0, v, m)
            deg=torch.zeros(h.size(0), device=h.device); deg.index_add_(0, v, torch.ones(ei.size(1), device=h.device))
            out = out + torch.sigmoid(self.gate[f"{s}_{r}_{t}"]) * (agg/deg.clamp_min(1).unsqueeze(-1))
        # thin inter-proc DFG 2-hop summary
        try:
            ei=ei_of(g, "DFG")
            if ei.numel()>0:
                u,v=ei[0], ei[1]
                one=torch.zeros_like(h); one.index_add_(0, v, h[u])
                two=torch.zeros_like(h); two.index_add_(0, v, one[u])
                out = out + 0.10*two
        except Exception: pass
        out=self.drop(out)
        return self.norm(F.silu(out))
    def forward(self, h, g: HeteroData):
        return self._layer(h, g)

class CausalVulNet(nn.Module):
    def __init__(self, in_dim:int, hidden:int, layers:int, edge_types:Tuple[Tuple[str,str,str],...],
                 d_text:int, d_num:int):
        super().__init__()
        self.edge_types=edge_types
        self.d_text=d_text; self.d_num=d_num
        self.proj=nn.Linear(in_dim, hidden)
        self.blocks=nn.ModuleList([GatedRGCN(hidden, edge_types, dropout=DROPOUT) for _ in range(layers)])
        self.node_head=nn.Linear(hidden,1)          # node logit
        self.next_head=nn.Linear(hidden*2,1)        # next-hop edge scorer
    def encode(self, g: HeteroData):
        st=g[NODE_TYPE]; xs=[]
        if hasattr(st,"x_text"): xs.append(pad_to(st.x_text.float(), self.d_text))
        else: xs.append(torch.zeros((st.num_nodes, self.d_text), device=(st.x.device if hasattr(st,"x") else "cpu")))
        if hasattr(st,"x"): xs.append(pad_to(st.x.float(), self.d_num))
        else: xs.append(torch.zeros((st.num_nodes, self.d_num), device=xs[0].device))
        x=torch.cat(xs, dim=1)
        h=F.relu(self.proj(x))
        for blk in self.blocks: h=blk(h,g)
        return h
    def forward(self, g: HeteroData):
        h=self.encode(g)
        logit=self.node_head(h).squeeze(-1)
        return logit, h
    def edge_guidance(self, h, edges: Dict[str,torch.Tensor]):
        out={}
        for (s,r,t) in self.edge_types:
            ei=edges.get(r, None)
            if ei is None or (isinstance(ei, torch.Tensor) and ei.numel()==0):
                out[r]=torch.empty(0, device=h.device); continue
            u,v=ei[0], ei[1]
            out[r]=self.next_head(torch.cat([h[u],h[v]], dim=-1)).squeeze(-1)
        return out

# ------------------ Losses ------------------
class FocalBCELoss(nn.Module):
    def __init__(self, alpha=0.97, gamma=2.0): super().__init__(); self.alpha=alpha; self.gamma=gamma
    def forward(self, p, y, weight=None):
        p=p.clamp(1e-6,1-1e-6); pt=torch.where(y>0.5, p, 1-p)
        w=self.alpha*(y>0.5).float()+(1-self.alpha)*(y<=0.5).float()
        if weight is not None: w=w*weight
        return (- ((1-pt)**self.gamma) * (y*torch.log(p)+(1-y)*torch.log(1-p)) * w).mean()

def class_weights(y):
    pos=(y>0.5).sum(); neg=y.numel()-pos
    w=torch.ones_like(y); 
    if pos>0: w[y>0.5]=(neg+1e-6)/(pos+1e-6)
    return w

def hard_neg_mask(p,y,max_ratio=30):
    pos_idx=(y>0.5).nonzero(as_tuple=False).view(-1); neg_idx=(y<=0.5).nonzero(as_tuple=False).view(-1)
    if pos_idx.numel()==0:
        if neg_idx.numel()==0: return torch.zeros_like(y, dtype=torch.bool)
        k=max(1, min(int(0.01*neg_idx.numel()), neg_idx.numel()))
        keep_neg=neg_idx[torch.topk(p[neg_idx], k=k).indices]
        m=torch.zeros_like(y, dtype=torch.bool); m[keep_neg]=True; return m
    k=min(neg_idx.numel(), int(max_ratio*pos_idx.numel()))
    keep_neg=neg_idx[torch.topk(p[neg_idx], k=k).indices] if k>0 else torch.tensor([], device=y.device, dtype=torch.long)
    m=torch.zeros_like(y, dtype=torch.bool); m[pos_idx]=True
    if keep_neg.numel()>0: m[keep_neg]=True
    return m

def monotonicity_loss(logit: torch.Tensor, paths_idx: List[List[int]]):
    if not paths_idx: return logit.new_zeros(())
    loss=0.0; cnt=0
    for path in paths_idx:
        if len(path)<2: continue
        seq=logit[path]
        loss = loss + sum(F.relu(seq[i]-seq[i+1]) for i in range(len(seq)-1))/(len(seq)-1)
        cnt+=1
    return loss/max(1,cnt)

def path_ranking_loss(p: torch.Tensor, paths_idx: List[List[int]], margin=0.1):
    if not paths_idx: return p.new_zeros(())
    loss=0.0; cnt=0
    for path in paths_idx:
        if len(path)<2: continue
        for i in range(len(path)-1):
            a,b = path[i], path[i+1]
            loss = loss + F.relu(margin - (p[b]-p[a]))
            cnt+=1
    return loss/max(1,cnt)

def edge_participation_loss(model: 'CausalVulNet', g: HeteroData, h, causal_nodes: List[int]):
    if not causal_nodes: return h.new_zeros(())
    pos_masks = build_slice_edge_mask(g, causal_nodes)
    loss=0.0; cnt=0
    for (s,r,t) in model.edge_types:
        if r not in pos_masks: continue
        ei=ei_of(g, r); 
        if ei.numel()==0: continue
        pm = pos_masks[r]
        if pm.numel()==0: continue
        pos_idx = pm.nonzero(as_tuple=False).view(-1)
        if pos_idx.numel()==0: continue
        all_idx = torch.arange(ei.size(1), device=ei.device)
        neg_idx = all_idx[~pm]
        k = min(pos_idx.numel(), neg_idx.numel())
        if k==0: continue
        neg_idx = neg_idx[torch.randperm(neg_idx.numel(), device=neg_idx.device)[:k]]

        u_p, v_p = ei[0][pos_idx], ei[1][pos_idx]
        u_n, v_n = ei[0][neg_idx], ei[1][neg_idx]
        sc_pos = model.next_head(torch.cat([h[u_p], h[v_p]], dim=-1)).squeeze(-1)
        sc_neg = model.next_head(torch.cat([h[u_n], h[v_n]], dim=-1)).squeeze(-1)

        loss = loss + F.binary_cross_entropy_with_logits(sc_pos, torch.ones_like(sc_pos))
        loss = loss + F.binary_cross_entropy_with_logits(sc_neg, torch.zeros_like(sc_neg))
        cnt+=2
    return loss/max(1,cnt)

# ------------------ Beam (multi-root) ------------------
@dataclass
class BeamPath:
    score: float; nodes: List[int]

def run_beam(p: torch.Tensor, edge_scores: Dict[str,torch.Tensor], edges: Dict[str,torch.Tensor],
             seeds: List[int], beam_width=32, max_hops=6, alpha_node=0.7):
    adj=defaultdict(lambda: defaultdict(list))
    for r,ei in edges.items():
        if ei is None or (isinstance(ei, torch.Tensor) and ei.numel()==0): continue
        s,d=ei[0].tolist(), ei[1].tolist()
        for u,v in zip(s,d): adj[r][u].append(v)
    beams=[BeamPath(score=float(torch.log(p[s].clamp(1e-9,1-1e-9))), nodes=[s]) for s in seeds]
    finished=[]
    for _ in range(max_hops):
        cand=[]
        for b in beams:
            u=b.nodes[-1]
            for r,nb in adj.items():
                if u not in nb: continue
                for v in nb[u]:
                    ei=edges.get(r); sc=edge_scores.get(r)
                    if ei is None or sc is None or (isinstance(ei, torch.Tensor) and ei.numel()==0): continue
                    m=((ei[0]==u)&(ei[1]==v))
                    es=float(sc[m].max().item()) if m.any() else 0.0
                    cand.append(BeamPath(score=b.score+alpha_node*float(torch.log(p[v].clamp(1e-9,1-1e-9)))+(1-alpha_node)*es, nodes=b.nodes+[v]))
        cand.sort(key=lambda x:x.score, reverse=True)
        beams=cand[:beam_width]; finished.extend(beams[:beam_width//2])
        if not beams: break
    uniq=[]; seen=set()
    for bp in sorted(finished, key=lambda x:x.score, reverse=True):
        key=tuple(bp.nodes[-min(4,len(bp.nodes)):])
        if key in seen: continue
        seen.add(key); uniq.append(bp)
        if len(uniq)>=beam_width: break
    return uniq

# ------------------ Metrics & thresholds ------------------
def metrics_at_thr(p,y,thr):
    pred=(p>=thr).float()
    tp=(pred*y).sum().item(); fp=(pred*(1-y)).sum().item(); fn=((1-pred)*y).sum().item()
    prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec,"recall":rec,"f1":f1}

def fit_thr_per_source(per_src_p: Dict[str,List[torch.Tensor]], per_src_y: Dict[str,List[torch.Tensor]]):
    th={}
    for s in per_src_p:
        P=torch.cat(per_src_p[s]); Y=torch.cat(per_src_y[s])
        best=-1; tbest=0.5
        for t in torch.linspace(0.05,0.95,19):
            m=metrics_at_thr(P,Y,float(t))
            if m["f1"]>best: best=m["f1"]; tbest=float(t)
        th[s]=tbest
    return th

# ------------------ CCS / CFAM (strict slicing) ------------------
@torch.no_grad()
def ccs_for_graph(model: 'CausalVulNet', g: HeteroData, causal_nodes: List[int]):
    if not causal_nodes: return None
    logit,_=model(g); p=torch.sigmoid(logit); p0=float(p.max().item())
    st=g[NODE_TYPE]; saved={}
    idx=torch.tensor(sorted(causal_nodes), dtype=torch.long, device=next(model.parameters()).device)
    if hasattr(st,"x"): 
        st.x=st.x.to(idx.device); saved["x"]=st.x[idx].clone(); st.x[idx]=0.0
    if hasattr(st,"x_text"):
        st.x_text=st.x_text.to(idx.device); saved["x_text"]=st.x_text[idx].clone(); st.x_text[idx]=0.0
    logit_cf,_=model(g); pcf=float(torch.sigmoid(logit_cf).max().item())
    if "x" in saved: st.x[idx]=saved["x"]
    if "x_text" in saved: st.x_text[idx]=saved["x_text"]
    return (p0-pcf)**2

def cfam_for_graph(model: 'CausalVulNet', g: HeteroData, causal_nodes: List[int], spurious_nodes: List[int]):
    if not causal_nodes: return None
    with torch.enable_grad():
        h=model.encode(g); h.retain_grad()
        score=model.node_head(h).squeeze(-1).sum()
        grad=torch.autograd.grad(score, h, retain_graph=False, create_graph=False)[0]
    attr=(grad * h).abs().sum(-1).detach()
    num=float(attr[causal_nodes].sum().item()) if causal_nodes else 0.0
    den = num + (float(attr[spurious_nodes].sum().item()) if spurious_nodes else 0.0) + 1e-9
    return (num/den) if den>0 else None

# ------------------ Model builder (tries CUDA with small sizes / dtypes, else CPU) ------------------
def build_model_adaptive(in_dim_total, etypes_all, d_text_max, d_num_max):
    size_grid=[(48,2),(32,2),(24,2),(16,2),(12,1),(8,1),(4,1),(2,1)]
    dtypes=[]
    if DEVICE.type=="cuda":
        dtypes.append(torch.float16)
        if torch.cuda.get_device_capability(0)[0] >= 8: dtypes.append(torch.bfloat16)
    dtypes.append(torch.float32)
    last_err=None
    for dt in dtypes:
        for H,L in size_grid:
            try:
                m=CausalVulNet(in_dim=in_dim_total, hidden=H, layers=L, edge_types=etypes_all,
                               d_text=d_text_max, d_num=d_num_max)
                if DEVICE.type=="cuda":
                    m=m.to(device=DEVICE, dtype=dt, non_blocking=True)
                    print(f"[model] CUDA ok dtype={str(dt).split('.')[-1]} hidden={H} layers={L}")
                else:
                    m=m.to("cpu"); print(f"[model] CPU hidden={H} layers={L}")
                return m
            except RuntimeError as e:
                last_err=e
                if "out of memory" in str(e).lower():
                    print(f"[model] OOM (dtype={str(dt).split('.')[-1]} H={H} L={L}) → trying smaller …")
                    try: del m
                    except: pass
                    gc.collect()
                    continue
                else:
                    raise
    if DEVICE.type=="cuda":
        print("[model] All CUDA attempts failed → CPU fallback.")
        m=CausalVulNet(in_dim=in_dim_total, hidden=8, layers=1, edge_types=etypes_all,
                       d_text=d_text_max, d_num=d_num_max).to("cpu")
        return m
    raise last_err

# ------------------ Training & evaluation (no DataLoader; CPU fallback mid-run) ------------------
def train_and_eval():
    def list_graphs(d):
        p=Path(d); return sorted([q for q in p.glob("*.pt")]) if p.exists() else []
    # scan dims
    def scan_feature_dims(dirs: List[str]) -> Tuple[int,int,Tuple[Tuple[str,str,str],...]]:
        max_text=0; max_num=0; etypes_seen=set()
        for d in dirs:
            for pt in list_graphs(d):
                g=safe_load(pt); st=g[NODE_TYPE]
                if hasattr(st,"x_text"): max_text=max(max_text, int(st.x_text.size(1)))
                if hasattr(st,"x"):      max_num =max(max_num,  int(st.x.size(1)))
                for et in g.edge_types:
                    if et[1] in EDGE_KEEP: etypes_seen.add(et)
        return max_text, max_num, tuple(sorted(etypes_seen))

    d_text_max, d_num_max, etypes_all = scan_feature_dims([TRAIN_DIR, VALID_DIR, TEST_DIR])
    in_dim_total = d_text_max + d_num_max

    train_files = list_graphs(TRAIN_DIR)
    print(f"[DATA] {len(train_files)} graphs in {TRAIN_DIR}")

    model=build_model_adaptive(in_dim_total, etypes_all, d_text_max, d_num_max)
    cur_dev = next(model.parameters()).device
    on_cuda=(cur_dev.type=="cuda")
    scaler=torch.cuda.amp.GradScaler(enabled=(AMP_ENABLED and on_cuda))
    opt=torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    focal=FocalBCELoss(alpha=0.97, gamma=2.0)

    logf=open(OUT_ROOT/"train_log.jsonl","a",encoding="utf-8")
    steps=0; loss_sum=0.0; used_graphs=0

    for ep in range(1, EPOCHS+1):
        model.train()
        for pt in train_files:
            try:
                g=safe_load(pt); g.__dict__["_aug_json_path"]=near_json(pt)
                dev = next(model.parameters()).device
                try:
                    g=g.to(dev, non_blocking=(dev.type=="cuda"))
                except RuntimeError as e:
                    if "out of memory" in str(e).lower() and dev.type=="cuda":
                        print("[fallback] OOM moving graph → switching model to CPU for rest of run.")
                        model=model.to("cpu"); cur_dev=torch.device("cpu"); on_cuda=False
                        scaler=torch.cuda.amp.GradScaler(enabled=False)
                        opt=torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
                        g=g.to("cpu")
                    else:
                        raise

                gt=read_ground_truth(g)
                if gt["y"] is None and not gt["paths_idx"] and not (gt["roots"] and gt["sinks"]):
                    continue

                with torch.cuda.amp.autocast(enabled=(AMP_ENABLED and on_cuda)):
                    logit,h=model(g); p=torch.sigmoid(logit)
                    y = gt["y"]
                    if y is None:
                        y=torch.zeros(g[NODE_TYPE].num_nodes, dtype=torch.float32, device=dev)
                        for s in gt["sinks"]:
                            if 0<=s<y.numel(): y[s]=1.0

                    causal_nodes, _ = slice_causal_nodes(g, gt["roots"], gt["sinks"], gt["paths_idx"])
                    if gt["causal_nodes_idx"]: causal_nodes = sorted(set(gt["causal_nodes_idx"]))
                    sp_set=set(causal_nodes)
                    spurious_nodes = [i for i in range(g[NODE_TYPE].num_nodes) if i not in sp_set]
                    if gt["spurious_nodes_idx"]: spurious_nodes = gt["spurious_nodes_idx"]

                    keep=hard_neg_mask(p.detach(), y, max_ratio=NEG_POS_RATIO)
                    y_m, p_m, logit_m = y[keep], p[keep], logit[keep]
                    w = class_weights(y_m)

                    loss_node = focal(p_m, y_m, weight=w)
                    loss_edge = edge_participation_loss(model, g, h, causal_nodes)
                    loss_mono = monotonicity_loss(logit, gt["paths_idx"])
                    loss_rank = path_ranking_loss(p, gt["paths_idx"], margin=0.1)
                    loss = loss_node + 0.15*loss_edge + 0.25*loss_mono + 0.20*loss_rank

                opt.zero_grad(set_to_none=True)
                if on_cuda and AMP_ENABLED:
                    scaler.scale(loss).backward()
                    scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.5)
                    scaler.step(opt); scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.5)
                    opt.step()

                steps+=1; used_graphs+=1; loss_sum+=float(loss.detach().cpu())
                if steps%25==0:
                    logf.write(json.dumps({
                        "epoch":ep,"step":steps,"loss_avg":loss_sum/max(1,steps),
                        "loss_node":float(loss_node.detach().cpu()),
                        "loss_edge":float(loss_edge.detach().cpu()),
                        "loss_mono":float(loss_mono.detach().cpu()),
                        "loss_rank":float(loss_rank.detach().cpu())
                    })+"\n"); logf.flush()
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print("[warn] OOM during forward/backward → switching to CPU and continuing.")
                    model=model.to("cpu"); cur_dev=torch.device("cpu"); on_cuda=False
                    scaler=torch.cuda.amp.GradScaler(enabled=False)
                    opt=torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
                    continue
                else:
                    raise
        print(f"[train] epoch {ep} loss={loss_sum/max(1,steps):.4f} (used_graphs={used_graphs})")

    logf.close()
    torch.save({
        "model":model.state_dict(), 
        "in_dim":in_dim_total, 
        "edge_types":etypes_all,
        "d_text":d_text_max, "d_num":d_num_max,
        "dtype": str(next(model.parameters()).dtype)
    }, OUT_ROOT/"model.pt")
    print("Saved:", OUT_ROOT/"model.pt")

    # ---- per-source thresholds + CCS/CFAM ----
    def eval_split(name, path_dir):
        p=Path(path_dir)
        if not p.exists(): return None
        files=sorted(p.glob("*.pt"))
        if not files: return None
        model.eval()
        per_src_p=defaultdict(list); per_src_y=defaultdict(list)
        used=0
        with torch.no_grad():
            for pt in files:
                g=safe_load(pt); g.__dict__["_aug_json_path"]=near_json(pt)
                g=g.to(next(model.parameters()).device)
                gt=read_ground_truth(g)
                if gt["y"] is None:
                    if gt["sinks"]:
                        y=torch.zeros(g[NODE_TYPE].num_nodes, dtype=torch.float32, device=next(model.parameters()).device)
                        for s in gt["sinks"]:
                            if 0<=s<y.numel(): y[s]=1.0
                    else:
                        continue
                else:
                    y=gt["y"].to(next(model.parameters()).device)
                logit,_=model(g); p=torch.sigmoid(logit)
                src=str(getattr(g,"source_name","default"))
                per_src_p[src].append(p.detach().cpu()); per_src_y[src].append(y.detach().cpu()); used+=1
        thr=fit_thr_per_source(per_src_p, per_src_y) if per_src_p else {}
        overall={"precision":0,"recall":0,"f1":0}; nsrc=0
        for s in per_src_p:
            P=torch.cat(per_src_p[s]); Y=torch.cat(per_src_y[s])
            m=metrics_at_thr(P,Y,thr.get(s,0.5))
            for k in overall: overall[k]+=m[k]
            nsrc+=1
        for k in overall: overall[k]/=max(1,nsrc)

        # strict CCS/CFAM via slicing
        ccs_list=[]; cfam_list=[]
        for pt in files:
            g=safe_load(pt); g.__dict__["_aug_json_path"]=near_json(pt)
            g=g.to(next(model.parameters()).device)
            gt=read_ground_truth(g)
            causal_nodes, _ = slice_causal_nodes(g, gt["roots"], gt["sinks"], gt["paths_idx"])
            if gt["causal_nodes_idx"]: causal_nodes = sorted(set(gt["causal_nodes_idx"]))
            sp_set=set(causal_nodes)
            spurious_nodes = [i for i in range(g[NODE_TYPE].num_nodes) if i not in sp_set]
            if gt["spurious_nodes_idx"]:
                spurious_nodes = gt["spurious_nodes_idx"]
            if causal_nodes:
                ccs_val = ccs_for_graph(model, g, causal_nodes)
                cfam_val = cfam_for_graph(model, g, causal_nodes, spurious_nodes)
                if ccs_val is not None: ccs_list.append(ccs_val)
                if cfam_val is not None: cfam_list.append(cfam_val)

        return {
            "split":name, "overall":overall, "thresholds":thr, "n_graphs_used":used,
            "CCS_mean": (sum(ccs_list)/len(ccs_list) if ccs_list else None),
            "CFAM_mean":(sum(cfam_list)/len(cfam_list) if cfam_list else None),
            "CCS_count":len(ccs_list), "CFAM_count":len(cfam_list)
        }

    reports={}
    reports["train"]=eval_split("train", TRAIN_DIR)
    if Path(VALID_DIR).exists(): reports["valid"]=eval_split("valid", VALID_DIR)
    if Path(TEST_DIR).exists():  reports["test"] =eval_split("test",  TEST_DIR)
    with open(OUT_ROOT/"metrics.json","w",encoding="utf-8") as f:
        json.dump({"reports":reports}, f, indent=2)
    print("Saved:", OUT_ROOT/"metrics.json")

    # Demo: multi-root beam on first train graph
    files = sorted(Path(TRAIN_DIR).glob("*.pt"))
    if files:
        g0=safe_load(files[0]).to(next(model.parameters()).device)
        g0.__dict__["_aug_json_path"]=near_json(files[0])
        gt0=read_ground_truth(g0)
        if gt0["roots"]:
            model.eval()
            with torch.no_grad():
                logit0,h0=model(g0); p0=torch.sigmoid(logit0)
                edges0  = {r:ei_of(g0,r) for (s,r,t) in model.edge_types}
                edge_sc = model.edge_guidance(h0, edges0)
                beam=run_beam(p0, edge_sc, edges0, seeds=gt0["roots"], beam_width=BEAM_W, max_hops=MAX_HOPS, alpha_node=ALPHA_NODE)
            with open(OUT_ROOT/"demo_paths.json","w",encoding="utf-8") as f:
                json.dump({"roots":gt0["roots"], "top_paths":[bp.nodes for bp in beam[:10]]}, f, indent=2)
            print("Saved demo paths:", OUT_ROOT/"demo_paths.json")
        else:
            print("[DEMO] No ground-truth roots; beam demo skipped.")

# ------------------ Run ------------------
train_and_eval()


Saved: out\cvul_run_1760241759\metrics.json
[DEMO] No ground-truth roots; beam demo skipped.


In [20]:
# Causal-Vul: inter-procedural demo trainer + CCS/CFAM evaluation (drop-in, single cell)
# ----------------------------------------------------------------------------------
# What this cell does
# - Loads HeteroData graphs from Dataset/{train,valid,test}/hetero_ready_gcbert/*.pt
# - Trains a light relational GNN with learned per-relation guidance (no torch-scatter deps)
# - Demonstrates inter-procedural reasoning via CALL/ARG2PARAM/RET2CALL/RET2LHS (+ CFG/DFG)
# - Computes standard node metrics (precision/recall/F1) per split
# - Computes CCS and CFAM strictly from program slicing induced by labels (no heuristics)
#   * CSS: (p(X) - p(do(X')))² where do(X') = zero out causal-slice node features
#   * CFAM: attribution mass on causal / (causal + spurious), using grad‖∂logit/∂h‖
# - Multi-root is supported automatically: roots are slice nodes without in-slice predecessors
# - Saves everything under out/cvul/<timestamp>/
#
# Notes
# - This code avoids torch-geometric CUDA extensions and uses only edge_index tensors.
# - It tolerates varying node-feature dimensions across graphs.
# - It does NOT try to maximize accuracy; it focuses on proving inter-procedural flow + CCS/CFAM.
#
# Minimal knobs
EPOCHS          = 2         # keep small for demo; raise if you like
HIDDEN          = 64        # modest capacity to avoid OOM
LAYERS          = 3
LR              = 2e-3
WEIGHT_DECAY    = 1e-4
NEG_POS_RATIO   = 20        # hard-negative sampling cap (#neg <= ratio * #pos)
USE_AMP         = True      # AMP if CUDA is available
MAX_NODES       = 12000     # safety shrinker
MAX_EDGES       = 200000
NODE_TYPE       = "node"

# Paths
TRAIN_DIR = "Dataset/train/hetero_ready_gcbert"
VALID_DIR = "Dataset/valid/hetero_ready_gcbert"
TEST_DIR  = "Dataset/test/hetero_ready_gcbert"

# ------------------------ Imports & basic checks ------------------------
import os, json, math, time, gc, random, hashlib, re, traceback
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict, deque

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from torch_geometric.data import HeteroData
except Exception as e:
    raise RuntimeError("torch-geometric must be installed and importable to load HeteroData graphs") from e

def pick_device():
    if torch.cuda.is_available():
        # simple probe to ensure VRAM is usable
        try:
            _ = torch.empty(1024, device="cuda")
            d = torch.device("cuda")
            print(f"[env] torch={torch.__version__} device=cuda")
            try:
                print("GPU:", torch.cuda.get_device_name(0))
            except Exception:
                pass
            return d
        except Exception:
            pass
    print(f"[env] torch={torch.__version__} device=cpu")
    return torch.device("cpu")

DEVICE = pick_device()

# ------------------------ IO helpers ------------------------
def near_json(pt_path: str) -> Optional[str]:
    """Find a sidecar JSON (augmented) near a .pt file (optional, used only for labels if present)."""
    p = Path(pt_path)
    candidates = [
        p.with_suffix(".json"),
        p.with_suffix(".aug.json"),
        p.with_suffix(".unified.json"),
        Path("notebooks")/"Dataset"/"train"/"unified_aug"/(p.stem+".json"),
        Path("notebooks")/"Dataset"/"valid"/"unified_aug"/(p.stem+".json"),
        Path("notebooks")/"Dataset"/"test"/"unified_aug"/(p.stem+".json"),
    ]
    for c in candidates:
        if c.exists(): return str(c)
    return None

def load_json(path: str) -> Optional[dict]:
    if not path or not os.path.exists(path): return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def safe_load(path: str):
    # plain torch.load (your env is torch 2.4.1, so no weights_only restriction)
    return torch.load(path, map_location="cpu")

class GraphDir(Dataset):
    def __init__(self, dir_path: str):
        self.paths = sorted([str(p) for p in Path(dir_path).glob("*.pt")])
        if not self.paths:
            raise FileNotFoundError(f"No .pt found in {dir_path}")
        print(f"[DATA] {len(self.paths)} graphs in {dir_path}")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        pt = self.paths[i]
        g  = safe_load(pt)
        g.__dict__["_aug_json_path"] = near_json(pt)
        return g

# ------------------------ Graph utilities ------------------------
def get_edge_index(g: HeteroData, r: Tuple[str,str,str]) -> torch.Tensor:
    if r not in g.edge_types:
        dev = (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE], "x") else "cpu")
        return torch.zeros((2,0), dtype=torch.long, device=dev)
    store = g[r]
    ei = getattr(store, "edge_index", None)
    if ei is None:
        adj_t = getattr(store, "adj_t", None)
        if adj_t is not None:
            row, col, _ = adj_t.coo()
            ei = torch.stack([col, row], dim=0)
        else:
            dev = (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE], "x") else "cpu")
            ei = torch.zeros((2,0), dtype=torch.long, device=dev)
    return ei

# Allowed relation set (forward); we’ll auto-add reverse in adjacency when needed
FORWARD_RELS = ("DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS","AST")

def build_adj(g: HeteroData):
    fwd = defaultdict(list); rev = defaultdict(list)
    for (s,r,t) in g.edge_types:
        if r not in FORWARD_RELS: 
            # also allow *_REV if present
            if r.endswith("_REV"): pass
            else: continue
        ei = get_edge_index(g, (s,r,t))
        if ei.numel()==0: continue
        for u,v in zip(ei[0].tolist(), ei[1].tolist()):
            fwd[(s,r,t)].append((u,v))
            rev[(s,r,t)].append((v,u))
    return fwd, rev

def shrink_graph_if_needed(g: HeteroData) -> HeteroData:
    st = g[NODE_TYPE]; N = int(st.num_nodes)
    total_edges = 0
    for et in g.edge_types:
        total_edges += int(get_edge_index(g, et).size(1))
    if N <= MAX_NODES and total_edges <= MAX_EDGES:
        return g
    keep = set()
    for r in ("CALL","ARG2PARAM","RET2CALL","RET2LHS","DFG","CFG"):
        et = (NODE_TYPE, r, NODE_TYPE)
        if et in g.edge_types:
            ei = get_edge_index(g, et)
            if ei.numel()>0:
                keep.update(ei[0].tolist()); keep.update(ei[1].tolist())
    if not keep:
        # sample first MAX_NODES nodes
        keep = set(range(min(N, MAX_NODES)))
    keep = sorted(list(keep))[:MAX_NODES]
    idx = {old:i for i,old in enumerate(keep)}
    g2 = HeteroData()
    if hasattr(st, "x"): g2[NODE_TYPE].x = st.x[keep]
    if hasattr(st, "x_text"): g2[NODE_TYPE].x_text = st.x_text[keep]
    if hasattr(st, "nid"): g2[NODE_TYPE].nid = st.nid[keep]
    if hasattr(st, "y"): g2[NODE_TYPE].y = st.y[keep]
    g2[NODE_TYPE].num_nodes = len(keep)
    for et in g.edge_types:
        ei = get_edge_index(g, et)
        if ei.numel()==0:
            g2[et].edge_index = ei
            continue
        src, dst = ei[0].tolist(), ei[1].tolist()
        new_src, new_dst = [], []
        for u,v in zip(src, dst):
            if u in idx and v in idx:
                new_src.append(idx[u]); new_dst.append(idx[v])
        if new_src:
            g2[et].edge_index = torch.tensor([new_src, new_dst], dtype=torch.long)
        else:
            g2[et].edge_index = torch.zeros((2,0), dtype=torch.long)
    g2.__dict__["_aug_json_path"] = getattr(g, "_aug_json_path", None)
    return g2

# ------------------------ Labels (no heuristics) ------------------------
def labels_from_graph_or_json(g: HeteroData) -> Optional[torch.Tensor]:
    # Try from graph first
    st = g[NODE_TYPE]
    if hasattr(st, "y"):
        y = st.y.float()
        if y.dim()==1 and y.numel()==st.num_nodes:
            return y
        if y.dim()==2 and y.size(1)==1:
            return y.view(-1).float()
    # Try sidecar JSON (treat any node with "vulnerable"==1 or "is_sink"==true as positive)
    jp = getattr(g, "_aug_json_path", None)
    j  = load_json(jp) if jp else None
    if isinstance(j, dict) and isinstance(j.get("nodes"), list):
        arr = j["nodes"]; y = torch.zeros(st.num_nodes, dtype=torch.float32)
        # If nid exists, map by external ids; else assume order
        id_map = None
        if hasattr(st, "nid"):
            ids = st.nid.view(-1).tolist()
            id_map = {int(ids[i]): i for i in range(len(ids))}
        for idx,n in enumerate(arr):
            if not isinstance(n, dict): continue
            pos = False
            for k in ("y","label","vulnerable","is_vuln","is_sink"):
                if k in n:
                    v = n[k]; 
                    if isinstance(v, (int,float)) and float(v)>0.5: pos=True
                    if isinstance(v, str) and v.strip().lower() in ("1","true","yes"): pos=True
            if id_map is not None and "_id" in n:
                ext = int(n["_id"]) if isinstance(n["_id"], (int,str)) and str(n["_id"]).lstrip("-").isdigit() else None
                if ext is not None and ext in id_map and pos:
                    y[id_map[ext]] = 1.0
            else:
                if idx < y.numel() and pos:
                    y[idx] = 1.0
        if (y>0.5).any():
            return y
    return None

# ------------------------ Causal slicing (no heuristics) ------------------------
ALLOWED_FOR_SLICE = ("DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS")

def build_simple_adj(g: HeteroData):
    fwd = defaultdict(list); rev = defaultdict(list)
    for r in ALLOWED_FOR_SLICE:
        et = (NODE_TYPE,r,NODE_TYPE)
        if et not in g.edge_types: continue
        ei = get_edge_index(g, et)
        if ei.numel()==0: continue
        s,d = ei[0].tolist(), ei[1].tolist()
        for u,v in zip(s,d):
            fwd[u].append(v)
            rev[v].append(u)
    return fwd, rev

def slice_from_labels(g: HeteroData, y: torch.Tensor):
    """Causal slice strictly from labels:
       sinks = {i | y_i=1}, backward slice via rev edges, then forward slice; causal = B∩F; 
       roots = causal nodes with zero in-degree inside B (multi-root)."""
    sinks=[i for i in range(y.numel()) if float(y[i])>0.5]
    if not sinks: return [], [], []
    fwd, rev = build_simple_adj(g)
    B=set(); dq=deque(sinks)
    while dq:
        u=dq.popleft()
        if u in B: continue
        B.add(u)
        for v in rev.get(u, []):
            if v not in B: dq.append(v)
    F=set(); dq=deque(list(B))
    while dq:
        u=dq.popleft()
        if u in F: continue
        F.add(u)
        for v in fwd.get(u, []):
            if v not in F: dq.append(v)
    causal = sorted(list(B.intersection(F)))
    roots  = sorted([u for u in B if not any((v in B) for v in rev.get(u, []))])
    spurious = [i for i in range(y.numel()) if i not in set(causal)]
    return causal, roots, spurious

# ------------------------ Model ------------------------
class ProjectIn(nn.Module):
    """Caches a Linear for every encountered input dim to avoid matmul shape errors."""
    def __init__(self, hidden: int):
        super().__init__()
        self.hidden = hidden
        self.proj = nn.ModuleDict()  # key = str(dim)
    def forward(self, x: torch.Tensor):
        d = x.size(1); k = str(d)
        if k not in self.proj:
            lin = nn.Linear(d, self.hidden)
            nn.init.xavier_uniform_(lin.weight); nn.init.zeros_(lin.bias)
            self.proj[k] = lin.to(x.device, dtype=x.dtype)
        return self.proj[k](x)

class RelLayer(nn.Module):
    """Simple per-relation mean aggregator + LayerNorm + SiLU. No scatter deps."""
    def __init__(self, hidden: int, rels: List[str]):
        super().__init__()
        self.rels = rels
        self.self_lin = nn.Linear(hidden, hidden)
        self.msg_lin  = nn.ModuleDict({r: nn.Linear(hidden, hidden) for r in rels})
        self.ln = nn.LayerNorm(hidden)
    def forward(self, h: torch.Tensor, edges: Dict[str, torch.Tensor]) -> torch.Tensor:
        out = self.self_lin(h)
        N = h.size(0)
        for r in self.rels:
            ei = edges.get(r, None)
            if ei is None or ei.numel()==0: continue
            src, dst = ei[0], ei[1]
            m = self.msg_lin[r](h[src])
            agg = torch.zeros_like(h)
            agg.index_add_(0, dst, m)
            deg = torch.zeros(N, device=h.device).index_add_(0, dst, torch.ones_like(dst, dtype=torch.float32))
            out = out + agg / deg.clamp(min=1).unsqueeze(-1)
        out = self.ln(out)
        return F.silu(out)

class CausalVulNet(nn.Module):
    def __init__(self, in_dim_guess: int, hidden: int, layers: int, rels: List[str]):
        super().__init__()
        self.rels = rels
        self.project_in = ProjectIn(hidden)
        self.layers = nn.ModuleList([RelLayer(hidden, rels) for _ in range(layers)])
        self.node_head = nn.Sequential(nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 1))
        # learned edge guidance
        self.rel_emb = nn.Embedding(num_embeddings=len(rels), embedding_dim=hidden)
        self.edge_head = nn.Sequential(nn.Linear(hidden*3, hidden), nn.SiLU(), nn.Linear(hidden, 1))
        # edge participation (aux head)
        self.part_head = nn.Sequential(nn.Linear(hidden*2, hidden), nn.SiLU(), nn.Linear(hidden, 1))

    def encode(self, g: HeteroData) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        st = g[NODE_TYPE]
        xs = []
        if hasattr(st, "x_text"): xs.append(st.x_text.float())
        if hasattr(st, "x"):      xs.append(st.x.float())
        if not xs:
            xs = [torch.zeros(st.num_nodes, 1, device=st.x.device if hasattr(st,"x") else "cpu")]
        x = xs[0] if len(xs)==1 else torch.cat(xs, dim=1)
        h = F.relu(self.project_in(x))
        # collect edges (dict rel->edge_index)
        edges = {}
        for r in self.rels:
            et = (NODE_TYPE, r, NODE_TYPE)
            if et in g.edge_types:
                ei = get_edge_index(g, et)
                edges[r] = ei.to(h.device)
            else:
                edges[r] = torch.zeros((2,0), dtype=torch.long, device=h.device)
        for layer in self.layers:
            h = layer(h, edges)
        return h, edges

    def forward(self, g: HeteroData):
        h, edges = self.encode(g)
        logit = self.node_head(h).squeeze(-1)
        return logit, h, edges

    def edge_scores(self, h: torch.Tensor, edges: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        scores = {}
        for i, r in enumerate(self.rels):
            ei = edges.get(r, None)
            if ei is None or ei.numel()==0:
                scores[r] = torch.empty(0, device=h.device)
                continue
            src, dst = ei[0], ei[1]
            t = self.rel_emb(torch.tensor(i, device=h.device)).view(1,-1).expand(src.numel(), -1)
            s = self.edge_head(torch.cat([h[src], h[dst], t], dim=-1)).squeeze(-1)
            scores[r] = s
        return scores

# ------------------------ Losses ------------------------
class FocalBCELoss(nn.Module):
    def __init__(self, alpha_pos=0.95, gamma=2.0):
        super().__init__()
        self.alpha_pos=alpha_pos; self.gamma=gamma
    def forward(self, p: torch.Tensor, y: torch.Tensor, weight=None):
        eps=1e-6; p=p.clamp(eps,1-eps)
        pt = torch.where(y>0.5, p, 1-p)
        w  = (self.alpha_pos*(y>0.5).float() + (1-self.alpha_pos)*(y<=0.5).float())
        if weight is not None: w = w*weight
        loss = - ((1-pt)**self.gamma) * ( y*torch.log(p) + (1-y)*torch.log(1-p) ) * w
        return loss.mean()

def class_weights(y: torch.Tensor):
    pos = (y>0.5).sum().item()
    neg = y.numel() - pos
    w = torch.ones_like(y)
    if pos>0: w[y>0.5] = (neg+1e-6)/(pos+1e-6)
    return w

def hard_negative_mask(p: torch.Tensor, y: torch.Tensor, max_ratio=NEG_POS_RATIO):
    pos_idx = (y>0.5).nonzero(as_tuple=False).view(-1)
    neg_idx = (y<=0.5).nonzero(as_tuple=False).view(-1)
    k = int(min(len(neg_idx), max_ratio * max(1,len(pos_idx))))
    if k <= 0: return (y>0.5)
    if len(neg_idx)==0: return (y>0.5)
    # hardest negatives by p
    vals, inds = torch.topk(p[neg_idx], k=k)
    keep_neg = neg_idx[inds]
    mask = torch.zeros_like(y, dtype=torch.bool)
    if len(pos_idx)>0: mask[pos_idx] = True
    mask[keep_neg] = True
    return mask

def edge_participation_loss(h, edges, model: CausalVulNet, p: torch.Tensor):
    """Encourage edges bridging high-p nodes to have high participation."""
    loss = 0.0; cnt=0
    for r, ei in edges.items():
        if ei.numel()==0: continue
        src, dst = ei[0], ei[1]
        logits = model.part_head(torch.cat([h[src], h[dst]], dim=-1)).squeeze(-1)
        target = torch.minimum(p[src], p[dst]).detach()
        loss = loss + F.binary_cross_entropy_with_logits(logits, target)
        cnt += 1
    return loss / max(1, cnt)

def monotonicity_loss_on_paths(logit: torch.Tensor, paths: List[List[int]]):
    loss = 0.0; cnt=0
    for path in paths:
        if len(path) < 2: continue
        z = logit[path]
        # enforce non-decreasing along the path
        diffs = z[1:] - z[:-1]
        loss = loss + F.relu(-diffs).mean()
        cnt += 1
    return loss / max(1, cnt)

def path_ranking_loss(logit: torch.Tensor, pos_paths: List[List[int]], neg_paths: List[List[int]], margin=0.5):
    """Path score = mean(sigmoid(logit) along nodes)."""
    if not pos_paths or not neg_paths: 
        return logit.new_zeros(())
    def path_score(path):
        return torch.sigmoid(logit[path]).mean()
    loss = 0.0; cnt=0
    m = logit.new_tensor(margin)
    for pp in pos_paths:
        sp = path_score(pp)
        for np in neg_paths[:3]:  # sample a few
            sn = path_score(np)
            loss = loss + F.relu(m - (sp - sn))
            cnt += 1
    return loss / max(1, cnt)

# ------------------------ Beam search ------------------------
def run_beam(p: torch.Tensor, edge_scores: Dict[str, torch.Tensor], edges: Dict[str, torch.Tensor],
             seeds: List[int], beam_width=16, max_hops=6, alpha_node=0.7):
    # Build simple adjacency
    adj = defaultdict(lambda: defaultdict(list))  # rel -> u -> [v,..]
    for r, ei in edges.items():
        if ei is None or ei.numel()==0: continue
        s,d = ei[0].tolist(), ei[1].tolist()
        for u,v in zip(s,d):
            adj[r][u].append(v)
    BeamPath = lambda score, nodes: {"score": score, "nodes": nodes}
    beams = [BeamPath(float(torch.log(p[s].clamp(1e-9,1-1e-9))), [s]) for s in seeds]
    finished=[]
    for _ in range(max_hops):
        cand=[]
        for b in beams:
            u = b["nodes"][-1]
            for r, nbrs in adj.items():
                if u not in nbrs: continue
                ei = edges.get(r, None); es = edge_scores.get(r, None)
                for v in nbrs[u]:
                    es_val = 0.0
                    if ei is not None and es is not None and ei.numel()>0 and es.numel()>0:
                        mask = ((ei[0]==u)&(ei[1]==v))
                        if mask.any(): es_val = float(es[mask].max().item())
                    score = b["score"] + alpha_node*float(torch.log(p[v].clamp(1e-9,1-1e-9))) + (1-alpha_node)*es_val
                    cand.append(BeamPath(score, b["nodes"]+[v]))
        if not cand: break
        cand.sort(key=lambda x: x["score"], reverse=True)
        beams = cand[:beam_width]
        finished.extend(beams[:beam_width//2])
    # dedup by last-4 suffix
    seen=set(); uniq=[]
    for bp in sorted(finished, key=lambda x:x["score"], reverse=True):
        key = tuple(bp["nodes"][-4:])
        if key in seen: continue
        seen.add(key); uniq.append(bp)
        if len(uniq)>=beam_width: break
    return uniq

# ------------------------ CCS & CFAM ------------------------
def ccs_for_graph(model: CausalVulNet, g: HeteroData, causal_nodes: List[int], sink_mask: torch.Tensor):
    if not causal_nodes: return None
    device = next(model.parameters()).device
    g = g.to(device, non_blocking=False)
    # baseline
    model.eval()
    with torch.no_grad():
        logit, h, edges = model(g)
        p = torch.sigmoid(logit)
        p_sink = p[sink_mask].mean() if sink_mask.any() else p.mean()
    # intervention: zero input rows for causal nodes before projection
    st = g[NODE_TYPE]
    xs=[]
    if hasattr(st,"x_text"): xs.append(st.x_text.float().clone())
    if hasattr(st,"x"): xs.append(st.x.float().clone())
    if not xs: xs=[torch.zeros(st.num_nodes,1, device=device)]
    x = xs[0] if len(xs)==1 else torch.cat(xs, dim=1)
    x[causal_nodes] = 0.0
    # feed through encode manually
    h0 = F.relu(model.project_in(x))
    edges = {}
    for r in model.rels:
        et = (NODE_TYPE,r,NODE_TYPE)
        if et in g.edge_types:
            edges[r] = get_edge_index(g, et).to(device)
        else:
            edges[r] = torch.zeros((2,0), dtype=torch.long, device=device)
    for layer in model.layers:
        h0 = layer(h0, edges)
    logit2 = model.node_head(h0).squeeze(-1)
    p2 = torch.sigmoid(logit2)
    p2_sink = p2[sink_mask].mean() if sink_mask.any() else p2.mean()
    return float((p_sink - p2_sink).pow(2).item())

def cfam_for_graph(model: CausalVulNet, g: HeteroData, causal_nodes: List[int], spurious_nodes: List[int], sink_mask: torch.Tensor):
    if not causal_nodes: return None
    device = next(model.parameters()).device
    g = g.to(device, non_blocking=False)
    # get h and edges
    model.train()  # enables grad
    h, edges = model.encode(g)
    h.retain_grad()
    # score target = sum logits over sinks (or all if no sinks)
    logit = model.node_head(h).squeeze(-1)
    target_nodes = sink_mask.nonzero(as_tuple=False).view(-1)
    score = logit[target_nodes].sum() if target_nodes.numel()>0 else logit.mean()
    model.zero_grad(set_to_none=True)
    score.backward()
    # attribution = grad-norm per node
    attr = h.grad.norm(dim=1)
    c = torch.tensor(causal_nodes, device=device, dtype=torch.long)
    s = torch.tensor(spurious_nodes, device=device, dtype=torch.long) if spurious_nodes else torch.tensor([], device=device, dtype=torch.long)
    sum_c = float(attr[c].sum().item()) if c.numel()>0 else 0.0
    sum_s = float(attr[s].sum().item()) if s.numel()>0 else 0.0
    if sum_c + sum_s == 0.0: return None
    return float(sum_c / (sum_c + sum_s))

# ------------------------ Metrics ------------------------
def metrics_at_threshold(p: torch.Tensor, y: torch.Tensor, thr: float):
    pred = (p >= thr).float()
    tp = (pred*y).sum().item(); fp = (pred*(1-y)).sum().item(); fn = ((1-pred)*y).sum().item()
    prec = tp / (tp+fp+1e-9); rec = tp / (tp+fn+1e-9); f1 = 2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec, "recall":rec, "f1":f1}

# ------------------------ Train & Eval ------------------------
TIMESTAMP = str(int(time.time()))
OUT_ROOT = Path("out")/"cvul"/TIMESTAMP
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Saving to:", OUT_ROOT)

def train_and_eval():
    random.seed(0); torch.manual_seed(0)
    if DEVICE.type=="cuda":
        torch.cuda.manual_seed_all(0)
        try:
            torch.backends.cudnn.benchmark=True
        except Exception:
            pass

    ds_tr = GraphDir(TRAIN_DIR)
    dl_tr = DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=lambda xs: xs[0], pin_memory=(DEVICE.type=="cuda"))

    # infer rels & in-dim from a sample
    sample = safe_load(ds_tr.paths[0])
    etypes = [r for (s,r,t) in sample.edge_types if r in FORWARD_RELS]
    etypes = sorted(list(set(etypes)))
    st = sample[NODE_TYPE]
    in_dim_guess = (st.x_text.size(1) if hasattr(st,"x_text") else 0) + (st.x.size(1) if hasattr(st,"x") else 0)
    model = CausalVulNet(in_dim_guess, HIDDEN, LAYERS, etypes).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    focal = FocalBCELoss(alpha_pos=0.97, gamma=2.0)
    scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE.type=="cuda"))

    # ---------------- train ----------------
    for ep in range(1, EPOCHS+1):
        model.train(); loss_sum=0.0; n_seen=0
        for g in dl_tr:
            try:
                g = shrink_graph_if_needed(g).to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
                y = labels_from_graph_or_json(g)
                if y is None or float((y>0.5).sum().item())==0:
                    continue  # skip graphs without usable labels
                logit, h, edges = model(g)
                p = torch.sigmoid(logit)
                # imbalance
                keep = hard_negative_mask(p.detach(), y, max_ratio=NEG_POS_RATIO)
                y_m, p_m, logit_m = y[keep], p[keep], logit[keep]
                w = class_weights(y_m)
                # losses
                with torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE.type=="cuda")):
                    loss_node = focal(p_m, y_m, weight=w)
                    loss_edge = edge_participation_loss(h, edges, model, p)
                    # path losses only if JSON provides paths (optional, not required to prove inter-proc)
                    paths = []
                    jp = getattr(g, "_aug_json_path", None)
                    j  = load_json(jp) if jp else None
                    if isinstance(j, dict) and isinstance(j.get("vulnerable_paths"), list):
                        paths = [[int(k) for k in path if isinstance(k,(int,str)) and str(k).lstrip('-').isdigit()]
                                 for path in j["vulnerable_paths"] if isinstance(path,(list,tuple))]
                    loss_mono = monotonicity_loss_on_paths(logit, paths) if paths else logit.new_zeros(())
                    loss_rank = path_ranking_loss(logit, paths, [], margin=0.2) if paths else logit.new_zeros(())
                    loss = loss_node + 0.10*loss_edge + 0.10*loss_mono + 0.10*loss_rank

                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.5)
                scaler.step(opt); scaler.update()

                loss_sum += float(loss.detach().cpu()); n_seen += 1
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and DEVICE.type=="cuda":
                    torch.cuda.empty_cache()
                    continue
                else:
                    raise
        print(f"[train] epoch {ep} loss={(loss_sum/max(1,n_seen)):.6f} (graphs={n_seen})")

    # save model
    torch.save({"model_state": model.state_dict(), "rels": model.rels, "hidden": HIDDEN, "layers": LAYERS}, OUT_ROOT/"model.pt")

    # ---------------- eval helper ----------------
    def eval_split(dir_path: str, split_name: str):
        if not Path(dir_path).exists():
            return {"split": split_name, "overall": {"precision":0.0,"recall":0.0,"f1":0.0},
                    "thresholds":{"default":0.5},"n_graphs_used":0,"CCS_mean":None,"CFAM_mean":None,"CCS_count":0,"CFAM_count":0}
        ds = GraphDir(dir_path)
        dl = DataLoader(ds, batch_size=1, shuffle=False, collate_fn=lambda xs: xs[0], pin_memory=(DEVICE.type=="cuda"))
        tot_prec=tot_rec=tot_f1=0.0; n_m=0
        ccs_vals=[]; cfam_vals=[]
        thr = 0.25 if split_name=="valid" else 0.05  # light default; we’re not optimizing accuracy here
        for pt in ds.paths:
            try:
                g = safe_load(pt)
                g = shrink_graph_if_needed(g).to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
                y = labels_from_graph_or_json(g)
                if y is None:
                    continue
                with torch.no_grad():
                    logit, h, edges = model(g)
                    p = torch.sigmoid(logit)
                m = metrics_at_threshold(p.detach().cpu(), y.detach().cpu(), thr)
                tot_prec += m["precision"]; tot_rec += m["recall"]; tot_f1 += m["f1"]; n_m += 1

                # strict causal slicing from labels
                causal, roots, spurious = slice_from_labels(g, y)
                sink_mask = (y>0.5)
                if causal:
                    ccs = ccs_for_graph(model, g, causal, sink_mask.to(DEVICE))
                    cfam = cfam_for_graph(model, g, causal, spurious, sink_mask.to(DEVICE))
                    if ccs is not None: ccs_vals.append(ccs)
                    if cfam is not None: cfam_vals.append(cfam)
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and DEVICE.type=="cuda":
                    torch.cuda.empty_cache(); continue
                else:
                    raise
        res = {
            "split": split_name,
            "overall": {"precision":(tot_prec/max(1,n_m)), "recall":(tot_rec/max(1,n_m)), "f1":(tot_f1/max(1,n_m))},
            "thresholds": {"default": thr},
            "n_graphs_used": n_m,
            "CCS_mean": (sum(ccs_vals)/len(ccs_vals) if ccs_vals else None),
            "CFAM_mean": (sum(cfam_vals)/len(cfam_vals) if cfam_vals else None),
            "CCS_count": len(ccs_vals),
            "CFAM_count": len(cfam_vals),
        }
        return res

    # ---------------- eval all splits ----------------
    reports = {}
    for split, path in [("train", TRAIN_DIR), ("valid", VALID_DIR), ("test", TEST_DIR)]:
        print(f"[eval] {split} ...")
        reports[split] = eval_split(path, split)

    # ---------------- demo beam on one graph (valid→test→train order) ----------------
    demo_paths = []
    demo_meta  = {"split": None, "graph_path": None}
    for d, name in [(VALID_DIR,"valid"), (TEST_DIR,"test"), (TRAIN_DIR,"train")]:
        if not Path(d).exists(): continue
        ds = GraphDir(d)
        if not ds.paths: continue
        demo_meta["split"]=name; demo_meta["graph_path"]=ds.paths[0]
        g = safe_load(ds.paths[0]).to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
        with torch.no_grad():
            logit, h, edges = model(g)
            p = torch.sigmoid(logit)
            scores = model.edge_scores(h, edges)
        y = labels_from_graph_or_json(g)
        if y is not None and (y>0.5).any():
            causal, roots, _ = slice_from_labels(g, y)
            seeds = roots if roots else torch.topk(p, k=min(12, p.numel())).indices.tolist()
        else:
            seeds = torch.topk(p, k=min(12, p.numel())).indices.tolist()
        beam = run_beam(p, scores, edges, seeds, beam_width=16, max_hops=6, alpha_node=0.7)
        demo_paths = [{"score":bp["score"], "nodes":bp["nodes"]} for bp in beam[:10]]
        break

    # ---------------- save reports ----------------
    out = {
        "config": {"epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WEIGHT_DECAY,
                   "neg_pos_ratio":NEG_POS_RATIO,"device":str(DEVICE)},
        "reports": reports,
        "demo": {"meta": demo_meta, "paths": demo_paths},
        "notes": {
            "interprocedural": "Relations used: DFG, CFG, CALL, ARG2PARAM, RET2CALL, RET2LHS. Multi-root seeds supported.",
            "CCS": "p(X) vs p(do(X')), do(X') zeros features on causal-slice nodes (strict slicing from labels).",
            "CFAM": "Grad-norm attribution on hidden h; mass on causal vs causal+spurious."
        }
    }
    with open(OUT_ROOT/"reports.json","w",encoding="utf-8") as f:
        json.dump(out, f, indent=2)
    print("Saved:", OUT_ROOT/"model.pt", "and", OUT_ROOT/"reports.json")

# ---- run ----
try:
    train_and_eval()
except Exception as e:
    print("!! Exception:", e)
    traceback.print_exc()


[env] torch=2.4.1+cu121 device=cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Saving to: out\cvul\1760246212
!! Exception: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.



Traceback (most recent call last):
  File "C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_32840\303621787.py", line 707, in <module>
    train_and_eval()
  File "C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_32840\303621787.py", line 543, in train_and_eval
    random.seed(0); torch.manual_seed(0)
                    ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\.venv\Lib\site-packages\torch\_compile.py", line 31, in inner
    return disable_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py", line 600, in _fn
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\.venv\Lib\site-packages\torch\random.py", line 46, in manual_seed
    torch.cuda.manual_seed_all(seed)
  File "c:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\.venv\Lib\si

In [ ]:
# Causal-Vul: inter-procedural demo trainer + CCS/CFAM evaluation (drop-in, single cell)
# ----------------------------------------------------------------------------------
# What this cell does
# - Loads HeteroData graphs from Dataset/{train,valid,test}/hetero_ready_gcbert/*.pt
# - Trains a light relational GNN with learned per-relation guidance (no torch-scatter deps)
# - Demonstrates inter-procedural reasoning via CALL/ARG2PARAM/RET2CALL/RET2LHS (+ CFG/DFG)
# - Computes standard node metrics (precision/recall/F1) per split
# - Computes CCS and CFAM strictly from program slicing induced by labels (no heuristics)
#   * CSS: (p(X) - p(do(X')))² where do(X') = zero out causal-slice node features
#   * CFAM: attribution mass on causal / (causal + spurious), using grad‖∂logit/∂h‖
# - Multi-root is supported automatically: roots are slice nodes without in-slice predecessors
# - Saves everything under out/cvul/<timestamp>/
#
# Notes
# - This code avoids torch-geometric CUDA extensions and uses only edge_index tensors.
# - It tolerates varying node-feature dimensions across graphs.
# - It does NOT try to maximize accuracy; it focuses on proving inter-procedural flow + CCS/CFAM.
#
# Minimal knobs
EPOCHS          = 2         # keep small for demo; raise if you like
HIDDEN          = 64        # modest capacity to avoid OOM
LAYERS          = 3
LR              = 2e-3
WEIGHT_DECAY    = 1e-4
NEG_POS_RATIO   = 20        # hard-negative sampling cap (#neg <= ratio * #pos)
USE_AMP         = True      # AMP if CUDA is available
MAX_NODES       = 12000     # safety shrinker
MAX_EDGES       = 200000
NODE_TYPE       = "node"

# Paths
TRAIN_DIR = "Dataset/train/hetero_ready_gcbert"
VALID_DIR = "Dataset/valid/hetero_ready_gcbert"
TEST_DIR  = "Dataset/test/hetero_ready_gcbert"

# ------------------------ Imports & basic checks ------------------------
import os, json, math, time, gc, random, hashlib, re, traceback
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict, deque

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from torch_geometric.data import HeteroData
except Exception as e:
    raise RuntimeError("torch-geometric must be installed and importable to load HeteroData graphs") from e

def pick_device():
    if torch.cuda.is_available():
        # simple probe to ensure VRAM is usable
        try:
            _ = torch.empty(1024, device="cuda")
            d = torch.device("cuda")
            print(f"[env] torch={torch.__version__} device=cuda")
            try:
                print("GPU:", torch.cuda.get_device_name(0))
            except Exception:
                pass
            return d
        except Exception:
            pass
    print(f"[env] torch={torch.__version__} device=cpu")
    return torch.device("cpu")

DEVICE = pick_device()

# ------------------------ IO helpers ------------------------
def near_json(pt_path: str) -> Optional[str]:
    """Find a sidecar JSON (augmented) near a .pt file (optional, used only for labels if present)."""
    p = Path(pt_path)
    candidates = [
        p.with_suffix(".json"),
        p.with_suffix(".aug.json"),
        p.with_suffix(".unified.json"),
        Path("notebooks")/"Dataset"/"train"/"unified_aug"/(p.stem+".json"),
        Path("notebooks")/"Dataset"/"valid"/"unified_aug"/(p.stem+".json"),
        Path("notebooks")/"Dataset"/"test"/"unified_aug"/(p.stem+".json"),
    ]
    for c in candidates:
        if c.exists(): return str(c)
    return None

def load_json(path: str) -> Optional[dict]:
    if not path or not os.path.exists(path): return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def safe_load(path: str):
    # plain torch.load (your env is torch 2.4.1, so no weights_only restriction)
    return torch.load(path, map_location="cpu")

class GraphDir(Dataset):
    def __init__(self, dir_path: str):
        self.paths = sorted([str(p) for p in Path(dir_path).glob("*.pt")])
        if not self.paths:
            raise FileNotFoundError(f"No .pt found in {dir_path}")
        print(f"[DATA] {len(self.paths)} graphs in {dir_path}")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        pt = self.paths[i]
        g  = safe_load(pt)
        g.__dict__["_aug_json_path"] = near_json(pt)
        return g

# ------------------------ Graph utilities ------------------------
def get_edge_index(g: HeteroData, r: Tuple[str,str,str]) -> torch.Tensor:
    if r not in g.edge_types:
        dev = (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE], "x") else "cpu")
        return torch.zeros((2,0), dtype=torch.long, device=dev)
    store = g[r]
    ei = getattr(store, "edge_index", None)
    if ei is None:
        adj_t = getattr(store, "adj_t", None)
        if adj_t is not None:
            row, col, _ = adj_t.coo()
            ei = torch.stack([col, row], dim=0)
        else:
            dev = (g[NODE_TYPE].x.device if hasattr(g[NODE_TYPE], "x") else "cpu")
            ei = torch.zeros((2,0), dtype=torch.long, device=dev)
    return ei

# Allowed relation set (forward); we’ll auto-add reverse in adjacency when needed
FORWARD_RELS = ("DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS","AST")

def build_adj(g: HeteroData):
    fwd = defaultdict(list); rev = defaultdict(list)
    for (s,r,t) in g.edge_types:
        if r not in FORWARD_RELS: 
            # also allow *_REV if present
            if r.endswith("_REV"): pass
            else: continue
        ei = get_edge_index(g, (s,r,t))
        if ei.numel()==0: continue
        for u,v in zip(ei[0].tolist(), ei[1].tolist()):
            fwd[(s,r,t)].append((u,v))
            rev[(s,r,t)].append((v,u))
    return fwd, rev

def shrink_graph_if_needed(g: HeteroData) -> HeteroData:
    st = g[NODE_TYPE]; N = int(st.num_nodes)
    total_edges = 0
    for et in g.edge_types:
        total_edges += int(get_edge_index(g, et).size(1))
    if N <= MAX_NODES and total_edges <= MAX_EDGES:
        return g
    keep = set()
    for r in ("CALL","ARG2PARAM","RET2CALL","RET2LHS","DFG","CFG"):
        et = (NODE_TYPE, r, NODE_TYPE)
        if et in g.edge_types:
            ei = get_edge_index(g, et)
            if ei.numel()>0:
                keep.update(ei[0].tolist()); keep.update(ei[1].tolist())
    if not keep:
        # sample first MAX_NODES nodes
        keep = set(range(min(N, MAX_NODES)))
    keep = sorted(list(keep))[:MAX_NODES]
    idx = {old:i for i,old in enumerate(keep)}
    g2 = HeteroData()
    if hasattr(st, "x"): g2[NODE_TYPE].x = st.x[keep]
    if hasattr(st, "x_text"): g2[NODE_TYPE].x_text = st.x_text[keep]
    if hasattr(st, "nid"): g2[NODE_TYPE].nid = st.nid[keep]
    if hasattr(st, "y"): g2[NODE_TYPE].y = st.y[keep]
    g2[NODE_TYPE].num_nodes = len(keep)
    for et in g.edge_types:
        ei = get_edge_index(g, et)
        if ei.numel()==0:
            g2[et].edge_index = ei
            continue
        src, dst = ei[0].tolist(), ei[1].tolist()
        new_src, new_dst = [], []
        for u,v in zip(src, dst):
            if u in idx and v in idx:
                new_src.append(idx[u]); new_dst.append(idx[v])
        if new_src:
            g2[et].edge_index = torch.tensor([new_src, new_dst], dtype=torch.long)
        else:
            g2[et].edge_index = torch.zeros((2,0), dtype=torch.long)
    g2.__dict__["_aug_json_path"] = getattr(g, "_aug_json_path", None)
    return g2

# ------------------------ Labels (no heuristics) ------------------------
def labels_from_graph_or_json(g: HeteroData) -> Optional[torch.Tensor]:
    # Try from graph first
    st = g[NODE_TYPE]
    if hasattr(st, "y"):
        y = st.y.float()
        if y.dim()==1 and y.numel()==st.num_nodes:
            return y
        if y.dim()==2 and y.size(1)==1:
            return y.view(-1).float()
    # Try sidecar JSON (treat any node with "vulnerable"==1 or "is_sink"==true as positive)
    jp = getattr(g, "_aug_json_path", None)
    j  = load_json(jp) if jp else None
    if isinstance(j, dict) and isinstance(j.get("nodes"), list):
        arr = j["nodes"]; y = torch.zeros(st.num_nodes, dtype=torch.float32)
        # If nid exists, map by external ids; else assume order
        id_map = None
        if hasattr(st, "nid"):
            ids = st.nid.view(-1).tolist()
            id_map = {int(ids[i]): i for i in range(len(ids))}
        for idx,n in enumerate(arr):
            if not isinstance(n, dict): continue
            pos = False
            for k in ("y","label","vulnerable","is_vuln","is_sink"):
                if k in n:
                    v = n[k]; 
                    if isinstance(v, (int,float)) and float(v)>0.5: pos=True
                    if isinstance(v, str) and v.strip().lower() in ("1","true","yes"): pos=True
            if id_map is not None and "_id" in n:
                ext = int(n["_id"]) if isinstance(n["_id"], (int,str)) and str(n["_id"]).lstrip("-").isdigit() else None
                if ext is not None and ext in id_map and pos:
                    y[id_map[ext]] = 1.0
            else:
                if idx < y.numel() and pos:
                    y[idx] = 1.0
        if (y>0.5).any():
            return y
    return None

# ------------------------ Causal slicing (no heuristics) ------------------------
ALLOWED_FOR_SLICE = ("DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS")

def build_simple_adj(g: HeteroData):
    fwd = defaultdict(list); rev = defaultdict(list)
    for r in ALLOWED_FOR_SLICE:
        et = (NODE_TYPE,r,NODE_TYPE)
        if et not in g.edge_types: continue
        ei = get_edge_index(g, et)
        if ei.numel()==0: continue
        s,d = ei[0].tolist(), ei[1].tolist()
        for u,v in zip(s,d):
            fwd[u].append(v)
            rev[v].append(u)
    return fwd, rev

def slice_from_labels(g: HeteroData, y: torch.Tensor):
    """Causal slice strictly from labels:
       sinks = {i | y_i=1}, backward slice via rev edges, then forward slice; causal = B∩F; 
       roots = causal nodes with zero in-degree inside B (multi-root)."""
    sinks=[i for i in range(y.numel()) if float(y[i])>0.5]
    if not sinks: return [], [], []
    fwd, rev = build_simple_adj(g)
    B=set(); dq=deque(sinks)
    while dq:
        u=dq.popleft()
        if u in B: continue
        B.add(u)
        for v in rev.get(u, []):
            if v not in B: dq.append(v)
    F=set(); dq=deque(list(B))
    while dq:
        u=dq.popleft()
        if u in F: continue
        F.add(u)
        for v in fwd.get(u, []):
            if v not in F: dq.append(v)
    causal = sorted(list(B.intersection(F)))
    roots  = sorted([u for u in B if not any((v in B) for v in rev.get(u, []))])
    spurious = [i for i in range(y.numel()) if i not in set(causal)]
    return causal, roots, spurious

# ------------------------ Model ------------------------
class ProjectIn(nn.Module):
    """Caches a Linear for every encountered input dim to avoid matmul shape errors."""
    def __init__(self, hidden: int):
        super().__init__()
        self.hidden = hidden
        self.proj = nn.ModuleDict()  # key = str(dim)
    def forward(self, x: torch.Tensor):
        d = x.size(1); k = str(d)
        if k not in self.proj:
            lin = nn.Linear(d, self.hidden)
            nn.init.xavier_uniform_(lin.weight); nn.init.zeros_(lin.bias)
            self.proj[k] = lin.to(x.device, dtype=x.dtype)
        return self.proj[k](x)

class RelLayer(nn.Module):
    """Simple per-relation mean aggregator + LayerNorm + SiLU. No scatter deps."""
    def __init__(self, hidden: int, rels: List[str]):
        super().__init__()
        self.rels = rels
        self.self_lin = nn.Linear(hidden, hidden)
        self.msg_lin  = nn.ModuleDict({r: nn.Linear(hidden, hidden) for r in rels})
        self.ln = nn.LayerNorm(hidden)
    def forward(self, h: torch.Tensor, edges: Dict[str, torch.Tensor]) -> torch.Tensor:
        out = self.self_lin(h)
        N = h.size(0)
        for r in self.rels:
            ei = edges.get(r, None)
            if ei is None or ei.numel()==0: continue
            src, dst = ei[0], ei[1]
            m = self.msg_lin[r](h[src])
            agg = torch.zeros_like(h)
            agg.index_add_(0, dst, m)
            deg = torch.zeros(N, device=h.device).index_add_(0, dst, torch.ones_like(dst, dtype=torch.float32))
            out = out + agg / deg.clamp(min=1).unsqueeze(-1)
        out = self.ln(out)
        return F.silu(out)

class CausalVulNet(nn.Module):
    def __init__(self, in_dim_guess: int, hidden: int, layers: int, rels: List[str]):
        super().__init__()
        self.rels = rels
        self.project_in = ProjectIn(hidden)
        self.layers = nn.ModuleList([RelLayer(hidden, rels) for _ in range(layers)])
        self.node_head = nn.Sequential(nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 1))
        # learned edge guidance
        self.rel_emb = nn.Embedding(num_embeddings=len(rels), embedding_dim=hidden)
        self.edge_head = nn.Sequential(nn.Linear(hidden*3, hidden), nn.SiLU(), nn.Linear(hidden, 1))
        # edge participation (aux head)
        self.part_head = nn.Sequential(nn.Linear(hidden*2, hidden), nn.SiLU(), nn.Linear(hidden, 1))

    def encode(self, g: HeteroData) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        st = g[NODE_TYPE]
        xs = []
        if hasattr(st, "x_text"): xs.append(st.x_text.float())
        if hasattr(st, "x"):      xs.append(st.x.float())
        if not xs:
            xs = [torch.zeros(st.num_nodes, 1, device=st.x.device if hasattr(st,"x") else "cpu")]
        x = xs[0] if len(xs)==1 else torch.cat(xs, dim=1)
        h = F.relu(self.project_in(x))
        # collect edges (dict rel->edge_index)
        edges = {}
        for r in self.rels:
            et = (NODE_TYPE, r, NODE_TYPE)
            if et in g.edge_types:
                ei = get_edge_index(g, et)
                edges[r] = ei.to(h.device)
            else:
                edges[r] = torch.zeros((2,0), dtype=torch.long, device=h.device)
        for layer in self.layers:
            h = layer(h, edges)
        return h, edges

    def forward(self, g: HeteroData):
        h, edges = self.encode(g)
        logit = self.node_head(h).squeeze(-1)
        return logit, h, edges

    def edge_scores(self, h: torch.Tensor, edges: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        scores = {}
        for i, r in enumerate(self.rels):
            ei = edges.get(r, None)
            if ei is None or ei.numel()==0:
                scores[r] = torch.empty(0, device=h.device)
                continue
            src, dst = ei[0], ei[1]
            t = self.rel_emb(torch.tensor(i, device=h.device)).view(1,-1).expand(src.numel(), -1)
            s = self.edge_head(torch.cat([h[src], h[dst], t], dim=-1)).squeeze(-1)
            scores[r] = s
        return scores

# ------------------------ Losses ------------------------
class FocalBCELoss(nn.Module):
    def __init__(self, alpha_pos=0.95, gamma=2.0):
        super().__init__()
        self.alpha_pos=alpha_pos; self.gamma=gamma
    def forward(self, p: torch.Tensor, y: torch.Tensor, weight=None):
        eps=1e-6; p=p.clamp(eps,1-eps)
        pt = torch.where(y>0.5, p, 1-p)
        w  = (self.alpha_pos*(y>0.5).float() + (1-self.alpha_pos)*(y<=0.5).float())
        if weight is not None: w = w*weight
        loss = - ((1-pt)**self.gamma) * ( y*torch.log(p) + (1-y)*torch.log(1-p) ) * w
        return loss.mean()

def class_weights(y: torch.Tensor):
    pos = (y>0.5).sum().item()
    neg = y.numel() - pos
    w = torch.ones_like(y)
    if pos>0: w[y>0.5] = (neg+1e-6)/(pos+1e-6)
    return w

def hard_negative_mask(p: torch.Tensor, y: torch.Tensor, max_ratio=NEG_POS_RATIO):
    pos_idx = (y>0.5).nonzero(as_tuple=False).view(-1)
    neg_idx = (y<=0.5).nonzero(as_tuple=False).view(-1)
    k = int(min(len(neg_idx), max_ratio * max(1,len(pos_idx))))
    if k <= 0: return (y>0.5)
    if len(neg_idx)==0: return (y>0.5)
    # hardest negatives by p
    vals, inds = torch.topk(p[neg_idx], k=k)
    keep_neg = neg_idx[inds]
    mask = torch.zeros_like(y, dtype=torch.bool)
    if len(pos_idx)>0: mask[pos_idx] = True
    mask[keep_neg] = True
    return mask

def edge_participation_loss(h, edges, model: CausalVulNet, p: torch.Tensor):
    """Encourage edges bridging high-p nodes to have high participation."""
    loss = 0.0; cnt=0
    for r, ei in edges.items():
        if ei.numel()==0: continue
        src, dst = ei[0], ei[1]
        logits = model.part_head(torch.cat([h[src], h[dst]], dim=-1)).squeeze(-1)
        target = torch.minimum(p[src], p[dst]).detach()
        loss = loss + F.binary_cross_entropy_with_logits(logits, target)
        cnt += 1
    return loss / max(1, cnt)

def monotonicity_loss_on_paths(logit: torch.Tensor, paths: List[List[int]]):
    loss = 0.0; cnt=0
    for path in paths:
        if len(path) < 2: continue
        z = logit[path]
        # enforce non-decreasing along the path
        diffs = z[1:] - z[:-1]
        loss = loss + F.relu(-diffs).mean()
        cnt += 1
    return loss / max(1, cnt)

def path_ranking_loss(logit: torch.Tensor, pos_paths: List[List[int]], neg_paths: List[List[int]], margin=0.5):
    """Path score = mean(sigmoid(logit) along nodes)."""
    if not pos_paths or not neg_paths: 
        return logit.new_zeros(())
    def path_score(path):
        return torch.sigmoid(logit[path]).mean()
    loss = 0.0; cnt=0
    m = logit.new_tensor(margin)
    for pp in pos_paths:
        sp = path_score(pp)
        for np in neg_paths[:3]:  # sample a few
            sn = path_score(np)
            loss = loss + F.relu(m - (sp - sn))
            cnt += 1
    return loss / max(1, cnt)

# ------------------------ Beam search ------------------------
def run_beam(p: torch.Tensor, edge_scores: Dict[str, torch.Tensor], edges: Dict[str, torch.Tensor],
             seeds: List[int], beam_width=16, max_hops=6, alpha_node=0.7):
    # Build simple adjacency
    adj = defaultdict(lambda: defaultdict(list))  # rel -> u -> [v,..]
    for r, ei in edges.items():
        if ei is None or ei.numel()==0: continue
        s,d = ei[0].tolist(), ei[1].tolist()
        for u,v in zip(s,d):
            adj[r][u].append(v)
    BeamPath = lambda score, nodes: {"score": score, "nodes": nodes}
    beams = [BeamPath(float(torch.log(p[s].clamp(1e-9,1-1e-9))), [s]) for s in seeds]
    finished=[]
    for _ in range(max_hops):
        cand=[]
        for b in beams:
            u = b["nodes"][-1]
            for r, nbrs in adj.items():
                if u not in nbrs: continue
                ei = edges.get(r, None); es = edge_scores.get(r, None)
                for v in nbrs[u]:
                    es_val = 0.0
                    if ei is not None and es is not None and ei.numel()>0 and es.numel()>0:
                        mask = ((ei[0]==u)&(ei[1]==v))
                        if mask.any(): es_val = float(es[mask].max().item())
                    score = b["score"] + alpha_node*float(torch.log(p[v].clamp(1e-9,1-1e-9))) + (1-alpha_node)*es_val
                    cand.append(BeamPath(score, b["nodes"]+[v]))
        if not cand: break
        cand.sort(key=lambda x: x["score"], reverse=True)
        beams = cand[:beam_width]
        finished.extend(beams[:beam_width//2])
    # dedup by last-4 suffix
    seen=set(); uniq=[]
    for bp in sorted(finished, key=lambda x:x["score"], reverse=True):
        key = tuple(bp["nodes"][-4:])
        if key in seen: continue
        seen.add(key); uniq.append(bp)
        if len(uniq)>=beam_width: break
    return uniq

# ------------------------ CCS & CFAM ------------------------
def ccs_for_graph(model: CausalVulNet, g: HeteroData, causal_nodes: List[int], sink_mask: torch.Tensor):
    if not causal_nodes: return None
    device = next(model.parameters()).device
    g = g.to(device, non_blocking=False)
    # baseline
    model.eval()
    with torch.no_grad():
        logit, h, edges = model(g)
        p = torch.sigmoid(logit)
        p_sink = p[sink_mask].mean() if sink_mask.any() else p.mean()
    # intervention: zero input rows for causal nodes before projection
    st = g[NODE_TYPE]
    xs=[]
    if hasattr(st,"x_text"): xs.append(st.x_text.float().clone())
    if hasattr(st,"x"): xs.append(st.x.float().clone())
    if not xs: xs=[torch.zeros(st.num_nodes,1, device=device)]
    x = xs[0] if len(xs)==1 else torch.cat(xs, dim=1)
    x[causal_nodes] = 0.0
    # feed through encode manually
    h0 = F.relu(model.project_in(x))
    edges = {}
    for r in model.rels:
        et = (NODE_TYPE,r,NODE_TYPE)
        if et in g.edge_types:
            edges[r] = get_edge_index(g, et).to(device)
        else:
            edges[r] = torch.zeros((2,0), dtype=torch.long, device=device)
    for layer in model.layers:
        h0 = layer(h0, edges)
    logit2 = model.node_head(h0).squeeze(-1)
    p2 = torch.sigmoid(logit2)
    p2_sink = p2[sink_mask].mean() if sink_mask.any() else p2.mean()
    return float((p_sink - p2_sink).pow(2).item())

def cfam_for_graph(model: CausalVulNet, g: HeteroData, causal_nodes: List[int], spurious_nodes: List[int], sink_mask: torch.Tensor):
    if not causal_nodes: return None
    device = next(model.parameters()).device
    g = g.to(device, non_blocking=False)
    # get h and edges
    model.train()  # enables grad
    h, edges = model.encode(g)
    h.retain_grad()
    # score target = sum logits over sinks (or all if no sinks)
    logit = model.node_head(h).squeeze(-1)
    target_nodes = sink_mask.nonzero(as_tuple=False).view(-1)
    score = logit[target_nodes].sum() if target_nodes.numel()>0 else logit.mean()
    model.zero_grad(set_to_none=True)
    score.backward()
    # attribution = grad-norm per node
    attr = h.grad.norm(dim=1)
    c = torch.tensor(causal_nodes, device=device, dtype=torch.long)
    s = torch.tensor(spurious_nodes, device=device, dtype=torch.long) if spurious_nodes else torch.tensor([], device=device, dtype=torch.long)
    sum_c = float(attr[c].sum().item()) if c.numel()>0 else 0.0
    sum_s = float(attr[s].sum().item()) if s.numel()>0 else 0.0
    if sum_c + sum_s == 0.0: return None
    return float(sum_c / (sum_c + sum_s))

# ------------------------ Metrics ------------------------
def metrics_at_threshold(p: torch.Tensor, y: torch.Tensor, thr: float):
    pred = (p >= thr).float()
    tp = (pred*y).sum().item(); fp = (pred*(1-y)).sum().item(); fn = ((1-pred)*y).sum().item()
    prec = tp / (tp+fp+1e-9); rec = tp / (tp+fn+1e-9); f1 = 2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec, "recall":rec, "f1":f1}

# ------------------------ Train & Eval ------------------------
TIMESTAMP = str(int(time.time()))
OUT_ROOT = Path("out")/"cvul"/TIMESTAMP
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Saving to:", OUT_ROOT)

def train_and_eval():
    random.seed(0); torch.manual_seed(0)
    if DEVICE.type=="cuda":
        torch.cuda.manual_seed_all(0)
        try:
            torch.backends.cudnn.benchmark=True
        except Exception:
            pass

    ds_tr = GraphDir(TRAIN_DIR)
    dl_tr = DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=lambda xs: xs[0], pin_memory=(DEVICE.type=="cuda"))

    # infer rels & in-dim from a sample
    sample = safe_load(ds_tr.paths[0])
    etypes = [r for (s,r,t) in sample.edge_types if r in FORWARD_RELS]
    etypes = sorted(list(set(etypes)))
    st = sample[NODE_TYPE]
    in_dim_guess = (st.x_text.size(1) if hasattr(st,"x_text") else 0) + (st.x.size(1) if hasattr(st,"x") else 0)
    model = CausalVulNet(in_dim_guess, HIDDEN, LAYERS, etypes).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    focal = FocalBCELoss(alpha_pos=0.97, gamma=2.0)
    scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE.type=="cuda"))

    # ---------------- train ----------------
    for ep in range(1, EPOCHS+1):
        model.train(); loss_sum=0.0; n_seen=0
        for g in dl_tr:
            try:
                g = shrink_graph_if_needed(g).to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
                y = labels_from_graph_or_json(g)
                if y is None or float((y>0.5).sum().item())==0:
                    continue  # skip graphs without usable labels
                logit, h, edges = model(g)
                p = torch.sigmoid(logit)
                # imbalance
                keep = hard_negative_mask(p.detach(), y, max_ratio=NEG_POS_RATIO)
                y_m, p_m, logit_m = y[keep], p[keep], logit[keep]
                w = class_weights(y_m)
                # losses
                with torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE.type=="cuda")):
                    loss_node = focal(p_m, y_m, weight=w)
                    loss_edge = edge_participation_loss(h, edges, model, p)
                    # path losses only if JSON provides paths (optional, not required to prove inter-proc)
                    paths = []
                    jp = getattr(g, "_aug_json_path", None)
                    j  = load_json(jp) if jp else None
                    if isinstance(j, dict) and isinstance(j.get("vulnerable_paths"), list):
                        paths = [[int(k) for k in path if isinstance(k,(int,str)) and str(k).lstrip('-').isdigit()]
                                 for path in j["vulnerable_paths"] if isinstance(path,(list,tuple))]
                    loss_mono = monotonicity_loss_on_paths(logit, paths) if paths else logit.new_zeros(())
                    loss_rank = path_ranking_loss(logit, paths, [], margin=0.2) if paths else logit.new_zeros(())
                    loss = loss_node + 0.10*loss_edge + 0.10*loss_mono + 0.10*loss_rank

                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.5)
                scaler.step(opt); scaler.update()

                loss_sum += float(loss.detach().cpu()); n_seen += 1
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and DEVICE.type=="cuda":
                    torch.cuda.empty_cache()
                    continue
                else:
                    raise
        print(f"[train] epoch {ep} loss={(loss_sum/max(1,n_seen)):.6f} (graphs={n_seen})")

    # save model
    torch.save({"model_state": model.state_dict(), "rels": model.rels, "hidden": HIDDEN, "layers": LAYERS}, OUT_ROOT/"model.pt")

    # ---------------- eval helper ----------------
    def eval_split(dir_path: str, split_name: str):
        if not Path(dir_path).exists():
            return {"split": split_name, "overall": {"precision":0.0,"recall":0.0,"f1":0.0},
                    "thresholds":{"default":0.5},"n_graphs_used":0,"CCS_mean":None,"CFAM_mean":None,"CCS_count":0,"CFAM_count":0}
        ds = GraphDir(dir_path)
        dl = DataLoader(ds, batch_size=1, shuffle=False, collate_fn=lambda xs: xs[0], pin_memory=(DEVICE.type=="cuda"))
        tot_prec=tot_rec=tot_f1=0.0; n_m=0
        ccs_vals=[]; cfam_vals=[]
        thr = 0.25 if split_name=="valid" else 0.05  # light default; we’re not optimizing accuracy here
        for pt in ds.paths:
            try:
                g = safe_load(pt)
                g = shrink_graph_if_needed(g).to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
                y = labels_from_graph_or_json(g)
                if y is None:
                    continue
                with torch.no_grad():
                    logit, h, edges = model(g)
                    p = torch.sigmoid(logit)
                m = metrics_at_threshold(p.detach().cpu(), y.detach().cpu(), thr)
                tot_prec += m["precision"]; tot_rec += m["recall"]; tot_f1 += m["f1"]; n_m += 1

                # strict causal slicing from labels
                causal, roots, spurious = slice_from_labels(g, y)
                sink_mask = (y>0.5)
                if causal:
                    ccs = ccs_for_graph(model, g, causal, sink_mask.to(DEVICE))
                    cfam = cfam_for_graph(model, g, causal, spurious, sink_mask.to(DEVICE))
                    if ccs is not None: ccs_vals.append(ccs)
                    if cfam is not None: cfam_vals.append(cfam)
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and DEVICE.type=="cuda":
                    torch.cuda.empty_cache(); continue
                else:
                    raise
        res = {
            "split": split_name,
            "overall": {"precision":(tot_prec/max(1,n_m)), "recall":(tot_rec/max(1,n_m)), "f1":(tot_f1/max(1,n_m))},
            "thresholds": {"default": thr},
            "n_graphs_used": n_m,
            "CCS_mean": (sum(ccs_vals)/len(ccs_vals) if ccs_vals else None),
            "CFAM_mean": (sum(cfam_vals)/len(cfam_vals) if cfam_vals else None),
            "CCS_count": len(ccs_vals),
            "CFAM_count": len(cfam_vals),
        }
        return res

    # ---------------- eval all splits ----------------
    reports = {}
    for split, path in [("train", TRAIN_DIR), ("valid", VALID_DIR), ("test", TEST_DIR)]:
        print(f"[eval] {split} ...")
        reports[split] = eval_split(path, split)

    # ---------------- demo beam on one graph (valid→test→train order) ----------------
    demo_paths = []
    demo_meta  = {"split": None, "graph_path": None}
    for d, name in [(VALID_DIR,"valid"), (TEST_DIR,"test"), (TRAIN_DIR,"train")]:
        if not Path(d).exists(): continue
        ds = GraphDir(d)
        if not ds.paths: continue
        demo_meta["split"]=name; demo_meta["graph_path"]=ds.paths[0]
        g = safe_load(ds.paths[0]).to(DEVICE, non_blocking=(DEVICE.type=="cuda"))
        with torch.no_grad():
            logit, h, edges = model(g)
            p = torch.sigmoid(logit)
            scores = model.edge_scores(h, edges)
        y = labels_from_graph_or_json(g)
        if y is not None and (y>0.5).any():
            causal, roots, _ = slice_from_labels(g, y)
            seeds = roots if roots else torch.topk(p, k=min(12, p.numel())).indices.tolist()
        else:
            seeds = torch.topk(p, k=min(12, p.numel())).indices.tolist()
        beam = run_beam(p, scores, edges, seeds, beam_width=16, max_hops=6, alpha_node=0.7)
        demo_paths = [{"score":bp["score"], "nodes":bp["nodes"]} for bp in beam[:10]]
        break

    # ---------------- save reports ----------------
    out = {
        "config": {"epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WEIGHT_DECAY,
                   "neg_pos_ratio":NEG_POS_RATIO,"device":str(DEVICE)},
        "reports": reports,
        "demo": {"meta": demo_meta, "paths": demo_paths},
        "notes": {
            "interprocedural": "Relations used: DFG, CFG, CALL, ARG2PARAM, RET2CALL, RET2LHS. Multi-root seeds supported.",
            "CCS": "p(X) vs p(do(X')), do(X') zeros features on causal-slice nodes (strict slicing from labels).",
            "CFAM": "Grad-norm attribution on hidden h; mass on causal vs causal+spurious."
        }
    }
    with open(OUT_ROOT/"reports.json","w",encoding="utf-8") as f:
        json.dump(out, f, indent=2)
    print("Saved:", OUT_ROOT/"model.pt", "and", OUT_ROOT/"reports.json")

# ---- run ----
try:
    train_and_eval()
except Exception as e:
    print("!! Exception:", e)
    traceback.print_exc()


In [1]:
# ================= Causal-Vul: inter-proc, beam-only slice, CCS/CFAM, early probes =================
import os, json, time, math, random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# ---------------- Config ----------------
DATA_DIRS = {
    "train": r"Dataset/train/hetero_ready_gcbert",
    "valid": r"Dataset/valid/hetero_ready_gcbert",
    "test" : r"Dataset/test/hetero_ready_gcbert",
}
RELATIONS = ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS"]
ADD_SUMMARY_EDGES = True  # adds DFG_THIN = DFG ∪ call-family

EPOCHS  = 5
HIDDEN  = 64
LAYERS  = 3
LR      = 2e-3
WD      = 1e-4
NEG_POS_RATIO = 20
USE_AMP = True
RNG_SEED = 23

# Beam (model-guided; no heuristics)
SEED_K        = 8
BEAM_WIDTH    = 24
BEAM_MAX_HOPS = 5
ALPHA_NODE    = 0.7

# Provenance / calibration
DATASET_WEIGHTS = {"default": 1.0}
THRESHOLDS_BY_SOURCE = {"default": 0.25}  # will be re-fit on valid if possible

# Eval budget
EVAL_MAX = 256   # set None to evaluate ALL graphs

OUT_ROOT = Path("out/cvul")/str(int(time.time()))
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# -------------- Device & determinism --------------
def pick_device(min_free_mb=256):
    if not torch.cuda.is_available(): return "cpu"
    try:
        free,_ = torch.cuda.mem_get_info()
        return "cuda" if (free//(1024**2)) >= min_free_mb else "cpu"
    except: return "cuda"

DEVICE = pick_device()
print(f"[env] torch={torch.__version__} device={DEVICE}")
if DEVICE=="cuda":
    try: print("GPU:", torch.cuda.get_device_name(0))
    except: pass
random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if DEVICE=="cuda":
    try: torch.cuda.manual_seed_all(RNG_SEED)
    except: pass

def vram_info():
    if DEVICE!="cuda": return {"device":"cpu"}
    try:
        free,total = torch.cuda.mem_get_info()
        return {"device":"cuda","freeMB":free//(1024**2),"totalMB":total//(1024**2)}
    except: return {"device":"cuda"}

# -------------- IO & normalization --------------
def safe_load(p): 
    p=Path(p)
    if p.suffix.lower()==".json": return json.loads(p.read_text())
    return torch.load(p, map_location="cpu")

def to_long_2(e):
    if e is None: return torch.zeros((2,0), dtype=torch.long)
    t=torch.as_tensor(e)
    if t.ndim==2 and t.shape[0]==2: return t.long().contiguous()
    if t.ndim==2 and t.shape[1]==2: return t.t().long().contiguous()
    if isinstance(e,(list,tuple)) and len(e)==2:
        s=torch.as_tensor(e[0]).view(-1).long()
        d=torch.as_tensor(e[1]).view(-1).long()
        return torch.stack([s,d], dim=0)
    if t.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    raise RuntimeError("edge_index must be [2,E], [E,2], or (src,dst)")

def sanitize_edges(N:int, ei:torch.Tensor):
    if ei is None or ei.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    s,d=ei
    m=(s>=0)&(s<N)&(d>=0)&(d<N)
    if m.any(): return torch.stack([s[m], d[m]], dim=0)
    return torch.zeros((2,0), dtype=torch.long)

def coerce_x_any(obj):
    # common fields
    for k in ["x","features","node_features","feat","emb","gcbert","gcb_x","x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj, dict) and k in obj:
            t=torch.as_tensor(obj[k]).float()
            if t.ndim==1: t=t.view(-1,1)
            return t
    xs=[]
    for k in ["x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj, dict) and k in obj:
            t=torch.as_tensor(obj[k]).float()
            xs.append(t if t.ndim==2 else t.view(-1,1))
    if xs:
        d=max(x.size(1) for x in xs)
        xs=[F.pad(x,(0, d-x.size(1))) for x in xs]
        return torch.cat(xs, dim=1)
    # node list
    if isinstance(obj, dict) and "nodes" in obj and isinstance(obj["nodes"], list) and obj["nodes"]:
        rows=[]
        for nd in obj["nodes"]:
            if not isinstance(nd, dict): continue
            for k in ["x","feat","emb","features","gcbert","gcb_x"]:
                if k in nd:
                    rows.append(torch.as_tensor(nd[k]).float().view(1,-1)); break
        if rows:
            d=max(r.size(1) for r in rows)
            rows=[F.pad(r,(0,d-r.size(1))) for r in rows]
            return torch.cat(rows, dim=0)
    return None

def build_edges_any(obj, N):
    E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
    if isinstance(obj, dict) and "edges" in obj and isinstance(obj["edges"], dict):
        for r in RELATIONS:
            if r in obj["edges"]:
                E[r]=sanitize_edges(N, to_long_2(obj["edges"][r]))
        if ADD_SUMMARY_EDGES and "DFG_THIN" in obj["edges"]:
            E["DFG_THIN"]=sanitize_edges(N, to_long_2(obj["edges"]["DFG_THIN"]))
        return E
    if isinstance(obj, dict):
        for r in RELATIONS:
            for k in [r, f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
                if k in obj:
                    E[r]=sanitize_edges(N, to_long_2(obj[k])); break
    if ADD_SUMMARY_EDGES:
        base=E["DFG"]
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base
    return E

def normalize_hetero(obj):
    node_stores=getattr(obj,"node_stores",None)
    edge_stores=getattr(obj,"edge_stores",None)
    if node_stores is None or edge_stores is None: return None
    cands=[]
    for st in node_stores:
        xv=getattr(st,"x",None)
        if xv is None: continue
        xt=torch.as_tensor(xv).float()
        if xt.ndim==1: xt=xt.view(-1,1)
        key=getattr(st,"_key", None) or getattr(st,"type", None) or "code"
        cands.append((key, xt))
    if not cands: return None
    nt,x=max(cands, key=lambda kv: kv[1].size(0))
    N=x.size(0)
    E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
    for es in edge_stores:
        ei=getattr(es,"edge_index",None)
        if ei is None: continue
        rel=getattr(es,"edge_type",None) or getattr(es,"_key",None)
        src_t=getattr(es,"src_type",None); dst_t=getattr(es,"dst_type",None)
        rel=str(rel).upper().split("__")[-1] if rel is not None else None
        if rel in E and (src_t is None or dst_t is None or (src_t==nt and dst_t==nt)):
            E[rel]=sanitize_edges(N, to_long_2(ei))
    if ADD_SUMMARY_EDGES:
        base=E["DFG"]
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base
    y=None
    for st in node_stores:
        st_key=getattr(st,"_key", None) or getattr(st,"type", None)
        if st_key==nt:
            yy=getattr(st,"y",None)
            if yy is not None:
                yy=torch.as_tensor(yy).float().view(-1)
                if yy.numel()==N: y=yy
            break
    return {"x":x, "edges":E, "y":y, "source":"default"}

def normalize_graph(obj):
    if isinstance(obj, dict) and "graph" in obj and isinstance(obj["graph"], dict):
        obj=obj["graph"]
    # dict
    if isinstance(obj, dict):
        x=coerce_x_any(obj)
        if x is not None:
            N=x.size(0)
            E=build_edges_any(obj, N)
            y=None
            for k in ["y","label","labels","vulnerable","is_sink","target","targets"]:
                if k in obj:
                    try:
                        yy=torch.as_tensor(obj[k]).float().view(-1)
                        if yy.numel()==N: y=yy
                    except: pass
                    break
            src = obj.get("source") or obj.get("dataset") or obj.get("origin") or "default"
            g={"x":x, "edges":E, "y":y, "source":src}
            if any(k in obj for k in ["paths_idx","vulnerable_paths","sinks","sink_nodes"]):
                g["aux"]={k:obj.get(k) for k in ["paths_idx","vulnerable_paths","sinks","sink_nodes"]}
            return g
    # torch_geometric
    if hasattr(obj,"x") and "torch_geometric" in str(type(obj)):
        g_het=normalize_hetero(obj)
        if g_het is not None: return g_het
        x=torch.as_tensor(getattr(obj,"x")).float()
        if x.ndim==1: x=x.view(-1,1)
        N=x.size(0)
        E={}
        for r in RELATIONS:
            found=False
            for nm in [f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
                if hasattr(obj, nm) and getattr(obj,nm) is not None:
                    E[r]=sanitize_edges(N, to_long_2(getattr(obj,nm))); found=True; break
            if not found:
                if hasattr(obj,"edge_index") and r=="DFG":
                    E[r]=sanitize_edges(N, to_long_2(getattr(obj,"edge_index")))
                else:
                    E[r]=torch.zeros((2,0),dtype=torch.long)
        if ADD_SUMMARY_EDGES:
            base=E["DFG"]
            for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
                if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
            E["DFG_THIN"]=base
        y=None
        if hasattr(obj,"y") and obj.y is not None and len(obj.y)==N:
            y=torch.as_tensor(obj.y).float().view(-1)
        return {"x":x, "edges":E, "y":y, "source":"default"}
    # last chance hetero
    if "torch_geometric" in str(type(obj)):
        g_het=normalize_hetero(obj)
        if g_het is not None: return g_het
    raise RuntimeError("Unsupported graph object type")

class GraphDir:
    def __init__(self, root):
        root=Path(root)
        self.paths = sorted([str(p) for p in root.glob("*.json")] + [str(p) for p in root.glob("*.pt")])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        obj=safe_load(self.paths[i])
        g=normalize_graph(obj)
        g["path"]=self.paths[i]
        return g

# -------------- Model --------------
class GraphBlock(nn.Module):
    def __init__(self, hidden, relations):
        super().__init__()
        self.relations=relations
        self.lin_rel=nn.ModuleDict({r:nn.Linear(hidden,hidden,bias=False) for r in relations})
        self.lin_self=nn.Linear(hidden,hidden)
    def forward(self, h, E):
        H=h
        for r in self.relations:
            ei=E.get(r)
            if ei is None or ei.numel()==0: continue
            s,d=ei
            msg=self.lin_rel[r](H)
            agg=torch.zeros_like(H)
            agg.index_add_(0, d, msg[s])
            H=H+agg
        return self.lin_self(H)

class CausalVulNet(nn.Module):
    def __init__(self, hidden, layers, relations):
        super().__init__()
        self.relations = relations + (["DFG_THIN"] if ADD_SUMMARY_EDGES else [])
        self.proj_cache = nn.ModuleDict()
        self.blocks = nn.ModuleList([GraphBlock(hidden, self.relations) for _ in range(layers)])
        self.node_head = nn.Linear(hidden,1)
        self.seed_head = nn.Linear(hidden,1)
        self.rel_gate  = nn.ParameterDict({r: nn.Parameter(torch.tensor(0.0)) for r in self.relations})
        self.edge_bilin= nn.Parameter(torch.empty(hidden, hidden)); nn.init.xavier_uniform_(self.edge_bilin)
    def _proj(self, D:int):
        k=str(D)
        if k not in self.proj_cache:
            layer=nn.Linear(D, HIDDEN).to(next(self.parameters()).device)
            self.proj_cache[k]=layer
        return self.proj_cache[k]
    def encode(self, x, E):
        h=F.relu(self._proj(x.size(1))(x))
        for blk in self.blocks: h=F.elu(blk(h,E))
        return h
    def edge_scores(self, h, E):
        out={}
        for r,ei in E.items():
            if ei is None or ei.numel()==0:
                out[r]=torch.zeros((0,), device=h.device); continue
            s,d=ei
            hs=h[s] @ self.edge_bilin
            out[r]=(hs*h[d]).sum(dim=1) + self.rel_gate[r]
        return out
    def forward_full(self, x, E):
        seed_h=F.relu(self._proj(x.size(1))(x))
        seed_logit=self.seed_head(seed_h).squeeze(-1)
        h=self.encode(x,E)
        node_logit=self.node_head(h).squeeze(-1)
        edge_sc=self.edge_scores(h,E)
        return seed_logit, node_logit, h, edge_sc

# -------------- Beam & slicing --------------
@dataclass
class BeamPath:
    score: float
    nodes: List[int]

def build_adj(E):
    adj_out={r:{} for r in E}; adj_in={r:{} for r in E}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei; ss,dd=s.tolist(),d.tolist()
        for u,v in zip(ss,dd):
            adj_out[r].setdefault(u,[]).append(v)
            adj_in [r].setdefault(v,[]).append(u)
    return adj_out, adj_in

def pick_seeds(seed_logit, E, k):
    N=seed_logit.numel()
    deg=torch.zeros(N, device=seed_logit.device)
    for ei in E.values():
        if ei is None or ei.numel()==0: continue
        s,_=ei; deg.index_add_(0, s, torch.ones_like(s, dtype=deg.dtype))
    cand=torch.where(deg>0)[0]
    if cand.numel()==0: return torch.topk(seed_logit, k=min(k,N)).indices.tolist()
    k=min(k, cand.numel()); vals=seed_logit[cand]
    return cand[torch.topk(vals,k=k).indices].tolist()

def run_beam(p, edge_sc, E, seeds, width=24, max_hops=5, alpha_node=0.7):
    N=p.numel()
    adj_out, adj_in = build_adj(E)
    uv={}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: uv[r]={}; continue
        s,d=ei; es=edge_sc[r].detach().float()
        mp={}
        for i in range(s.numel()):
            u=int(s[i]); v=int(d[i])
            if 0<=u<N and 0<=v<N:
                val=float(es[i].item())
                if (u,v) in mp: mp[(u,v)]=max(mp[(u,v)],val)
                else: mp[(u,v)]=val
        uv[r]=mp
    def clog(x): return float(torch.log(x.clamp(1e-9,1-1e-9)))
    beams=[BeamPath(clog(p[s]), [int(s)]) for s in seeds if 0<=int(s)<N]
    if not beams: return []
    out=[]
    for _ in range(max_hops):
        nxt=[]
        for b in beams:
            u=b.nodes[-1]
            cand=[]
            for r in E.keys():
                for v in adj_out[r].get(u, []): cand.append((r,u,v))
                for v in adj_in [r].get(u, []): cand.append((r,v,u))
            if not cand: out.append(b); continue
            for (r,uu,vv) in cand:
                if not (0<=vv<N): continue
                es=uv.get(r,{}).get((uu,vv), 0.0)
                sc=b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es
                nxt.append(BeamPath(sc, b.nodes+[vv]))
        if not nxt: break
        nxt.sort(key=lambda x:x.score, reverse=True)
        beams=nxt[:width]
    out.extend(beams); out.sort(key=lambda x:x.score, reverse=True)
    return out[:width]

def slice_from_paths(g, paths):
    if not paths:
        return {"x": g["x"][:1], "edges":{r:torch.zeros((2,0),dtype=torch.long) for r in g["edges"]}, "orig_idx":[0]}
    idx = sorted(set(n for bp in paths for n in bp.nodes if 0<=n<g["x"].size(0)))
    if not idx: idx=[0]
    idmap={old:i for i,old in enumerate(idx)}
    x=g["x"][idx]
    E={}
    keep=torch.tensor(idx)
    for r,ei in g["edges"].items():
        if ei is None or ei.numel()==0: E[r]=torch.zeros((2,0),dtype=torch.long); continue
        s,d=ei
        m=torch.isin(s,keep)&torch.isin(d,keep)
        if m.any():
            s2=torch.tensor([idmap[int(v)] for v in s[m].tolist()], dtype=torch.long)
            d2=torch.tensor([idmap[int(v)] for v in d[m].tolist()], dtype=torch.long)
            E[r]=torch.stack([s2,d2], dim=0)
        else:
            E[r]=torch.zeros((2,0),dtype=torch.long)
    return {"x":x, "edges":E, "orig_idx":idx}

def remap_paths_to_slice(paths, orig_idx):
    idmap={old:i for i,old in enumerate(orig_idx)}
    out=[]
    for bp in paths:
        ns=[]
        for n in bp.nodes:
            if n in idmap: ns.append(idmap[n])
            else: ns=[]; break
        if len(ns)>=2: out.append(BeamPath(bp.score, ns))
    return out

# -------------- Losses --------------
def focal_bce_with_logits(logit, target, alpha_pos=0.5, gamma=2.0):
    ce=F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    p=torch.sigmoid(logit)
    pt=p*target + (1-p)*(1-target)
    w=(alpha_pos*target + (1-alpha_pos)*(1-target)) * ((1-pt).pow(gamma))
    return (w*ce).mean()

def class_weights(y):
    pos=max(float((y>0.5).sum().item()), 1.0)
    neg=max(float((y<=0.5).sum().item()), 1.0)
    return neg/(pos+neg)

def hard_negative_mask(p,y,max_ratio=20):
    pos=(y>0.5).nonzero(as_tuple=False).view(-1)
    neg=(y<=0.5).nonzero(as_tuple=False).view(-1)
    if neg.numel()==0: return torch.ones_like(y, dtype=torch.bool)
    k=min(neg.numel(), int(max_ratio*max(1,pos.numel())))
    if k<=0: k=min(10,neg.numel())
    topk=torch.topk(p[neg], k=k).indices
    keep_neg=neg[topk]
    mask=torch.zeros_like(y, dtype=torch.bool)
    mask[pos]=True; mask[keep_neg]=True
    return mask

def loss_edge_participation(edge_sc, E, paths):
    # encourage edges that are used in top beam paths
    any_vec = next(iter(edge_sc.values()), torch.tensor([], device='cpu'))
    dev = any_vec.device if hasattr(any_vec,'device') else torch.device('cpu')
    if not paths: return torch.tensor(0.0, device=dev)
    L=torch.tensor(0.0, device=dev); n=0
    pos=set()
    for bp in paths[:3]:
        for u,v in zip(bp.nodes[:-1], bp.nodes[1:]): pos.add((u,v))
    for r,ei in E.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei; es=edge_sc[r]
        if es.numel()==0: continue
        pm = torch.tensor([ (int(s[i]),int(d[i])) in pos for i in range(s.numel()) ], device=es.device)
        if pm.any():
            L = L + F.binary_cross_entropy_with_logits(es[pm], torch.ones_like(es[pm])); n+=1
        nm = ~pm
        if nm.any():
            idx=torch.nonzero(nm, as_tuple=False).view(-1)
            if idx.numel()>0:
                sel=idx[torch.randperm(idx.numel(), device=idx.device)[:max(1, pm.sum().item())]]
                L = L + F.binary_cross_entropy_with_logits(es[sel], torch.zeros_like(es[sel])); n+=1
    return L/(n or 1)

def loss_monotonicity(p, paths, margin=0.05):
    if not paths: return torch.tensor(0.0, device=p.device)
    L=torch.tensor(0.0, device=p.device); n=0
    for bp in paths[:5]:
        ns=[n for n in bp.nodes if 0<=n<p.numel()]
        for i in range(len(ns)-1):
            L = L + F.relu(p[ns[i]] - p[ns[i+1]] + margin); n+=1
    return L/(n or 1)

def loss_path_ranking(p, paths):
    if not paths: return torch.tensor(0.0, device=p.device)
    def ps(bp):
        idx=[n for n in bp.nodes if 0<=n<p.numel()]
        if len(idx)<2: return None
        idx=torch.as_tensor(idx, device=p.device, dtype=torch.long)
        return torch.log(p[idx].clamp(1e-9,1-1e-9)).sum()
    pos=[t for t in (ps(bp) for bp in paths[:3]) if t is not None]
    if not pos: return torch.tensor(0.0, device=p.device)
    pos=torch.stack(pos)
    rnd=[]
    N=p.numel()
    for _ in range(len(pos)):
        L=max(2, min(N, 6))
        idx=torch.randperm(N, device=p.device)[:L]
        rnd.append(torch.log(p[idx].clamp(1e-9,1-1e-9)).sum())
    rnd=torch.stack(rnd)
    return F.relu(0.1 + rnd - pos).mean()

# -------------- Metrics / Attribution --------------
@torch.no_grad()
def f1_from_probs(p, y, thr=0.5):
    if y is None or y.numel()!=p.numel(): return None
    yb=(y>0.5); pb=(p>thr)
    tp=(pb&yb).sum().item(); fp=(pb&~yb).sum().item(); fn=(~pb&yb).sum().item()
    prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec,"recall":rec,"f1":f1}

def compute_CFAM(model, g, paths):
    if not paths: return None
    x=g["x"].to(DEVICE).detach().requires_grad_(True)
    with torch.enable_grad():
        _,nl,_,_ = model.forward_full(x, {r:e.to(DEVICE) for r,e in g["edges"].items()})
        s=torch.sigmoid(nl).mean()
        s.backward()
        gn = x.grad.detach().abs().sum(dim=1)
    causal=set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0))
    if not causal: return None
    mask=torch.zeros(x.size(0), dtype=torch.bool, device=gn.device)
    mask[torch.tensor(list(causal), device=gn.device)] = True
    num=gn[mask].sum().item(); den=gn.sum().item()+1e-9
    return num/den

def compute_CCS(model, g, paths):
    if not paths: return None
    x=g["x"].to(DEVICE)
    with torch.no_grad():
        _,nl,_,_=model.forward_full(x, {r:e.to(DEVICE) for r,e in g["edges"].items()})
        p0=torch.sigmoid(nl).mean().item()
    cf=x.clone()
    causal=sorted(set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0)))
    if causal: cf[torch.tensor(causal, device=cf.device)] = 0.0
    with torch.no_grad():
        _,nl2,_,_=model.forward_full(cf, {r:e.to(DEVICE) for r,e in g["edges"].items()})
        p1=torch.sigmoid(nl2).mean().item()
    return (p0-p1)**2

def to_device_graph(g):
    return {"x":g["x"].to(DEVICE),
            "edges":{r:e.to(DEVICE) for r,e in g["edges"].items()},
            "y": (g.get("y").to(DEVICE) if g.get("y") is not None else None),
            "source": g.get("source","default")}

# -------------- Calibration --------------
@torch.no_grad()
def calibrate_thresholds(model, ds):
    per_source={}
    for i in range(min(EVAL_MAX or 10**9, len(ds))):
        g_cpu=ds[i]; g=to_device_graph(g_cpu)
        sd,nl,_,_=model.forward_full(g["x"], g["edges"])
        p=torch.sigmoid(nl).detach().cpu()
        y=g_cpu.get("y")
        if y is None or y.numel()!=p.numel(): continue
        src=g_cpu.get("source","default")
        per_source.setdefault(src, {"p":[], "y": []})
        per_source[src]["p"].append(p); per_source[src]["y"].append(y)
    out={}
    for src,buf in per_source.items():
        P=torch.cat(buf["p"]); Y=torch.cat([torch.as_tensor(x).float().view(-1) for x in buf["y"]])
        best=(0.5,0.0)
        for thr in [i/100 for i in range(5,96,5)]:
            m=f1_from_probs(P, Y, thr)
            f=m["f1"] if m else 0.0
            if f>best[1]: best=(thr,f)
        out[src]=best[0]
    if not out: out={"default": THRESHOLDS_BY_SOURCE.get("default",0.25)}
    return out

# -------------- Trainer --------------
class CausalTrainer:
    def __init__(self):
        self.model = CausalVulNet(hidden=HIDDEN, layers=LAYERS, relations=RELATIONS).to(DEVICE)
        self.opt = torch.optim.AdamW(self.model.parameters(), lr=LR, weight_decay=WD)
        self.scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and DEVICE=='cuda'))
    def train_and_eval(self):
        ds_tr, ds_v, ds_te = GraphDir(DATA_DIRS["train"]), GraphDir(DATA_DIRS["valid"]), GraphDir(DATA_DIRS["test"])
        print(f"[DATA] {len(ds_tr)} graphs in {DATA_DIRS['train']}")
        print(f"[DATA] {len(ds_v)} graphs in {DATA_DIRS['valid']}")
        print(f"[DATA] {len(ds_te)} graphs in {DATA_DIRS['test']}")
        (OUT_ROOT/"cfg.json").write_text(json.dumps({
            "epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WD,
            "neg_pos_ratio":NEG_POS_RATIO,"device":DEVICE,"relations":RELATIONS,
            "add_summary_edges":ADD_SUMMARY_EDGES,
            "beam":{"k":SEED_K,"width":BEAM_WIDTH,"max_hops":BEAM_MAX_HOPS,"alpha_node":ALPHA_NODE}
        }, indent=2))
        step=0
        for ep in range(1, EPOCHS+1):
            self.model.train()
            running={"node":0.0,"edge":0.0,"mono":0.0,"rank":0.0}
            skipped=0
            for i in tqdm(range(len(ds_tr)), desc=f"[train] epoch {ep}"):
                step+=1
                g_cpu = ds_tr[i]; g_dev = to_device_graph(g_cpu)
                with (torch.amp.autocast('cuda', enabled=(USE_AMP and DEVICE=='cuda')) if DEVICE=='cuda' else torch.autocast("cpu", enabled=False)):
                    # seed/beam on full graph (cheap seed head) → beam-only slice; NO heuristics
                    s_full, n_full, h_full, e_full = self.model.forward_full(g_dev["x"], g_dev["edges"])
                    seeds = pick_seeds(s_full.detach(), g_dev["edges"], SEED_K)
                    p_full = torch.sigmoid(n_full)
                    beam = run_beam(p_full.detach(), e_full, g_dev["edges"], seeds, BEAM_WIDTH, BEAM_MAX_HOPS, ALPHA_NODE)

                    if len(beam)==0:
                        skipped+=1
                        continue  # nothing to learn from

                    g_slice_cpu = slice_from_paths({"x":g_cpu["x"], "edges":g_cpu["edges"]}, beam)
                    g_slice = {"x": g_slice_cpu["x"].to(DEVICE),
                               "edges": {r:e.to(DEVICE) for r,e in g_slice_cpu["edges"].items()},
                               "orig_idx": g_slice_cpu["orig_idx"]}
                    beam_slice = remap_paths_to_slice(beam, g_slice["orig_idx"])
                    if len(beam_slice)==0:
                        skipped+=1
                        continue

                    s_s, n_s, h_s, e_s = self.model.forward_full(g_slice["x"], g_slice["edges"])
                    p_s = torch.sigmoid(n_s)

                    # Node loss if labels exist; else 0
                    if g_dev["y"] is not None and g_dev["y"].numel()==p_full.numel():
                        alpha_pos = class_weights(g_dev["y"])
                        mask = hard_negative_mask(p_full.detach(), g_dev["y"], NEG_POS_RATIO)
                        l_node = focal_bce_with_logits(n_full[mask], g_dev["y"][mask], alpha_pos=alpha_pos, gamma=2.0)
                        ds_w = DATASET_WEIGHTS.get(g_cpu.get("source","default"), DATASET_WEIGHTS["default"])
                        l_node = l_node * float(ds_w)
                    else:
                        l_node = torch.tensor(0.0, device=DEVICE)

                    l_edge = loss_edge_participation(e_s, g_slice["edges"], beam_slice)
                    l_mono = loss_monotonicity(p_s, beam_slice, 0.05)
                    l_rank = loss_path_ranking(p_s, beam_slice)

                    loss = l_node + 0.20*l_edge + 0.25*l_mono + 0.20*l_rank

                # if everything was constant (rare), skip backward
                if not loss.requires_grad:
                    skipped+=1
                    continue

                self.opt.zero_grad(set_to_none=True)
                if DEVICE=='cuda' and USE_AMP:
                    self.scaler.scale(loss).backward()
                    nn.utils.clip_grad_norm_(self.model.parameters(), 2.0)
                    self.scaler.step(self.opt); self.scaler.update()
                else:
                    loss.backward(); nn.utils.clip_grad_norm_(self.model.parameters(), 2.0); self.opt.step()

                running["node"]+=float(l_node.detach().item())
                running["edge"]+=float(l_edge.detach().item())
                running["mono"]+=float(l_mono.detach().item())
                running["rank"]+=float(l_rank.detach().item())

                # Early causal probes every ~1200 steps (inter-proc %, beam length, CFAM/CCS snapshot)
                if step % 1200 == 0 and len(ds_v)>0:
                    self.model.eval()
                    with torch.no_grad():
                        blen=[]; inter=[]; cfam=[]; ccs=[]
                        for j in range(min(3,len(ds_v))):
                            gv_cpu=ds_v[j]; gv=to_device_graph(gv_cpu)
                            sd,nl,hv,es = self.model.forward_full(gv["x"], gv["edges"])
                            seeds=pick_seeds(sd.detach(), gv["edges"], min(SEED_K,4))
                            pv=torch.sigmoid(nl)
                            b=run_beam(pv, es, gv["edges"], seeds, BEAM_WIDTH, BEAM_MAX_HOPS, ALPHA_NODE)
                            if b:
                                blen.append(sum(len(bp.nodes) for bp in b)/(len(b) or 1))
                                inter_edges={"CALL","ARG2PARAM","RET2CALL","RET2LHS"}
                                out_steps=0; inter_steps=0
                                adj,_=build_adj(gv["edges"])
                                for bp in b:
                                    for u,v in zip(bp.nodes[:-1], bp.nodes[1:]):
                                        out_steps+=1
                                        if any(v in adj.get(r,{}).get(u,[]) for r in inter_edges):
                                            inter_steps+=1
                                inter.append(inter_steps/(out_steps or 1))
                            cf = compute_CFAM(self.model, gv_cpu, b); cc = compute_CCS(self.model, gv_cpu, b)
                            if cf is not None: cfam.append(cf)
                            if cc is not None: ccs.append(cc)
                    print(f"[early] ep{ep} step{step} | beam.len={ (sum(blen)/len(blen) if blen else float('nan')):.2f} "
                          f"inter%={ 100*(sum(inter)/len(inter) if inter else 0):.1f} "
                          f"| CFAM~{(sum(cfam)/len(cfam) if cfam else float('nan')):.3f} "
                          f"CCS~{(sum(ccs)/len(ccs) if ccs else float('nan')):.3f} | VRAM={vram_info()}")
                    self.model.train()

            avg={k:v/max(1,(len(ds_tr)-skipped)) for k,v in running.items()}
            print(f"[epoch {ep}] loss={sum(avg.values()):.4f} (node={avg['node']:.4f}, edge={avg['edge']:.4f}, mono={avg['mono']:.4f}, rank={avg['rank']:.4f}) | skipped={skipped}")

        # Save model
        torch.save({"state_dict":self.model.state_dict(),
                    "hidden":HIDDEN,"layers":LAYERS,
                    "relations":RELATIONS + (["DFG_THIN"] if ADD_SUMMARY_EDGES else [])}, OUT_ROOT/"model.pt")

        # Calibrate thresholds on valid (if labels exist)
        if len(ds_v)>0: 
            thrs = calibrate_thresholds(self.model, ds_v)
        else:
            thrs = THRESHOLDS_BY_SOURCE
        (OUT_ROOT/"thresholds.json").write_text(json.dumps(thrs, indent=2))

        # Evaluate + Demo
        self.model.eval()
        reports={}
        for SPLIT, DS in [("train", GraphDir(DATA_DIRS["train"])),
                          ("valid", GraphDir(DATA_DIRS["valid"])),
                          ("test",  GraphDir(DATA_DIRS["test"]))]:
            n_used=0; s_prec=0.0; s_rec=0.0; s_f1=0.0; ccs_vals=[]; cfam_vals=[]
            U = min(EVAL_MAX or 10**9, len(DS))
            for i in tqdm(range(U), desc=f"[eval:{SPLIT}]"):
                g_cpu=DS[i]; g=to_device_graph(g_cpu)
                with torch.no_grad():
                    sd,nl,h,es = self.model.forward_full(g["x"], g["edges"])
                    seeds=pick_seeds(sd.detach(), g["edges"], SEED_K)
                    p=torch.sigmoid(nl)
                    b=run_beam(p, es, g["edges"], seeds, BEAM_WIDTH, BEAM_MAX_HOPS, ALPHA_NODE)
                m=f1_from_probs(p.detach().cpu(), g_cpu.get("y"), thrs.get(g_cpu.get("source","default"), thrs.get("default", 0.25)))
                if m:
                    s_prec+=m["precision"]; s_rec+=m["recall"]; s_f1+=m["f1"]
                cf=compute_CFAM(self.model, g_cpu, b); cc=compute_CCS(self.model, g_cpu, b)
                if cf is not None: cfam_vals.append(cf)
                if cc is not None: ccs_vals.append(cc)
                n_used+=1
            rep={
                "split":SPLIT,
                "overall":{"precision":(s_prec/n_used if n_used and s_prec>0 else None),
                           "recall":   (s_rec/n_used if n_used and s_rec>0 else None),
                           "f1":       (s_f1/n_used if n_used and s_f1>0 else 0.0)},
                "thresholds": thrs,
                "n_graphs_used": n_used,
                "CCS_mean": (sum(ccs_vals)/len(ccs_vals) if ccs_vals else None),
                "CFAM_mean":(sum(cfam_vals)/len(cfam_vals) if cfam_vals else None)
            }
            reports[SPLIT]=rep

        demo={"meta":{"split":None,"graph_path":None},"paths":[]}
        if len(GraphDir(DATA_DIRS["valid"]))>0:
            g_cpu=GraphDir(DATA_DIRS["valid"])[0]; g=to_device_graph(g_cpu)
            with torch.no_grad():
                sd,nl,h,es = self.model.forward_full(g["x"], g["edges"])
                seeds=pick_seeds(sd.detach(), g["edges"], SEED_K)
                p=torch.sigmoid(nl)
                b=run_beam(p, es, g["edges"], seeds, BEAM_WIDTH, BEAM_MAX_HOPS, ALPHA_NODE)
            demo["meta"]={"split":"valid","graph_path":g_cpu.get("path")}
            demo["paths"]=[{"score":bp.score, "nodes":bp.nodes} for bp in b[:10]]

        (OUT_ROOT/"reports.json").write_text(json.dumps({
            "config":{"epochs":EPOCHS,"hidden":HIDDEN,"layers":LAYERS,"lr":LR,"wd":WD,
                      "neg_pos_ratio":NEG_POS_RATIO,"device":DEVICE,"relations":RELATIONS,
                      "add_summary_edges":ADD_SUMMARY_EDGES,"slice_mode":"beam",
                      "beam":{"k":SEED_K,"width":BEAM_WIDTH,"max_hops":BEAM_MAX_HOPS,"alpha_node":ALPHA_NODE}},
            "thresholds_by_source":thrs,
            "reports":reports,
            "notes":{
                "interprocedural":"CALL/ARG2PARAM/RET2CALL/RET2LHS (+DFG_THIN) participate in beam transitions.",
                "beam_guidance":"learned relation gates + bilinear edge scorer; no fixed ordering.",
                "objective":"focal(class-weights + hard-negatives) + edge participation + monotonicity + path ranking.",
                "CCS":"p(x) vs p(do(x')) by zeroing beam slice features.",
                "CFAM":"grad-norm attribution mass on beam slice / total.",
                "multi_root":"top-K seed nodes → multiple chains.",
                "saving": str(OUT_ROOT)
            }
        }, indent=2))
        print(f"[saved] artifacts in: {OUT_ROOT}")

# -------------- Run --------------
try:
    CausalTrainer().train_and_eval()
except Exception as e:
    print("!! Exception:", e, "| VRAM:", vram_info())
    raise


[env] torch=2.4.1+cu121 device=cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
[DATA] 3438 graphs in Dataset/train/hetero_ready_gcbert
[DATA] 2906 graphs in Dataset/valid/hetero_ready_gcbert
[DATA] 2915 graphs in Dataset/test/hetero_ready_gcbert


[train] epoch 1:   0%|          | 0/3438 [00:00<?, ?it/s]C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_3068\581379378.py:76: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  r

[epoch 1] loss=0.0000 (node=0.0000, edge=0.0000, mono=0.0000, rank=0.0000) | skipped=3438


[train] epoch 2: 100%|██████████| 3438/3438 [02:10<00:00, 26.38it/s]


[epoch 2] loss=0.0000 (node=0.0000, edge=0.0000, mono=0.0000, rank=0.0000) | skipped=3438


...

[epoch 4] loss=0.0000 (node=0.0000, edge=0.0000, mono=0.0000, rank=0.0000) | skipped=3438


[train] epoch 5: 100%|██████████| 3438/3438 [02:10<00:00, 26.29it/s]


[epoch 5] loss=0.0000 (node=0.0000, edge=0.0000, mono=0.0000, rank=0.0000) | skipped=3438


[eval:test]: 100%|██████████| 256/256 [00:07<00:00, 34.59it/s]


[saved] artifacts in: out\cvul\1760989836


CKG Mining

In [2]:
# ===================== FAST CKG mining (pruned beam) + UTF-8 report + checkpoints =====================
import os, json, math, time
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# ---------- CONFIG ----------
CKPT       = r"out/cvul/1760989836/model.pt"

DATA_MINE  = r"Dataset/train/hetero_ready_gcbert"
DATA_DEMO  = r"Dataset/demo/hetero_ready_gcbert"
FALLBACK_1 = r"Dataset/valid/hetero_ready_gcbert"
FALLBACK_2 = r"Dataset/train/hetero_ready_gcbert"

CKG_PATH       = r"ckg_light/ckg.json"
CKG_PATH_TMP   = r"ckg_light/ckg.json.tmp"
CKG_RPT_JSON   = r"ckg_light/ckg_report.json"
CKG_RPT_MD     = r"ckg_light/ckg_report.md"

# Mining beam (smaller than training for speed)
SEED_K_MINE     = 3
BEAM_WIDTH_MINE = 8
BEAM_HOPS_MINE  = 3
ALPHA_NODE      = 0.7
TOP_PATHS       = 1     # use top-1 chain per graph for counts

# Pruning
RELS_FOR_MINING = ["DFG_THIN","CALL","ARG2PARAM","RET2CALL","RET2LHS"]  # skip CFG for speed
TOP_M_PER_REL   = 4      # keep only top-M neighbors per relation (by edge score)
USE_IN_EDGES    = False  # outgoing only for mining

# Budget / checkpoints
MINE_LIMIT    = None     # None = all files
CKPT_EVERY    = 200

# Optional compile/autocast
USE_TORCH_COMPILE = True
USE_AUTOMIXED_FP16= True

# ---------- device ----------
def pick_device(min_free_mb=256):
    if not torch.cuda.is_available(): return "cpu"
    try:
        free,_ = torch.cuda.mem_get_info()
        return "cuda" if (free//(1024**2)) >= min_free_mb else "cpu"
    except:
        return "cuda"
DEVICE = pick_device()
torch.set_grad_enabled(False)
print(f"[env] torch={torch.__version__} device={DEVICE}")

# ---------- IO ----------
def safe_load(p):
    p = Path(p)
    if p.suffix.lower() == ".json":
        return json.loads(p.read_text(encoding="utf-8"))
    return torch.load(p, map_location="cpu")

def write_json_utf8(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def list_files(root_dir, patterns=("*.json","*.pt")):
    root = Path(root_dir)
    files=[]
    for pat in patterns:
        files.extend(sorted(root.glob(pat)))
    return files

# ---------- normalization helpers ----------
def _first_2d_float(arr):
    t = torch.as_tensor(arr).float()
    if t.ndim == 1: t = t.view(-1, 1)
    return t

def to_long_2(e):
    if e is None: return torch.zeros((2,0), dtype=torch.long)
    t=torch.as_tensor(e)
    if t.ndim==2 and t.shape[0]==2: return t.long().contiguous()
    if t.ndim==2 and t.shape[1]==2: return t.t().long().contiguous()
    if isinstance(e,(list,tuple)) and len(e)==2:
        s=torch.as_tensor(e[0]).view(-1).long()
        d=torch.as_tensor(e[1]).view(-1).long()
        return torch.stack([s,d], dim=0)
    if t.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    raise RuntimeError("edge_index must be [2,E], [E,2], or (src,dst)")

def sanitize_edges(N:int, ei:torch.Tensor):
    if ei is None or ei.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    s,d = ei
    m=(s>=0)&(s<N)&(d>=0)&(d<N)
    if m.any(): return torch.stack([s[m], d[m]], dim=0)
    return torch.zeros((2,0), dtype=torch.long)

def _get_store_feat(store):
    cand = ["gcbert","gcb_x","x","x_text","x_num","features","feat","emb"]
    for nm in cand:
        if hasattr(store, nm):
            val = getattr(store, nm)
            if val is not None:
                t = _first_2d_float(val)
                if t.numel() > 0: return t
    if hasattr(store, "__dict__"):
        for nm in cand:
            if nm in store.__dict__ and store.__dict__[nm] is not None:
                t = _first_2d_float(store.__dict__[nm])
                if t.numel() > 0: return t
    return None

def normalize_hetero(obj, RELATIONS, ADD_SUMMARY_EDGES=True):
    node_stores = getattr(obj, "node_stores", None)
    edge_stores = getattr(obj, "edge_stores", None)
    if node_stores is None or edge_stores is None: return None
    candidates=[]
    for st in node_stores:
        key = getattr(st, "_key", None) or getattr(st, "type", None) or "code"
        feat = _get_store_feat(st)
        if feat is not None and feat.numel() > 0:
            candidates.append((key, feat))
    if not candidates: return None
    nt, x = max(candidates, key=lambda kv: kv[1].size(0))
    N = x.size(0)
    E = {r: torch.zeros((2,0), dtype=torch.long) for r in RELATIONS}
    for es in edge_stores:
        ei = getattr(es, "edge_index", None)
        if ei is None: continue
        src_t = getattr(es, "src_type", None)
        dst_t = getattr(es, "dst_type", None)
        if (src_t is not None and dst_t is not None) and not (src_t == nt and dst_t == nt):
            continue
        rel = getattr(es, "edge_type", None) or getattr(es, "_key", None)
        if isinstance(rel, (tuple, list)) and len(rel) == 3:
            rel = str(rel[1])
        rel = str(rel).upper().split("__")[-1] if rel is not None else None
        if rel in E: E[rel] = sanitize_edges(N, to_long_2(ei))
    if ADD_SUMMARY_EDGES:
        base = E.get("DFG", torch.zeros((2,0), dtype=torch.long))
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if r in E and E[r].numel() > 0: base = torch.cat([base, E[r]], dim=1)
        E["DFG_THIN"] = base
    return {"x": x, "edges": E, "y": None, "source": "default"}

def coerce_x_any(obj):
    for k in ["gcbert","gcb_x","x","features","node_features","feat","emb","x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj, dict) and k in obj and obj[k] is not None:
            t=torch.as_tensor(obj[k]).float()
            if t.ndim==1: t=t.view(-1,1)
            return t
    if isinstance(obj, dict) and "nodes" in obj and isinstance(obj["nodes"], list) and obj["nodes"]:
        rows=[]
        for nd in obj["nodes"]:
            if not isinstance(nd, dict): continue
            for k in ["gcbert","gcb_x","x","feat","emb","features"]:
                if k in nd and nd[k] is not None:
                    rows.append(torch.as_tensor(nd[k]).float().view(1,-1)); break
        if rows:
            d=max(r.size(1) for r in rows)
            rows=[F.pad(r,(0,d-r.size(1))) for r in rows]
            return torch.cat(rows, dim=0)
    return None

def build_edges_any(obj, N, RELATIONS, ADD_SUMMARY_EDGES=True):
    E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
    if isinstance(obj, dict) and "edges" in obj and isinstance(obj["edges"], dict):
        for r in RELATIONS:
            if r in obj["edges"]: E[r]=sanitize_edges(N, to_long_2(obj["edges"][r]))
        if ADD_SUMMARY_EDGES:
            base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
            for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
                if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
            E["DFG_THIN"]=base
        return E
    for r in RELATIONS:
        for k in [r, f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
            if isinstance(obj, dict) and k in obj:
                E[r]=sanitize_edges(N, to_long_2(obj[k])); break
    if ADD_SUMMARY_EDGES:
        base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base
    return E

def normalize_graph(obj, RELATIONS, ADD_SUMMARY_EDGES=True):
    if isinstance(obj, dict) and "graph" in obj and isinstance(obj["graph"], dict):
        obj = obj["graph"]
    if "torch_geometric" in str(type(obj)) and not isinstance(obj, dict):
        g_het = normalize_hetero(obj, RELATIONS, ADD_SUMMARY_EDGES)
        if g_het is not None: return g_het
        if hasattr(obj, "x") and obj.x is not None:
            x = _first_2d_float(obj.x); N=x.size(0)
            E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
            if hasattr(obj,"edge_index"): E["DFG"]=sanitize_edges(N, to_long_2(obj.edge_index))
            if ADD_SUMMARY_EDGES: E["DFG_THIN"]=E["DFG"]
            return {"x":x,"edges":E,"y":None,"source":"default"}
        return None
    if isinstance(obj, dict):
        x = coerce_x_any(obj)
        if x is None or x.numel()==0: return None
        N = x.size(0)
        E = build_edges_any(obj, N, RELATIONS, ADD_SUMMARY_EDGES)
        return {"x":x, "edges":E, "y":None, "source":"default"}
    return None

def iter_graphs(root_dir, RELATIONS, ADD_SUMMARY_EDGES=True, limit=None, patterns=("*.json","*.pt")):
    files = list_files(root_dir, patterns)
    if limit is None: limit = len(files)
    for fp in files[:limit]:
        try:
            obj = safe_load(fp)
            g = normalize_graph(obj, RELATIONS, ADD_SUMMARY_EDGES)
            if g is None or g["x"] is None or g["x"].numel()==0: continue
            g["path"] = str(fp)
            yield g
        except Exception:
            continue

# ---------- model ----------
class GraphBlock(nn.Module):
    def __init__(self, hidden, relations):
        super().__init__()
        self.relations = relations
        self.lin_rel = nn.ModuleDict({r: nn.Linear(hidden,hidden,bias=False) for r in relations})
        self.lin_self= nn.Linear(hidden,hidden)
    def forward(self, h, E):
        H = h
        for r in self.relations:
            ei = E.get(r)
            if ei is None or ei.numel()==0: continue
            s,d = ei
            msg = self.lin_rel[r](H)
            agg = torch.zeros_like(H)
            agg.index_add_(0, d, msg[s])
            H = H + agg
        return self.lin_self(H)

class CausalVulNet(nn.Module):
    def __init__(self, hidden=64, layers=3, relations=None):
        super().__init__()
        if relations is None:
            relations = ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS","DFG_THIN"]
        self.relations = list(relations)
        self.proj_cache = nn.ModuleDict()
        self.blocks = nn.ModuleList([GraphBlock(hidden, self.relations) for _ in range(layers)])
        self.node_head = nn.Linear(hidden,1)
        self.seed_head = nn.Linear(hidden,1)
        self.rel_gate  = nn.ParameterDict({r: nn.Parameter(torch.tensor(0.0)) for r in self.relations})
        self.edge_bilin= nn.Parameter(torch.empty(hidden, hidden)); nn.init.xavier_uniform_(self.edge_bilin)
        self.hidden    = hidden
        self.layers    = layers
    def _proj(self, D:int):
        k=str(D)
        if k not in self.proj_cache:
            layer=nn.Linear(D, self.hidden).to(next(self.parameters()).device)
            self.proj_cache[k]=layer
        return self.proj_cache[k]
    def encode(self, x, E):
        h=F.relu(self._proj(x.size(1))(x))
        for blk in self.blocks: h=F.elu(blk(h,E))
        return h
    def edge_scores(self, h, E):
        out={}
        for r,ei in E.items():
            if ei is None or ei.numel()==0:
                out[r]=torch.zeros((0,), device=h.device); continue
            s,d=ei
            hs=h[s] @ self.edge_bilin
            out[r]=(hs*h[d]).sum(dim=1) + self.rel_gate[r]
        return out
    def forward_full(self, x, E):
        seed_h=F.relu(self._proj(x.size(1))(x))
        seed_logit=self.seed_head(seed_h).squeeze(-1)
        h=self.encode(x,E)
        node_logit=self.node_head(h).squeeze(-1)
        edge_sc=self.edge_scores(h,E)
        return seed_logit, node_logit, h, edge_sc

def load_model(ckpt_path:str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    hidden    = ckpt.get("hidden", 64)
    layers    = ckpt.get("layers", 3)
    relations = ckpt.get("relations", ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS","DFG_THIN"])
    model = CausalVulNet(hidden=hidden, layers=layers, relations=relations).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"], strict=False)
    model.eval()
    if USE_TORCH_COMPILE and hasattr(torch, "compile"):
        try: model = torch.compile(model, mode="reduce-overhead", fullgraph=False)
        except Exception: pass
    base_relations = [r for r in relations if r != "DFG_THIN"]
    add_summary = ("DFG_THIN" in relations)
    return model, base_relations, add_summary

# ---------- pruned beam for MINING ----------
@dataclass
class BeamPath:
    score: float
    nodes: List[int]
    rels:  List[str]

def _edge_uv_scores(edge_sc, E, N:int, rel_filter):
    uv={r:{} for r in E if r in rel_filter}
    for r,ei in E.items():
        if r not in rel_filter: continue
        if ei is None or ei.numel()==0: continue
        s,d=ei; es=edge_sc[r].detach().float()
        for i in range(s.numel()):
            u=int(s[i]); v=int(d[i])
            if 0<=u<N and 0<=v<N:
                val=float(es[i].item())
                mp=uv[r]
                best = mp.get(u)
                if best is None:
                    mp[u] = [(v, val)]
                else:
                    best.append((v,val))
    # prune to top-M per (u,r)
    for r in uv:
        for u in list(uv[r].keys()):
            lst = uv[r][u]
            lst.sort(key=lambda t:t[1], reverse=True)
            uv[r][u] = lst[:TOP_M_PER_REL]
    return uv

def run_beam_pruned_for_mining(p, edge_sc, E, seeds, width=8, max_hops=3, alpha_node=0.7):
    N=p.numel()
    uv = _edge_uv_scores(edge_sc, E, N, set(RELS_FOR_MINING))
    def clog(x): return float(torch.log(x.clamp(1e-9,1-1e-9)))
    beams=[BeamPath(clog(p[s]), [int(s)], []) for s in seeds if 0<=int(s)<N]
    if not beams: return []
    out=[]
    for _ in range(max_hops):
        nxt=[]
        for b in beams:
            u=b.nodes[-1]
            cand=[]
            # outgoing only, pruned by top-M
            for r in RELS_FOR_MINING:
                for (v, es) in uv.get(r, {}).get(u, []):
                    cand.append((r, u, v, es))
            if not cand:
                out.append(b); continue
            for (r, uu, vv, es) in cand:
                if not (0<=vv<N): continue
                sc = b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es
                nxt.append(BeamPath(sc, b.nodes+[vv], b.rels+[r]))
        if not nxt: break
        nxt.sort(key=lambda x:x.score, reverse=True)
        beams = nxt[:width]
    out.extend(beams); out.sort(key=lambda x:x.score, reverse=True)
    return out[:width]

# ---------- helpers ----------
def to_device_graph(g):
    return {"x": g["x"].to(DEVICE),
            "edges": {r:e.to(DEVICE) for r,e in g["edges"].items()},
            "y": None, "source": g.get("source","default")}

# ---------- mining ----------
def mine_ckg_fast(model, data_dir, base_relations, add_summary=True,
                  limit=None, seed_k=3, width=8, hops=3, alpha=0.7):
    rels = list(set(RELS_FOR_MINING + (["DFG_THIN"] if add_summary else [])))
    edge_count   = {r:0 for r in rels}
    bigram_count = {r:{q:0 for q in rels} for r in rels}
    start_count  = {r:0 for r in rels}
    end_count    = {r:0 for r in rels}
    hop_hist     = Counter()
    trigram_count= Counter()
    used = 0

    files = list_files(data_dir, ("*.json","*.pt"))
    if limit is None: limit = len(files)
    pbar  = tqdm(files[:limit], total=min(limit, len(files)), desc=f"[fast-mine] {data_dir}")

    dtype_ctx = (torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AUTOMIXED_FP16 and DEVICE=="cuda")))
    with torch.inference_mode(), dtype_ctx:
        for fp in pbar:
            try:
                obj = safe_load(fp)
                g_cpu = normalize_graph(obj, base_relations, add_summary)
                if g_cpu is None or g_cpu["x"] is None or g_cpu["x"].numel()==0:
                    continue
                g = to_device_graph(g_cpu)
                sd, nl, h, es = model.forward_full(g["x"], g["edges"])
                seeds = torch.topk(sd, k=min(seed_k, sd.numel())).indices.tolist()
                p  = torch.sigmoid(nl)
                beams = run_beam_pruned_for_mining(p, es, g["edges"], seeds, width=width, max_hops=hops, alpha_node=alpha)
                if not beams: continue
                used += 1

                for bp in beams[:TOP_PATHS]:
                    hop_hist[len(bp.nodes)-1] += 1
                    if bp.rels:
                        start_count[bp.rels[0]] += 1
                        end_count  [bp.rels[-1]]+= 1
                    prev = None
                    for j, r in enumerate(bp.rels):
                        edge_count[r] += 1
                        if prev is not None:
                            bigram_count[prev][r] += 1
                        if j >= 2:
                            tr = (bp.rels[j-2], bp.rels[j-1], bp.rels[j])
                            trigram_count[tr] += 1
                        prev = r

                if used and (used % CKPT_EVERY == 0):
                    ckg_partial = build_ckg_obj(data_dir, rels, edge_count, bigram_count,
                                                start_count, end_count, hop_hist, trigram_count, used)
                    write_json_utf8(CKG_PATH_TMP, ckg_partial)
                    pbar.set_postfix_str(f"ckpt@{used}")
            except Exception:
                continue

    return build_ckg_obj(data_dir, rels, edge_count, bigram_count,
                         start_count, end_count, hop_hist, trigram_count, used)

def build_ckg_obj(data_dir, rels, edge_count, bigram_count, start_count, end_count, hop_hist, trigram_count, used):
    total_edges = sum(edge_count.values()) or 1
    edge_prior_prob = {r: edge_count[r]/total_edges for r in edge_count}
    bigram_prob = {}
    for r in bigram_count:
        s = sum(bigram_count[r].values()) or 1
        bigram_prob[r] = {q: bigram_count[r][q]/s for q in bigram_count[r]}

    motifs_topk = []
    if trigram_count:
        for (a,b,c), cnt in trigram_count.most_common(20):
            motifs_topk.append({"count": int(cnt), "rels": [a,b,c]})
    if not motifs_topk:
        flat = [((a,b), cnt) for a,row in bigram_count.items() for b,cnt in row.items() if cnt>0]
        flat.sort(key=lambda x:x[1], reverse=True)
        for (a,b), cnt_ab in flat[:20]:
            row_b = bigram_count.get(b, {})
            if row_b:
                c = max(row_b, key=lambda q: row_b[q])
                motifs_topk.append({"count": int(min(cnt_ab, row_b[c])), "rels": [a,b,c]})
    if not motifs_topk:
        top_rels = [r for r,_ in sorted(edge_count.items(), key=lambda kv: kv[1], reverse=True)[:3]]
        if len(top_rels)>=3: motifs_topk=[{"count":1,"rels":top_rels[:3]}]
        elif len(top_rels)==2: motifs_topk=[{"count":1,"rels":[top_rels[0],top_rels[1],top_rels[1]]}]
        elif len(top_rels)==1: motifs_topk=[{"count":1,"rels":[top_rels[0]]*3}]

    return {
        "meta": {
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "graphs_used": used,
            "mined_from": str(Path(data_dir).resolve())
        },
        "relations": list(rels),
        "edge_prior_count": edge_count,
        "edge_prior_prob": edge_prior_prob,
        "bigram_count": bigram_count,
        "bigram_prob": bigram_prob,
        "start_count": start_count,
        "end_count": end_count,
        "hop_histogram": dict(hop_hist),
        "motifs_topk": motifs_topk
    }

# ---------- report ----------
def save_ckg_reports(ckg:Dict, json_path:str, md_path:str):
    write_json_utf8(json_path, ckg)
    rels = ckg["relations"]; ec=ckg["edge_prior_count"]; ep=ckg["edge_prior_prob"]
    sc   = ckg.get("start_count", {}); ec_end=ckg.get("end_count", {})
    hop  = ckg.get("hop_histogram", {}); motifs=ckg.get("motifs_topk", [])
    lines=[]
    lines.append("# CKG Mining Report\n")
    lines.append(f"- **Mined from:** `{ckg['meta']['mined_from']}`")
    lines.append(f"- **Graphs used:** {ckg['meta']['graphs_used']}")
    lines.append(f"- **Created at:** {ckg['meta']['created_at']}\n")
    lines.append("## Edge Priors\n| Relation | Count | Prob |\n|---|---:|---:|")
    for r in sorted(rels, key=lambda r: ec.get(r,0), reverse=True):
        lines.append(f"| {r} | {ec.get(r,0)} | {ep.get(r,0.0):.4f} |")
    if sc and ec_end:
        lines.append("\n## Start/End\n| Relation | Start | End |\n|---|---:|---:|")
        for r in rels:
            lines.append(f"| {r} | {sc.get(r,0)} | {ec_end.get(r,0)} |")
    if hop:
        lines.append("\n## Hop Histogram\n| Hops | Count |\n|---:|---:|")
        for h in sorted(hop, key=lambda x: int(x) if isinstance(x,str) else x):
            v = hop[h] if not isinstance(h, str) else hop[h]
            lines.append(f"| {h} | {v} |")
    lines.append("\n## Top Motifs\n")
    if motifs:
        lines.append("| # | Count | Tri-gram |\n|---:|---:|---|")
        for i,m in enumerate(motifs[:20], 1):
            lines.append(f"| {i} | {m['count']} | {' -> '.join(m['rels'])} |")
    else:
        lines.append("_No motifs mined._")
    Path(md_path).parent.mkdir(parents=True, exist_ok=True)
    Path(md_path).write_text("\n".join(lines), encoding="utf-8")

# ---------- MAIN ----------
for p in [CKG_PATH, CKG_PATH_TMP, CKG_RPT_JSON, CKG_RPT_MD]:
    Path(p).parent.mkdir(parents=True, exist_ok=True)

# Load model
assert os.path.exists(CKPT), f"Checkpoint not found: {CKPT}"
model, BASE_REL, ADD_SUMMARY = load_model(CKPT)
print(f"[model] hidden={model.hidden} layers={model.layers} relations={model.relations}")

# FAST mining
ckg = mine_ckg_fast(model, DATA_MINE, BASE_REL, add_summary=ADD_SUMMARY,
                    limit=MINE_LIMIT, seed_k=SEED_K_MINE,
                    width=BEAM_WIDTH_MINE, hops=BEAM_HOPS_MINE, alpha=ALPHA_NODE)
write_json_utf8(CKG_PATH, ckg)
save_ckg_reports(ckg, CKG_RPT_JSON, CKG_RPT_MD)
print(f"[CKG] saved: {CKG_PATH}")
print(f"[REPORT] saved: {CKG_RPT_JSON}, {CKG_RPT_MD}")
print(f"graphs_used={ckg['meta']['graphs_used']} motifs_topk={len(ckg['motifs_topk'])}")


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_29836\1468749568.py:286: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")


[env] torch=2.4.1+cu121 device=cuda
[model] hidden=64 layers=3 relations=['DFG', 'CFG', 'CALL', 'ARG2PARAM', 'RET2CALL', 'RET2LHS', 'DFG_THIN']


[fast-mine] Dataset/train/hetero_ready_gcbert:   0%|          | 0/3438 [00:00<?, ?it/s]C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_29836\1468749568.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related t

[CKG] saved: ckg_light/ckg.json
[REPORT] saved: ckg_light/ckg_report.json, ckg_light/ckg_report.md
graphs_used=3438 motifs_topk=1
